# Nimbus-5 NEMS sea ice concentration, 1972–1973: retrieval, GNN gap-filling and validation

This notebook produces the two datasets and every figure, table and number in

> Andersen, I. G.-S., Tonboe, R. T., Juul, J., Kolbe, W. M. and Tellefsen, E. H.: *Closing Gaps in the 1970's Satellite Sea Ice Concentration Record Using Nimbus-5 NEMS and Graph Neural Networks*.

It runs top to bottom. Each part writes its products to disk, and later parts read them back, so you can start at any part once the earlier products exist.

| Part | What it does | Output | Paper |
|---|---|---|---|
| 1 | NEMS ingestion and screening, dynamic tie points, ERA5/RTM correction, hybrid SIC, uncertainty budget, daily EASE2 gridding with land-spillover correction | **Dataset 1**: daily NEMS SIC (150 days per hemisphere) | Data; Methods (filtering → gridding); Fig. 2 |
| 2 | Period compositing, unobserved-area masks, graph construction, model | – | Methods (GNN) |
| 3 | Masked-reconstruction training and hold-out validation against baseline interpolators | checkpoints, validation tables | Methods; Results (reconstruction skill) |
| 4 | Inference, pole-hole fill, provenance flags and per-cell uncertainty, NetCDF export | **Dataset 2**: 14-day gap-filled NEMS/ESMR SIC (13 periods per hemisphere) | Methods; Results (datasets); Data availability |
| 5 | Validation and figures: extent series, monthly table, NEMS–NIC statistics (Fig. 6 period and every 14-day window), daily NEMS–ESMR cell-by-cell comparison, diagnostics, regional zoom, tie points, screening | figures, `table1_sie_monthly.tex`, `manuscript_numbers.txt` | Results; data figures; Table 1 |

The last cell prints every number quoted in the manuscript, with where it comes from. After any rerun, check the manuscript against that cell.

## Requirements

Python 3.11 with numpy, pandas, xarray, netCDF4, scipy, scikit-learn, xgboost, pyresample, pyproj, cartopy, matplotlib, PyTorch and PyTorch Geometric, plus the project modules `tiepoints_func`, `algos`, `RTM`, `uncertainty`, `correction` and `netcdf_writer` from the code repository (they must be importable; set `PROJECT_MODULES_DIR` in the first code cell if they are not on the Python path).

## Input data

Set the environment variable `NEMS_DATA_ROOT` (or edit `DATA_ROOT` in the configuration cell) to a folder with this layout:

```
NEMS_colocated/                  NEMS Tb co-located with ERA5 (one NetCDF per orbit file)
ESMR/<NH and SH ESA CCI ESMR v1.1 folders>
reference_files/LandOceanLakeMask_{nh,sh}_ease2-250_v2.nc
**/nic_weekly_*.bin               NIC weekly charts (NSIDC G02172), anywhere below DATA_ROOT
NIC/EASE2/*_NH_NIC-BIN.nc        optional: the same charts already on EASE2 (used instead if present)
G00917/arctic.dat, antarc.dat    NOAA/NMC/CAC monthly extent (NSIDC G00917)
```

Outputs go to `NEMS_OUTPUT_ROOT` (default `./output`).

## Reproducibility

All random draws (XGBoost split, block hold-out, mini-batch order, network initialisation) are seeded from `SEED`. The network skill reported in the paper (Results, reconstruction) is computed on the three interleaved test periods only.

In [ ]:
# =============================================================================
# Setup: project modules, imports, random seeds
# =============================================================================
import os
import re
import sys
import glob
import time
import pickle
import random
import json
import warnings
from pathlib import Path
from collections import defaultdict
from datetime import datetime, timedelta

# Folder holding tiepoints_func.py, algos.py, RTM.py, uncertainty.py,
# correction.py and netcdf_writer.py. Leave as None if they are already importable.
PROJECT_MODULES_DIR = os.environ.get("NEMS_MODULES_DIR")
if PROJECT_MODULES_DIR and PROJECT_MODULES_DIR not in sys.path:
    sys.path.insert(0, str(Path(PROJECT_MODULES_DIR)))

import numpy as np
import pandas as pd
import xarray as xr
import netCDF4 as nc

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import matplotlib.patheffects as pe
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

from scipy.ndimage import (zoom, binary_dilation, median_filter, gaussian_filter,
                           distance_transform_edt, uniform_filter)
from scipy.ndimage import label as cc_label
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader as PyGDataLoader
from torch_geometric.nn import SAGEConv

from tiepoints_func import (analyze_sea_ice_tiepoints, analyze_ow_tiepoints,
                            extract_tiepoints1, extract_tiepoints2, map_tiepoints_to_record,
                            remove_outliers_multivariate)
from algos import hybrid, onechannel_22, bootstrapF
from RTM import RadiativeTransferModel
from uncertainty import (resample_to_ease2, calculate_resampling_uncertainty,
                         calculate_total_uncertainty, calculate_sigma_ow_ice, calculate_weight,
                         calculate_delta_2ch, calculate_delta_hybrid, select_final_uncertainty,
                         _get_area_definition, _apply_masks)
from correction import apply_land_spillover_correction_reference
from netcdf_writer import create_NetCDF, make_status_flag

SEED = 0


def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}, seed: {SEED}")

In [ ]:
# =============================================================================
# Configuration: paths and parameters (the only cell that needs editing)
# =============================================================================
DATA_ROOT = Path(os.environ.get("NEMS_DATA_ROOT", "data"))
OUTPUT_ROOT = Path(os.environ.get("NEMS_OUTPUT_ROOT", "output"))

# --- Inputs
NEMS_COLOCATED_DIR = DATA_ROOT / "NEMS_colocated"
NH_ESMR_DIR = DATA_ROOT / "ESMR" / "ESACCI-SEAICE-L3C-SICONC-NIMBUS5_ESMR-EASE2_LDTPcorrected_NH-fv1.1"
SH_ESMR_DIR = DATA_ROOT / "ESMR" / "ESACCI-SEAICE-L3C-SICONC-NIMBUS5_ESMR-EASE2_LDTPcorrected_SH-fv1.1"
ARCTIC_LANDMASK_PATH = DATA_ROOT / "reference_files" / "LandOceanLakeMask_nh_ease2-250_v2.nc"
ANTARCTIC_LANDMASK_PATH = DATA_ROOT / "reference_files" / "LandOceanLakeMask_sh_ease2-250_v2.nc"
NIC_ROOT = DATA_ROOT                         # searched recursively for NSIDC G02172 nic_weekly_*.bin
NIC_EASE2_DIR = DATA_ROOT / "NIC" / "EASE2"  # optional: charts already on EASE2 (*_NH_NIC-BIN.nc)
G00917_DIR = DATA_ROOT / "G00917"            # NOAA/NMC/CAC monthly extent (arctic.dat, antarc.dat)

# --- Outputs
MODEL_DIR = OUTPUT_ROOT / "models"           # XGBoost bias models and GNN checkpoints
DAILY_MISSION_LABEL = "N5NEMS-SIC-1972-1973"          # Dataset 1
PERIOD_MISSION_LABEL = "N5NEMS-SIC-14day-GNN-1972-1973"  # Dataset 2
DAILY_DIR = OUTPUT_ROOT / DAILY_MISSION_LABEL
GNN_FILLED_DIR = OUTPUT_ROOT / PERIOD_MISSION_LABEL
GNN_FILLED_MASKED_DIR = GNN_FILLED_DIR / "ocean_masked"
PLOT_DIR = OUTPUT_ROOT / "figures"
VAL_DIR = OUTPUT_ROOT / "validation"
for _d in (MODEL_DIR, DAILY_DIR, GNN_FILLED_DIR, GNN_FILLED_MASKED_DIR, PLOT_DIR, VAL_DIR):
    _d.mkdir(parents=True, exist_ok=True)

HEMI_FOLDER = {"N": "Northern_Hemisphere", "S": "Southern_Hemisphere"}


def build_save_dir(root_dir, hemisphere, date):
    """<root>/<Northern|Southern>_Hemisphere/<MM>_<Month>_<YYYY>, created if needed."""
    d = Path(root_dir) / HEMI_FOLDER[hemisphere] / f"{date.month:02d}_{date:%B}_{date.year}"
    d.mkdir(parents=True, exist_ok=True)
    return d


def write_netcdf(data_vars, date, hemisphere, area_def, save_dir, retries=4):
    """create_NetCDF with a retry: Windows can keep a handle from the previous write open."""
    for attempt in range(retries):
        try:
            return create_NetCDF(data_vars, date.strftime("%Y-%m-%d"), hemisphere, area_def, str(save_dir))
        except PermissionError:
            xr.backends.file_manager.FILE_CACHE.clear()
            if attempt == retries - 1:
                raise
            time.sleep(1.5)


AREA_DEF = {"N": _get_area_definition("arctic"), "S": _get_area_definition("antarctic")}

# --- Screening and retrieval
TB_RANGE = (100.0, 290.0)      # K, both channels
MIN_ABS_LAT = 42.0             # polar domain
ICE_TP_MIN_SIC = 0.95          # ice tie-point selection
OW_TP_SST = (271.0, 275.0)     # K, open-water tie-point selection
OW_TP_MAX_WIND = 15.0          # m/s, open-water cluster for the RTM reference state
TP_WINDOW_DAYS = 14            # tie-point rolling window
TP_MIN_SAMPLES = 100
TP_IQR_MULTIPLIER = 1.0
RTM_REF_WINDOW_DAYS = 15       # centred running mean of the cluster atmospheres
RETRAIN_BIAS_MODELS = True     # False: load the XGBoost models saved in MODEL_DIR
RETRAIN_GNN = os.environ.get("NEMS_RETRAIN_GNN", "1") != "0"   # False: reload the network weights in MODEL_DIR
SPILLOVER_WINDOW_PX = 9        # 9 x 9 cells = 225 km, matches the 200 km footprint
RESAMPLING_WINDOW_PX = 3       # Eq. (resampling uncertainty)

# --- Compositing and GNN
PERIOD_DAYS = 14
BLOCK_SIZE = 12                # super-pixel = 12 x 12 cells = 300 km
HIDDEN_DIM = 64
NUM_LAYERS = 4
MASK_RATIO = 0.35              # self-supervised masking in make_graph(training=True)
HOLD_FRAC_TRAIN = 0.35         # composite-level block hold-out, training
HOLD_FRAC_EVAL = 0.20          # composite-level block hold-out, validation and test
BASELINES = ["nearest-valid", "persistence", "temporal-interp", "blended", "esmr-interp"]
OUTAGE_SIM_PROB = 0.8          # training: share of samples with the in-window ESMR removed
OUTAGE_SIM_MAX_DAYS = 75       # ... and up to this many days of ESMR either side (the spring
                               # outage leaves the April windows 40-60 days from ESMR)
VAL_OUTAGE_DAYS = 45           # validation also scores each period as a 45-day-margin outage
TRAIN_HOLD_SIZES = (1, 2, 4, 8)  # withheld squares of k x k blocks (300-2400 km), drawn at random
                               # per sample, so that the network sees gaps as large as the outages leave
PHYSICS_PENALTY_WEIGHT = 0.5   # lambda in Eq. (loss)
BATCH_SIZE = 8
NUM_EPOCHS = 100
LR = 2e-3
N_FEATURES = 11                # spatial node features
N_TEMPORAL = 4                 # previous/next-period features
N_BASELINE = 5                 # block means of the blend, density and ESMR field; ESMR distance; gap distance
GAP_DIST_SCALE_KM = 1200.0     # distance to the nearest defined cell is divided by this and capped at 1
N_FEATURES_V4 = N_FEATURES + N_TEMPORAL + N_BASELINE
# The network does not predict SIC directly. For each node it outputs mixing weights over
# three pixel-level baseline interpolations (nearest valid value; mean of the neighbouring
# periods; their density-weighted blend) plus a bounded correction. Weights and correction
# are interpolated to 25 km, so the network chooses per region which interpolator to trust.
RESIDUAL_ON_BASELINE = True
N_BASES = 4                    # + ESMR for the window, interpolated in time across outages
MAX_CORRECTION = 0.5           # largest node correction, as a SIC fraction (tanh-bounded)

UNOBSERVED_MIN_COVERAGE = 0.05  # cells observed on < 5 % of sampled dates are 'unobserved'
COMPOSITE_SMOOTH_PX = 2         # 5 x 5 NaN-aware median filter on the composite
GNN_CLAMP_THRESHOLD = 30.0      # % in the neighbourhood-consistency clamp
FRINGE_DILATION_PX = BLOCK_SIZE + 2
# The prediction fringe is discarded around never-observed OCEAN only. The original
# code dilated the full unobserved mask, which also contains all land, so every GNN
# prediction within 350 km of a coast was replaced by a nearest-neighbour copy.
FRINGE_AROUND_LAND = False

# --- Pole hole (ESMR fill)
POLE_ESMR_MIN_DAYS = 3
POLE_ESMR_SIDE_DAYS = 7
POLE_ESMR_MAX_GAP_DAYS = 120
POLE_FEATHER_PX = 4

# --- Extent and validation
SIE_THRESHOLD = 30.0           # % (consistent with the ESMR and SCAMS records)
PIXEL_AREA_KM2 = 625.0
ESMR_MIN_DAYS = 5              # a 14-day window with fewer usable ESMR days counts as an ESMR outage
FIRST_WINDOW = pd.Timestamp("1972-12-17")   # NEMS periods and all comparison windows start here
LAST_WINDOW_START = pd.Timestamp("1973-11-25")

In [ ]:
# =============================================================================
# Shared figure style
# =============================================================================
MM = 1 / 25.4
INK, FRAME, MUTED, GRID_COL, LAND_COL = "#1A1A1A", "#3C3C3C", "#5A5A5A", "#E4E6E9", "#B0B0B0"
BLUE_DEEP, BLUE_DARK, BLUE, BLUE_LIGHT = "#16324D", "#1E5079", "#3498C3", "#7ABBD6"
ORANGE_LIGHT, ORANGE, ORANGE_DARK = "#ECA458", "#F58220", "#A94A16"

plt.rcParams.update({
    "font.family": "sans-serif", "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 8, "axes.labelcolor": INK, "axes.edgecolor": FRAME, "axes.linewidth": 0.5,
    "xtick.color": INK, "ytick.color": INK, "text.color": INK, "legend.frameon": False,
    "figure.facecolor": "white", "savefig.facecolor": "white", "savefig.dpi": 300,
    "pdf.fonttype": 42, "ps.fonttype": 42,
})


def to_percent(arr):
    """SIC in percent, whether the array holds fractions (max <= 1.5) or percent."""
    if arr is None:
        return None
    a = np.asarray(arr, dtype=np.float32)
    finite = a[np.isfinite(a)]
    return a * 100.0 if finite.size and finite.max() <= 1.5 else a

## Part 1 — Daily NEMS SIC retrieval (Dataset 1)

Screening, dynamic tie points, ERA5/RTM noise reduction, hybrid one- and two-channel SIC, uncertainty budget, and daily gridding to the 25 km EASE2 grids with land-spillover correction.

In [ ]:
# =============================================================================
# 1a. Load the co-located NEMS/ERA5 observations and screen them
# =============================================================================
files = sorted(glob.glob(str(NEMS_COLOCATED_DIR / "**.nc")))
if not files:
    raise FileNotFoundError(f"No co-located NEMS files in {NEMS_COLOCATED_DIR}")
ds = xr.open_mfdataset(files, combine="nested", concat_dim="t")

with xr.open_dataset(ARCTIC_LANDMASK_PATH) as _m:
    arctic_smask = _m["smask_sicci"].to_numpy().astype(int)
with xr.open_dataset(ANTARCTIC_LANDMASK_PATH) as _m:
    antarctic_smask = _m["smask_sicci"].to_numpy().astype(int)

frames = []
for frame in range(ds.sizes["n14_frames"]):
    V = ds["tcwv"].isel(n16_frames=frame)
    tcw = ds["tcw"].isel(n16_frames=frame)
    frames.append(pd.DataFrame({
        "lat": ds["LAT"].isel(n16_frames=frame).values.ravel(),
        "lon": ds["LON"].isel(n16_frames=frame).values.ravel(),
        "V": V.values.ravel(),
        "Ta": ds["t2m"].isel(n16_frames=frame).values.ravel(),
        "Ts": ds["sst"].isel(n16_frames=frame).values.ravel(),
        "c_ice": ds["siconc"].isel(n16_frames=frame).values.ravel(),
        "tcw": tcw.values.ravel(),
        "L": (tcw - V).values.ravel(),                     # cloud liquid water = TCW - TCWV
        "W": np.sqrt(ds["u10"].isel(n16_frames=frame) ** 2 +
                     ds["v10"].isel(n16_frames=frame) ** 2).values.ravel(),
        "tb_22_measured": ds["TBNEMS"].isel(n5_channels=0, n14_frames=frame).values.ravel(),
        "tb_31_measured": ds["TBNEMS"].isel(n5_channels=1, n14_frames=frame).values.ravel(),
        "time": pd.to_datetime(ds["Time"].isel(n16_frames=frame).values),
    }))
all_obs = pd.concat(frames, ignore_index=True)

lo, hi = TB_RANGE
in_range = all_obs["tb_22_measured"].between(lo, hi) & all_obs["tb_31_measured"].between(lo, hi)
tb_ok = all_obs[in_range]
df = tb_ok.dropna().copy()          # any undefined ERA5 field (SST and SIC are undefined over land)
df["datetime"] = df["time"]
df["year"], df["month"], df["day"] = df["time"].dt.year, df["time"].dt.month, df["time"].dt.day

_cols = ["tb_22_measured", "tb_31_measured", "c_ice", "W", "lat", "lon",
         "Ts", "Ta", "V", "L", "datetime", "year", "month", "day"]
arctic_record = df.loc[df["lat"] >= MIN_ABS_LAT, _cols].copy()
antarctic_record = df.loc[df["lat"] <= -MIN_ABS_LAT, _cols].copy()

# Screening counts (Data and Methods sections, Fig. 2)
SCREENING = {
    "all": len(all_obs),
    "all_with_time": int(all_obs["time"].notna().sum()),
    "tb_range": len(tb_ok),
    "complete": len(df),
    "nh": len(arctic_record),
    "sh": len(antarctic_record),
}
for _h, _sel in (("nh", tb_ok["lat"] >= MIN_ABS_LAT), ("sh", tb_ok["lat"] <= -MIN_ABS_LAT)):
    _n_polar = int(_sel.sum())
    SCREENING[f"{_h}_polar_after_tb"] = _n_polar
    SCREENING[f"{_h}_discarded_frac"] = 1 - SCREENING[_h] / max(_n_polar, 1)
SCREENING["n_days"] = int(df["time"].dt.normalize().nunique())
for k, v in SCREENING.items():
    print(f"  {k:<24}{v:>12,.3f}" if isinstance(v, float) else f"  {k:<24}{v:>12,}")

In [ ]:
# =============================================================================
# 1b. First-pass tie points and hybrid SIC; XGBoost bias models for the RTM
# =============================================================================
TP_COLS = ["FYI_tp22", "FYI_tp31", "MYI_tp22", "MYI_tp31", "OW_tp22", "OW_tp31"]


def first_pass(record):
    ice = analyze_sea_ice_tiepoints(record, min_c_ice=ICE_TP_MIN_SIC,
                                    rolling_mean_window=TP_WINDOW_DAYS, iqr_multiplier=TP_IQR_MULTIPLIER)
    ow = analyze_ow_tiepoints(record, min_sst=OW_TP_SST[0], max_sst=OW_TP_SST[1], max_c_ice=0.0,
                              min_samples=TP_MIN_SAMPLES, rolling_mean_window=TP_WINDOW_DAYS)
    map_tiepoints_to_record(record, extract_tiepoints1(ice, ow))
    # Dates whose window has fewer than TP_MIN_SAMPLES observations get no tie point;
    # they take the next available one (stated in Methods).
    record[TP_COLS] = record[TP_COLS].bfill()
    ok = record[TP_COLS + ["tb_22_measured", "tb_31_measured"]].notna().all(axis=1)
    record["c_ice_hybrid"] = np.nan
    record.loc[ok, "c_ice_hybrid"] = [
        hybrid(r.tb_22_measured, r.tb_31_measured, r.FYI_tp22, r.MYI_tp22, r.OW_tp22,
               r.FYI_tp31, r.MYI_tp31, r.OW_tp31)
        for r in record.loc[ok].itertuples()]
    return record


arctic_record = first_pass(arctic_record)
antarctic_record = first_pass(antarctic_record)

# XGBoost models of the residual between the parametric RTM and the measured Tb, as a
# function of (V, W, L, Ta, Ts, c_ice). Trained on ERA5 c_ice, not on the retrieval,
# to avoid feeding the algorithm's own SIC back into its correction.
BIAS_FEATURES = ["V", "W", "L", "Ta", "Ts", "c_ice"]
BIAS_MODEL_PATHS = {(r, ch): MODEL_DIR / f"{r}_{ch}_model.pkl"
                    for r in ("arctic", "antarctic") for ch in ("22", "31")}
BIAS_MODEL_SKILL = {}


def train_bias_models(record, region):
    sub = record[BIAS_FEATURES + ["tb_22_measured", "tb_31_measured"]].dropna()
    rtm0 = RadiativeTransferModel(apply_correction=False)
    sim = np.array([rtm0.simulate(V=r.V, W=r.W, L=r.L, Ta=r.Ta, Ts=r.Ts, c_ice=r.c_ice)
                    for r in sub.itertuples()])
    X = sub[BIAS_FEATURES].values
    for k, ch in enumerate(("22", "31")):
        y = sim[:, k] - sub[f"tb_{ch}_measured"].values
        X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=SEED)
        model = XGBRegressor(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8,
                             colsample_bytree=0.8, random_state=SEED)
        model.fit(X_tr, y_tr)
        rmse = lambda a, b: float(np.sqrt(np.mean((a - b) ** 2)))
        BIAS_MODEL_SKILL[(region, ch)] = dict(
            n=len(X), resid_std_before=float(np.std(y)),
            train_rmse=rmse(model.predict(X_tr), y_tr), test_rmse=rmse(model.predict(X_te), y_te))
        with open(BIAS_MODEL_PATHS[(region, ch)], "wb") as f:
            pickle.dump(model, f)
        print(f"{region} {ch} GHz: {BIAS_MODEL_SKILL[(region, ch)]}")


if RETRAIN_BIAS_MODELS or not all(p.exists() for p in BIAS_MODEL_PATHS.values()):
    train_bias_models(arctic_record, "arctic")
    train_bias_models(antarctic_record, "antarctic")

In [ ]:
# =============================================================================
# 1c. RTM noise reduction with a SIC-weighted reference atmosphere
# =============================================================================
# Reference state per observation, mixed from the ice- and open-water-cluster mean
# atmospheres in proportion to the observation's first-pass hybrid SIC:
#   X_ref = c * X_ice(day) + (1 - c) * X_water(day),  X in {V, W, L, Ta, Ts}
# Delta Tb = RTM(X_ref, c) - RTM(X, c);  Tb_corrected = Tb_measured + Delta Tb
rtm = RadiativeTransferModel(apply_correction=True)
rtm.set_bias_models(BIAS_MODEL_PATHS[("arctic", "22")], BIAS_MODEL_PATHS[("arctic", "31")],
                    BIAS_MODEL_PATHS[("antarctic", "22")], BIAS_MODEL_PATHS[("antarctic", "31")])
ATMOS_VARS = ["V", "W", "L", "Ta", "Ts"]


def daily_cluster_means(record, ice_mask, water_mask):
    date = pd.to_datetime(record["datetime"]).dt.normalize()
    full = pd.date_range(date.min(), date.max(), freq="D")
    out = []
    for mask, tag in ((ice_mask, "ice"), (water_mask, "water")):
        m = record.loc[mask, ATMOS_VARS].groupby(date[mask]).mean().reindex(full)
        m = m.rolling(window=RTM_REF_WINDOW_DAYS, center=True, min_periods=1).mean()
        out.append(m.add_suffix(f"_{tag}_daily"))
    return out[0].join(out[1]).bfill().ffill()


def rtm_batch(region, X, c):
    rtm.set_region(region)
    out = np.array([rtm.simulate(V=v, W=w, L=l, Ta=t, Ts=s, c_ice=ci)
                    for v, w, l, t, s, ci in zip(X["V"], X["W"], X["L"], X["Ta"], X["Ts"], c)])
    return out[:, 0], out[:, 1]


def rtm_correct(record, region):
    c = record["c_ice_hybrid"]
    ice = c >= ICE_TP_MIN_SIC
    water = (c <= 0.0) & record["Ts"].between(*OW_TP_SST) & (record["W"] <= OW_TP_MAX_WIND)
    means = daily_cluster_means(record, ice, water)
    record = record.copy()
    record["date"] = pd.to_datetime(record["datetime"]).dt.normalize()
    record = record.join(means, on="date")
    ref = pd.DataFrame({v: c * record[f"{v}_ice_daily"] + (1 - c) * record[f"{v}_water_daily"]
                        for v in ATMOS_VARS})
    tb22_ref, tb31_ref = rtm_batch(region, ref, c)
    tb22_sim, tb31_sim = rtm_batch(region, record[ATMOS_VARS], c)
    record["tb_22_phys_corrected"] = record["tb_22_measured"] + (tb22_ref - tb22_sim)
    record["tb_31_phys_corrected"] = record["tb_31_measured"] + (tb31_ref - tb31_sim)
    return record


arctic_record = rtm_correct(arctic_record, "arctic")
antarctic_record = rtm_correct(antarctic_record, "antarctic")
for _n, _r in (("Arctic", arctic_record), ("Antarctic", antarctic_record)):
    print(f"{_n}: mean |Delta Tb| 22 GHz {np.nanmean(np.abs(_r.tb_22_phys_corrected - _r.tb_22_measured)):.2f} K, "
          f"31 GHz {np.nanmean(np.abs(_r.tb_31_phys_corrected - _r.tb_31_measured)):.2f} K")

In [ ]:
# =============================================================================
# 1d. Second-pass tie points, corrected hybrid SIC and algorithm uncertainty
# =============================================================================
# Second pass: ice tie points are taken where the FIRST-PASS hybrid SIC >= 0.95
# (col_c_ice='c_ice_hybrid'); open-water tie points still use ERA5 SIC = 0.
TP_CORR = ["FYI_tp22_corrected", "FYI_tp31_corrected", "MYI_tp22_corrected",
           "MYI_tp31_corrected", "OW_tp22_corrected", "OW_tp31_corrected"]
TB_CORR = ["tb_22_phys_corrected", "tb_31_phys_corrected"]


def second_pass(record):
    ice = analyze_sea_ice_tiepoints(record, min_c_ice=ICE_TP_MIN_SIC, col_c_ice="c_ice_hybrid",
                                    col_22=TB_CORR[0], col_31=TB_CORR[1],
                                    rolling_mean_window=TP_WINDOW_DAYS, iqr_multiplier=TP_IQR_MULTIPLIER)
    ow = analyze_ow_tiepoints(record, col_22=TB_CORR[0], col_31=TB_CORR[1],
                              min_sst=OW_TP_SST[0], max_sst=OW_TP_SST[1], max_c_ice=0.0,
                              min_samples=TP_MIN_SAMPLES, rolling_mean_window=TP_WINDOW_DAYS)
    map_tiepoints_to_record(record, extract_tiepoints2(ice, ow))
    cols = TP_CORR + [c + "_std" for c in TP_CORR]
    record[cols] = record[cols].bfill()

    ok = record[TB_CORR + TP_CORR].notna().all(axis=1)
    sub = record.loc[ok]
    for name in ("c_ice_hybrid_corrected", "c_ice_1ch_corrected", "c_ice_2ch_corrected"):
        record[name] = np.nan
    args = [(r.tb_22_phys_corrected, r.tb_31_phys_corrected, r.FYI_tp22_corrected, r.MYI_tp22_corrected,
             r.OW_tp22_corrected, r.FYI_tp31_corrected, r.MYI_tp31_corrected, r.OW_tp31_corrected)
            for r in sub.itertuples()]
    record.loc[ok, "c_ice_hybrid_corrected"] = [hybrid(*a) for a in args]
    record.loc[ok, "c_ice_1ch_corrected"] = [onechannel_22(a[0], a[2], a[3], a[4]) for a in args]
    record.loc[ok, "c_ice_2ch_corrected"] = [bootstrapF(a[1], a[0], *a[2:]) for a in args]
    return record


def delta_1ch(sic_1ch, tp_myi, tp_fyi, tp_water, std_myi, std_fyi, std_water):
    """Eq. (uncertainty_1ch): tie-point spread propagated through the 22 GHz algorithm."""
    vals = (sic_1ch, tp_myi, tp_fyi, tp_water, std_myi, std_fyi, std_water)
    if any(pd.isna(v) for v in vals):
        return np.nan
    tp_ice = (tp_myi + tp_fyi) / 2
    contrast = tp_ice - tp_water
    if contrast == 0:
        return np.nan
    d_ice = np.sqrt(std_myi ** 2 + std_fyi ** 2) / 2
    s = np.clip(sic_1ch, 0.0, 1.0)
    return float(np.sqrt(((1 - s) * std_water / contrast) ** 2 + (s * d_ice / contrast) ** 2))


def algorithm_uncertainty(record, region):
    # Eq. (uncertainty_1ch) is evaluated at SIC_1ch and Eq. (uncertainty_2ch) at SIC_2ch.
    # The original code evaluated both at the hybrid SIC.
    record["delta_1ch"] = [
        delta_1ch(r.c_ice_1ch_corrected, r.MYI_tp22_corrected, r.FYI_tp22_corrected, r.OW_tp22_corrected,
                  r.MYI_tp22_corrected_std, r.FYI_tp22_corrected_std, r.OW_tp22_corrected_std)
        for r in record.itertuples()]
    sub = record.dropna(subset=["c_ice_2ch_corrected", "c_ice"])
    s_ow, s_ice = calculate_sigma_ow_ice(sub, c_ice_2ch_col="c_ice_2ch_corrected", c_ice_ref_col="c_ice")
    SIGMA_2CH[region] = (float(s_ow), float(s_ice))
    record["delta_2ch"] = record["c_ice_2ch_corrected"].clip(0, 1).apply(
        lambda s: calculate_delta_2ch(s, s_ow, s_ice))
    record["weight"] = record["c_ice_1ch_corrected"].clip(0, 1).apply(calculate_weight)
    record["delta_hybrid"] = [calculate_delta_hybrid(a, b, w) for a, b, w in
                              zip(record["delta_1ch"], record["delta_2ch"], record["weight"])]
    record["delta_algorithm"] = [select_final_uncertainty(s, a, b, h) for s, a, b, h in
                                 zip(record["c_ice_1ch_corrected"], record["delta_1ch"],
                                     record["delta_2ch"], record["delta_hybrid"])]
    return record


SIGMA_2CH = {}
arctic_record = algorithm_uncertainty(second_pass(arctic_record), "arctic")
antarctic_record = algorithm_uncertainty(second_pass(antarctic_record), "antarctic")
for _r, (_ow, _ice) in SIGMA_2CH.items():
    print(f"{_r:<10} sigma_OW {_ow:.4f}  sigma_ICE {_ice:.4f}  (over the full record)")

In [ ]:
# =============================================================================
# 1e. Daily gridding to EASE2, land-spillover correction, total uncertainty, NetCDF
# =============================================================================
RESAMPLE_FEATURES = ["tb_22_phys_corrected", "tb_31_phys_corrected", "tb_22_measured", "tb_31_measured",
                     "c_ice_hybrid_corrected", "uncertainty_algorithm"]
SPILLOVER_FLAG = 8          # status_flag value OR'ed in where the spillover correction zeroed SIC
CLIMATOLOGY_FLAG = 64       # status_flag value: outside the monthly climatological maximum ice extent
_CLIM = {}


def climatology_zero(mask_path, month):
    """Ocean cells outside the monthly climatological maximum extent (climatology == 0
    in the LandOceanLakeMask file), where _apply_masks sets SIC and uncertainty to zero."""
    key = (str(mask_path), month)
    if key not in _CLIM:
        with xr.open_dataset(mask_path) as _m:
            _CLIM[key] = (_m["climatology"].values[month - 1] == 0) & (_m["lmask"].values == 0)
    return _CLIM[key]


def process_day(record, area, mask_path, smask, date):
    sub = record[record["date"] == date].copy()
    if sub.empty:
        return None
    sub["uncertainty_algorithm"] = sub["delta_algorithm"]
    g = resample_to_ease2(sub, area, RESAMPLE_FEATURES, mask_path=mask_path, month=date.month)
    get = lambda name: g[:, :, RESAMPLE_FEATURES.index(name)]
    sic_raw = get("c_ice_hybrid_corrected")                       # fraction
    sic, spill, _, _ = apply_land_spillover_correction_reference(
        sic_raw, smask, filter_size=SPILLOVER_WINDOW_PX, validate=False)
    unc_alg = get("uncertainty_algorithm")
    unc_res = calculate_resampling_uncertainty(sic, window_size=RESAMPLING_WINDOW_PX)
    return dict(sic=sic, sic_raw=sic_raw, spillover=spill, unc_alg=unc_alg, unc_res=unc_res,
                unc_total=calculate_total_uncertainty(unc_alg, unc_res),
                tb_22=get("tb_22_measured"), tb_31=get("tb_31_measured"),
                tb_22_corr=get("tb_22_phys_corrected"), tb_31_corr=get("tb_31_phys_corrected"),
                n_obs=len(sub))


DAILY_STATS = []
for hemi, record, region, mask_path, smask in (
        ("N", arctic_record, "arctic", ARCTIC_LANDMASK_PATH, arctic_smask),
        ("S", antarctic_record, "antarctic", ANTARCTIC_LANDMASK_PATH, antarctic_smask)):
    for date in sorted(pd.Timestamp(d) for d in record["date"].dropna().unique()):
        res = process_day(record, region, mask_path, smask, date)
        if res is None:
            continue
        flag = make_status_flag(res["sic"], smask)
        flag = flag | (np.asarray(res["spillover"]).astype(flag.dtype) * SPILLOVER_FLAG)
        flag = flag | (climatology_zero(mask_path, date.month).astype(flag.dtype) * CLIMATOLOGY_FLAG)
        write_netcdf({"ice_conc": res["sic"], "raw_ice_conc_values": res["sic_raw"],
                      "total_standard_error": res["unc_total"],
                      "algorithm_standard_error": res["unc_alg"],
                      "smearing_standard_error": res["unc_res"], "status_flag": flag,
                      "tb_22": res["tb_22"], "tb_31": res["tb_31"],
                      "tb_22_corr": res["tb_22_corr"], "tb_31_corr": res["tb_31_corr"]},
                     date, hemi, AREA_DEF[hemi], build_save_dir(DAILY_DIR, hemi, date))
        DAILY_STATS.append(dict(hemi=hemi, date=date, n_obs=res["n_obs"],
                                unc_alg=np.nanmean(res["unc_alg"]), unc_res=np.nanmean(res["unc_res"]),
                                unc_total=np.nanmean(res["unc_total"]),
                                n_spillover=int(np.nansum(res["spillover"]))))
DAILY_STATS = pd.DataFrame(DAILY_STATS)
print(DAILY_STATS.groupby("hemi")[["n_obs", "unc_alg", "unc_res", "unc_total", "n_spillover"]].mean())
print(f"Daily files written: {DAILY_STATS.groupby('hemi').size().to_dict()} -> {DAILY_DIR}")

## Part 2 — Compositing, masks and graph construction

Reads the daily product written in Part 1, so it can be run on its own once Dataset 1 exists.

In [ ]:
# =============================================================================
# 2a. Files, unobserved-area masks and 14-day periods
# =============================================================================
# Ocean mask for everything from compositing onward: the reference file's `lmask` (0 = ocean),
# which excludes a coastal band that `smask_sicci` (used for the daily land-spillover
# correction) counts as ocean. Extents in Table 3 and Fig. 5 are counted over it.
OCEAN = {}
for _h, _p in (("N", ARCTIC_LANDMASK_PATH), ("S", ANTARCTIC_LANDMASK_PATH)):
    with xr.open_dataset(_p) as _m:
        OCEAN[_h] = _m["lmask"].to_numpy() == 0

if "arctic_smask" not in globals():          # Part 2 run without Part 1
    with xr.open_dataset(ARCTIC_LANDMASK_PATH) as _m:
        arctic_smask = _m["smask_sicci"].to_numpy().astype(int)
    with xr.open_dataset(ANTARCTIC_LANDMASK_PATH) as _m:
        antarctic_smask = _m["smask_sicci"].to_numpy().astype(int)


def file_date(path):
    m = re.search(r"(\d{8})", Path(path).stem)
    return m.group(1) if m else None


def discover(hemi):
    tag = f"_{'NH' if hemi == 'N' else 'SH'}-"
    nems_dir = DAILY_DIR / HEMI_FOLDER[hemi]
    esmr_dir = NH_ESMR_DIR if hemi == "N" else SH_ESMR_DIR
    pick = lambda root: {file_date(f): f for f in sorted(Path(root).rglob("*.nc"))
                         if file_date(f) and (tag in f.name or tag.replace("-", "_") in f.name)}
    nems = pick(nems_dir)
    esmr = pick(esmr_dir) if esmr_dir.exists() else {}
    dates = sorted(nems)
    print(f"[{hemi}H] NEMS days {len(nems)} ({dates[0]} to {dates[-1]}), ESMR days {len(esmr)}")
    return nems, esmr, dates


def _read(path, var):
    with xr.open_dataset(path) as d:
        if var not in d:
            return None
        return d[var].values[0].astype(np.float32)


def load_ice(path):
    """SIC [%] from a daily NEMS or ESMR file."""
    return _read(path, "ice_conc")


def load_unc(path):
    """Total standard error as a fraction (files store percent); NaN if absent."""
    u = _read(path, "total_standard_error")
    if u is None:
        u = np.full_like(load_ice(path), np.nan)
    return u / 100.0


nh_nems, nh_esmr, nh_dates = discover("N")
sh_nems, sh_esmr, sh_dates = discover("S")


def unobserved_mask(dates, nems_dict, min_frac=UNOBSERVED_MIN_COVERAGE, subsample=3):
    """Cells defined on < min_frac of every third date. Land is never defined, so the
    mask contains all land as well as the never-observed ocean (the pole hole)."""
    sampled = dates[::subsample]
    count = sum((~np.isnan(load_ice(nems_dict[d]))).astype(np.int32) for d in sampled)
    return (count / len(sampled)) < min_frac


def pole_cap_mask(unobs, ocean):
    """Connected never-observed ocean region containing the grid centre (the pole).
    Empty in the Antarctic, where everything poleward of 81 S is land or ice shelf."""
    lab, _ = cc_label(unobs & ocean)
    c = lab[lab.shape[0] // 2, lab.shape[1] // 2]
    return lab == c if c > 0 else np.zeros(lab.shape, bool)


MASKS = {}
for hemi, dates, nems, smask in (("N", nh_dates, nh_nems, arctic_smask),
                                 ("S", sh_dates, sh_nems, antarctic_smask)):
    unobs = unobserved_mask(dates, nems)
    ocean = OCEAN[hemi]
    MASKS[hemi] = dict(unobs=unobs, ocean=ocean, land=~ocean,
                       fringe_src=unobs if FRINGE_AROUND_LAND else (unobs & ocean),
                       pole_cap=pole_cap_mask(unobs, ocean))
    m = MASKS[hemi]
    print(f"[{hemi}H] unobserved {m['unobs'].sum():,} cells ({m['unobs'].mean():.1%} of grid), of which ocean "
          f"{(m['unobs'] & ocean).sum():,}; pole hole {m['pole_cap'].sum():,} cells "
          f"({m['pole_cap'].sum() * PIXEL_AREA_KM2 / 1e6:.2f} million km2)")
nh_ph, sh_ph = MASKS["N"]["unobs"], MASKS["S"]["unobs"]


def group_by_period(dates):
    """'YYYYMMDD' dates -> {period start: [dates]}, 14-day windows counted from FIRST_WINDOW."""
    groups = defaultdict(list)
    for d in dates:
        k = (datetime.strptime(d, "%Y%m%d") - FIRST_WINDOW).days // PERIOD_DAYS
        groups[(FIRST_WINDOW + timedelta(days=k * PERIOD_DAYS)).strftime("%Y%m%d")].append(d)
    return dict(sorted(groups.items()))


def period_label(pk):
    s = datetime.strptime(pk, "%Y%m%d")
    return f"{s:%Y-%m-%d} to {s + timedelta(days=PERIOD_DAYS - 1):%Y-%m-%d}"


nh_periods, sh_periods = group_by_period(nh_dates), group_by_period(sh_dates)
for _h, _p in (("NH", nh_periods), ("SH", sh_periods)):
    print(f"[{_h}] {len(_p)} periods: " + ", ".join(f"{k} ({len(v)} d)" for k, v in _p.items()))

In [ ]:
# =============================================================================
# 2b. 14-day period composite
# =============================================================================
def _nan_median_smooth(arr, radius=COMPOSITE_SMOOTH_PX):
    """NaN-aware median filter that only updates cells that already have a value
    (and at least 3 valid neighbours); it never fills gaps."""
    if radius <= 0:
        return arr
    size = 2 * radius + 1
    win = np.lib.stride_tricks.sliding_window_view(np.pad(arr, radius, mode="edge"), (size, size))
    win = win.reshape(*arr.shape, size * size)
    with np.errstate(all="ignore"), warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)          # all-NaN windows
        med = np.nanmedian(win, axis=-1).astype(np.float32)
    out = arr.copy()
    ok = ~np.isnan(arr) & (np.sum(~np.isnan(win), axis=-1) >= 3)
    out[ok] = med[ok]
    return out


def _boundary_prior(comp):
    """Nearest defined value everywhere (Euclidean distance transform), as a fraction."""
    prior = comp / 100.0
    gap = np.isnan(prior)
    if gap.any():
        _, idx = distance_transform_edt(gap, return_indices=True)
        prior = prior[idx[0], idx[1]]
    return np.clip(prior, 0.0, 1.0).astype(np.float32)


def _stack_mean_std(arrays):
    cube = np.stack(arrays, 0)
    with np.errstate(all="ignore"), warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        return np.nanmean(cube, 0).astype(np.float32), np.nanstd(cube, 0).astype(np.float32)


def build_composite(period_dates, nems_dict, esmr_dict, unobs):
    """
    NEMS period mean, ESMR period mean where NEMS has no value (including unobserved
    ocean such as the pole hole), then a 5 x 5 median filter on defined cells.

    Returns comp [%], day_count, sic_std, esmr_mask, esmr_ph_mask, unc_mean, unc_std,
    esmr_sic_mean, esmr_sic_std, esmr_day_frac, boundary_prior, nems_cov, total_cov.
    """
    H, W = unobs.shape
    nems_days = [d for d in period_dates if d in nems_dict]
    if nems_days:
        ice = []
        unc = []
        for d in nems_days:
            a = load_ice(nems_dict[d]); a[unobs] = np.nan; ice.append(a)
            u = load_unc(nems_dict[d]); u[unobs] = np.nan; unc.append(u)
        day_count = np.sum([~np.isnan(a) for a in ice], axis=0).astype(np.int16)
        comp, sic_std = _stack_mean_std(ice)
        unc_mean, unc_std = _stack_mean_std(unc)
        fill = float(np.nanmean(unc_mean)) if np.isfinite(np.nanmean(unc_mean)) else 0.05
        unc_mean = np.where(np.isnan(unc_mean), fill, unc_mean)
        unc_std = np.where(np.isnan(unc_std), 0.0, unc_std)
    else:
        day_count = np.zeros((H, W), np.int16)
        comp = np.full((H, W), np.nan, np.float32)
        sic_std = np.full((H, W), np.nan, np.float32)
        unc_mean = np.full((H, W), 0.05, np.float32)
        unc_std = np.zeros((H, W), np.float32)

    obs = ~unobs
    nems_cov = float((~np.isnan(comp) & obs).sum() / max(int(obs.sum()), 1))

    esmr_mask = np.zeros((H, W), bool)
    esmr_ph_mask = np.zeros((H, W), bool)
    esmr_sic_mean = np.full((H, W), np.nan, np.float32)
    esmr_sic_std = np.zeros((H, W), np.float32)
    esmr_days = [d for d in period_dates if d in esmr_dict]
    esmr_day_frac = 0.0
    if esmr_days:
        esmr_sic_mean, esmr_sic_std = _stack_mean_std([load_esmr_fill(esmr_dict[d]) for d in esmr_days])
        esmr_day_frac = len(esmr_days) / max(len(period_dates), 1)
        outside = np.isnan(comp) & ~np.isnan(esmr_sic_mean) & obs
        inside = unobs & ~np.isnan(esmr_sic_mean)
        comp[outside] = esmr_sic_mean[outside]
        comp[inside] = esmr_sic_mean[inside]
        esmr_mask[outside] = True
        esmr_ph_mask[inside] = True

    total_cov = float((~np.isnan(comp)).sum() / comp.size)
    comp = _nan_median_smooth(comp)
    return (comp, day_count, sic_std, esmr_mask, esmr_ph_mask, unc_mean, unc_std,
            esmr_sic_mean, esmr_sic_std, esmr_day_frac, _boundary_prior(comp), nems_cov, total_cov)


def _esmr_mean(dates, esmr, loader=None):
    loader = loader or load_esmr_fill          # ESMR converted to NEMS (cell 2b2)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        return np.nanmean(np.stack([loader(esmr[d]) for d in dates], 0), axis=0).astype(np.float32)


def esmr_time_field(pk, esmr, loader=None):
    """ESMR SIC [%] (or, with loader=load_unc, its uncertainty) for the 14-day window starting at pk: the window mean when ESMR has
    >= POLE_ESMR_MIN_DAYS days in it, otherwise ESMR interpolated linearly in time between
    the nearest POLE_ESMR_SIDE_DAYS days before and after the window. Returns
    (field, description, distance in days from the window midpoint to the ESMR used)."""
    if not esmr:
        return None, "no ESMR files", np.nan
    w0 = datetime.strptime(pk, "%Y%m%d")
    w1 = w0 + timedelta(days=PERIOD_DAYS - 1)
    mid = w0 + timedelta(days=PERIOD_DAYS / 2)
    dt = {d: datetime.strptime(d, "%Y%m%d") for d in esmr}
    keys = sorted(esmr)

    inside = [d for d in keys if w0 <= dt[d] <= w1]
    if len(inside) >= POLE_ESMR_MIN_DAYS:
        return _esmr_mean(inside, esmr, loader), f"ESMR, same window ({len(inside)} days)", 0.0

    gap = timedelta(days=POLE_ESMR_MAX_GAP_DAYS)
    before = [d for d in keys if w0 - gap <= dt[d] < w0][-POLE_ESMR_SIDE_DAYS:]
    after = [d for d in keys if w1 < dt[d] <= w1 + gap][:POLE_ESMR_SIDE_DAYS]
    sides = []
    for side in (before, after):
        if side:
            t = dt[side[0]] + (dt[side[-1]] - dt[side[0]]) / 2
            sides.append((abs((t - mid).days), _esmr_mean(side, esmr, loader), f"{side[0]}-{side[-1]}"))
    if not sides:
        return None, f"no ESMR within {POLE_ESMR_MAX_GAP_DAYS} days", np.nan
    if len(sides) == 1:
        return sides[0][1], f"ESMR {sides[0][2]} (one side only)", float(sides[0][0])
    (db, B, lb), (da, A, la) = sides
    wb = da / max(db + da, 1)                       # nearer side weighs more
    E = np.where(np.isnan(B), A, np.where(np.isnan(A), B, wb * B + (1 - wb) * A))
    return (E.astype(np.float32), f"ESMR interpolated in time: {lb} (w={wb:.2f}) and {la} (w={1 - wb:.2f})",
            float(wb * db + (1 - wb) * da))


def esmr_without(esmr, pk, days_before, days_after):
    """ESMR file dict with the days from days_before before the window to days_after
    after it removed: a simulated hot-load outage around period pk."""
    w0 = datetime.strptime(pk, "%Y%m%d")
    lo, hi = w0 - timedelta(days=int(days_before)), w0 + timedelta(days=PERIOD_DAYS - 1 + int(days_after))
    return {d: f for d, f in esmr.items() if not lo <= datetime.strptime(d, "%Y%m%d") <= hi}


def esmr_context(pk, esmr):
    """(ESMR field for the window [%], distance in days) as used by the network."""
    field, _, dist = esmr_time_field(pk, esmr)
    return field, dist

In [ ]:
# =============================================================================
# 2b2. ESMR intercalibrated to NEMS: the ESMR used to fill NEMS gaps
# =============================================================================
# Wherever ESMR exists, NEMS gaps are filled with ESMR converted to what NEMS would
# have retrieved, so that the 14-day product has one calibration. The conversion is
# fitted on the same-day overlap of the two instruments (Dec 1972 - Feb 1973, June and
# October 1973) in two steps:
#   1. spatial degradation: ESMR is smoothed with a Gaussian of ESMR_FILL_SIGMA_PX cells,
#      standing in for the much larger NEMS footprint and gridding;
#   2. value mapping: the mean NEMS SIC as a monotone (isotonic) function of the smoothed
#      ESMR SIC, fitted per hemisphere and per overlap block. A day uses the mapping of
#      the block nearest in time; its uncertainty is the ESMR total standard error in
#      quadrature with the spread of the three block mappings at that value.
# The raw ESMR record is still what every comparison in Part 5 uses.
from functools import lru_cache
from sklearn.isotonic import IsotonicRegression

ESMR_FILL_SIGMA_PX = 4                          # 100 km; chosen by the cross-validation below
CAL_BLOCKS = {"Dec-Feb": (12, 1, 2), "Jun": (6,), "Oct": (10,)}
CAL_BLOCK_CENTRE = {"Dec-Feb": datetime(1973, 1, 22), "Jun": datetime(1973, 6, 23), "Oct": datetime(1973, 10, 23)}
CAL_X = np.arange(0.0, 100.5, 0.5)
_CAL_BINS = np.arange(0.0, 102.5, 2.5)


def _smooth_valid(a, sig, valid):
    w = gaussian_filter(valid.astype(np.float32), sig, mode="constant")
    s = gaussian_filter(np.where(valid, a, 0).astype(np.float32), sig, mode="constant")
    return np.where(valid & (w > 0.3), s / np.maximum(w, 1e-6), np.nan).astype(np.float32)


def _fit_iso(x, y):
    idx = np.clip(np.digitize(x, _CAL_BINS) - 1, 0, len(_CAL_BINS) - 2)
    cnt = np.bincount(idx, minlength=len(_CAL_BINS) - 1)
    sm = np.bincount(idx, weights=y, minlength=len(_CAL_BINS) - 1)
    ok = cnt > 50
    iso = IsotonicRegression(y_min=0, y_max=100, increasing=True, out_of_bounds="clip")
    iso.fit((_CAL_BINS[:-1] + 1.25)[ok], (sm / np.maximum(cnt, 1))[ok], sample_weight=cnt[ok])
    return iso


def _block_of(day):
    m = int(day[4:6])
    return next((b for b, ms in CAL_BLOCKS.items() if m in ms), None)


CAL_PAIRS, CAL_MAP, CAL_SPREAD, CAL_CV = {}, {}, {}, {}
for hemi, nems, esmr in (("N", nh_nems, nh_esmr), ("S", sh_nems, sh_esmr)):
    ocean = OCEAN[hemi]
    rows = []
    for d in sorted(set(nems) & set(esmr)):
        e = load_ice(esmr[d])
        if np.isfinite(e[ocean]).mean() < 0.05:
            continue
        n = load_ice(nems[d])
        es = _smooth_valid(e, ESMR_FILL_SIGMA_PX, ocean & np.isfinite(e))
        m = ocean & np.isfinite(n) & np.isfinite(es)
        rows.append(pd.DataFrame(dict(block=_block_of(d), n=n[m], es=es[m], e=e[m],
                                      ice=((n >= SIE_THRESHOLD) | (e >= SIE_THRESHOLD))[m])))
    df = pd.concat(rows, ignore_index=True)
    CAL_PAIRS[hemi] = df
    CAL_MAP[hemi] = {b: _fit_iso(df.es[df.block == b].values, df.n[df.block == b].values).predict(CAL_X)
                     for b in CAL_BLOCKS}
    CAL_SPREAD[hemi] = np.std(np.stack(list(CAL_MAP[hemi].values())), axis=0)
    # Leave-one-block-out cross-validation over the cells holding >= 30 % ice in either
    # product: how well a mapping fitted on the other blocks converts the held-out block.
    cv = []
    for hold in CAL_BLOCKS:
        tr, te = df[df.block != hold], df[(df.block == hold) & df.ice]
        pred = _fit_iso(tr.es.values, tr.n.values).predict(te.es.values)
        cv.append(dict(block=hold, n=len(te), raw_bias=float((te.n - te.e).mean()),
                       raw_rms=float(np.sqrt(((te.n - te.e) ** 2).mean())),
                       cal_bias=float((te.n.values - pred).mean()), cal_rms=float(np.sqrt(((te.n.values - pred) ** 2).mean()))))
    cv = pd.DataFrame(cv)
    w = cv.n / cv.n.sum()
    CAL_CV[hemi] = dict(table=cv, raw_bias=float((w * cv.raw_bias).sum()), raw_rms=float(np.sqrt((w * cv.raw_rms ** 2).sum())),
                        cal_bias=float((w * cv.cal_bias).sum()), cal_rms=float(np.sqrt((w * cv.cal_rms ** 2).sum())))
    print(f"[{hemi}H] ESMR -> NEMS, cells >= 30 % ice, leave-one-block-out: raw ESMR bias {CAL_CV[hemi]['raw_bias']:+.2f} / "
          f"RMS {CAL_CV[hemi]['raw_rms']:.2f} pp  ->  calibrated bias {CAL_CV[hemi]['cal_bias']:+.2f} / RMS {CAL_CV[hemi]['cal_rms']:.2f} pp")
    print(cv.round(2).to_string(index=False))
    print("  mapping (smoothed ESMR -> NEMS) at 50 / 70 / 90 / 100 %: "
          + "; ".join(f"{b} " + "/".join(f"{np.interp(v, CAL_X, CAL_MAP[hemi][b]):.0f}" for v in (50, 70, 90, 100)) for b in CAL_BLOCKS))
pd.concat([v["table"].assign(hemi=h) for h, v in CAL_CV.items()]).to_csv(VAL_DIR / "esmr_calibration_cv.csv", index=False)


def _hemi_of_path(path):
    name = Path(path).name.upper()
    return "N" if "_NH" in name else "S"


def _nearest_block(day):
    t = datetime.strptime(day, "%Y%m%d")
    return min(CAL_BLOCK_CENTRE, key=lambda b: min(abs((t - CAL_BLOCK_CENTRE[b].replace(year=y)).days) for y in (1972, 1973, 1974)))


@lru_cache(maxsize=None)
def _esmr_fill_cached(path):
    hemi = _hemi_of_path(path)
    e = load_ice(path)
    ocean = OCEAN[hemi]
    es = _smooth_valid(e, ESMR_FILL_SIGMA_PX, ocean & np.isfinite(e))
    b = _nearest_block(file_date(path))
    sic = np.where(np.isfinite(es), np.interp(np.nan_to_num(es), CAL_X, CAL_MAP[hemi][b]), np.nan).astype(np.float32)
    u_map = np.where(np.isfinite(es), np.interp(np.nan_to_num(es), CAL_X, CAL_SPREAD[hemi]), np.nan) / 100.0
    u_esmr = _smooth_valid(load_unc(path), ESMR_FILL_SIGMA_PX, ocean & np.isfinite(load_unc(path)))
    unc = np.hypot(np.nan_to_num(u_esmr), np.nan_to_num(u_map)).astype(np.float32)
    return sic, np.where(np.isfinite(sic), unc, np.nan).astype(np.float32)


def load_esmr_fill(path):
    """ESMR SIC [%] converted to NEMS (smoothed and mapped); used wherever ESMR fills NEMS."""
    return _esmr_fill_cached(str(path))[0].copy()


def load_esmr_fill_unc(path):
    """Uncertainty of load_esmr_fill as a fraction: ESMR standard error and conversion spread."""
    return _esmr_fill_cached(str(path))[1].copy()

In [ ]:
# =============================================================================
# 2c. Graph construction (Eq. node_features)
# =============================================================================
def pool_block(arr, r, c, B):
    patch = arr[r:r+B, c:c+B]
    v = patch[~np.isnan(patch)]
    return (float(v.mean()) if len(v) else 0.0,
            float(v.std()) if len(v) > 1 else 0.0,
            len(v) / patch.size)


_edge_cache = {}


def get_edge_index(nR, nC):
    if (nR, nC) not in _edge_cache:
        src, dst = [], []
        for i in range(nR):
            for j in range(nC):
                u = i * nC + j
                for di in (-1, 0, 1):
                    for dj in (-1, 0, 1):
                        if di == dj == 0:
                            continue
                        ni, nj = i + di, j + dj
                        if 0 <= ni < nR and 0 <= nj < nC:
                            src.append(ni * nC + nj)
                            dst.append(u)
        _edge_cache[(nR, nC)] = torch.tensor([src, dst], dtype=torch.long)
    return _edge_cache[(nR, nC)]


def _hide_node_features(node_feats, node_idx, um_fill):
    """
    Make masked training nodes look exactly like true-gap nodes at
    inference: NEMS SIC mean/std = 0, valid_day_frac = 0, and uncertainty
    set to the graph-typical fill value (a true gap has no daily
    uncertainty, so its unc_mean pixels carry the global-mean fill).
    """
    node_feats[node_idx, 0] = 0.0       # nems_sic_mean
    node_feats[node_idx, 1] = 0.0       # nems_sic_std
    node_feats[node_idx, 2] = 0.0       # valid_day_frac
    node_feats[node_idx, 3] = um_fill   # unc_mean -> gap-typical value


def make_graph(comp, day_count, sic_std, esmr_mask, esmr_ph_mask,
               unc_mean, unc_std, esmr_sic_mean, esmr_sic_std, em_vfrac,
               boundary_prior, nems_sic_target=None,
               training=True, mask_ratio=MASK_RATIO, block_size=BLOCK_SIZE):
    """
    Graph of B x B super-pixel nodes with 8-neighbour edges (Eq. node_features).

    training=False (used for training inputs in Part 3 and for inference) builds the
    features only. training=True is used only as a fallback for a degenerate period
    with no eligible hold-out blocks: it hides all NEMS nodes when ESMR is available,
    otherwise a random MASK_RATIO of them.

    Node features (N_FEATURES = 11):
      0 nems_sic_mean, 1 nems_sic_std, 2 valid_day_frac, 3 unc_mean,
      4 boundary_prior, 5 esmr_sic_mean, 6 esmr_sic_std, 7 esmr_valid_frac,
      8 esmr_fill_flag, 9 supervised_flag, 10 esmr_ph_flag
    """
    B = block_size
    H, W = comp.shape
    nR, nC = H // B, W // B
    N = nR * nC
    max_days = max(int(day_count.max()), 1)

    has_esmr = (esmr_sic_mean is not None and not np.all(np.isnan(esmr_sic_mean)))
    supervised = has_esmr
    sup_flag = 1.0 if supervised else 0.0

    if esmr_sic_mean is not None:
        esmr_real = (~np.isnan(esmr_sic_mean)).astype(np.float32)
    else:
        esmr_real = np.zeros((H, W), dtype=np.float32)

    # ESMR SIC feature: NaN -> boundary prior (not 0, which would read as
    # "ESMR says open water" and drive interior under-estimation).
    if esmr_sic_mean is not None:
        esmr_filled = np.where(np.isnan(esmr_sic_mean), boundary_prior * 100.0, esmr_sic_mean)
        esmr_n = (esmr_filled / 100.0).astype(np.float32)
    else:
        esmr_n = boundary_prior.copy().astype(np.float32)

    esmr_s = (np.where(np.isnan(esmr_sic_std), 0.0, esmr_sic_std) / 100.0
              if esmr_sic_std is not None else np.zeros((H, W), dtype=np.float32))

    node_feats = np.zeros((N, N_FEATURES), dtype=np.float32)
    node_y = np.zeros(N, dtype=np.float32)
    node_valid = np.zeros(N, dtype=np.float32)
    node_nems_valid = np.zeros(N, dtype=np.float32)
    node_esmr_valid = np.zeros(N, dtype=np.float32)
    zero_ctx = np.zeros(N, dtype=np.float32)
    prior_y = np.zeros(N, dtype=np.float32)

    for i in range(nR):
        for j in range(nC):
            idx = i * nC + j
            r, c = i * B, j * B

            sic_m, sic_s, sic_f = pool_block(comp / 100.0, r, c, B)
            dc_m = float(day_count[r:r+B, c:c+B].mean()) / max_days
            um, _, _ = pool_block(unc_mean, r, c, B)
            pr, _, _ = pool_block(boundary_prior, r, c, B)

            es_m, _, _ = pool_block(esmr_n, r, c, B)
            es_s, _, _ = pool_block(esmr_s, r, c, B)
            es_real_frac = float(esmr_real[r:r+B, c:c+B].mean())
            eph_frac = float(esmr_ph_mask[r:r+B, c:c+B].astype(np.float32).mean())
            ef_any = 1.0 if (es_real_frac > 0 or eph_frac > 0) else 0.0

            node_feats[idx] = [sic_m, sic_s, dc_m, um, pr, es_m, es_s,
                                float(em_vfrac), ef_any, sup_flag, eph_frac]
            prior_y[idx] = pr

            if sic_f > 0 and eph_frac == 0:
                node_y[idx] = sic_m
                node_valid[idx] = 1.0
                node_nems_valid[idx] = 1.0
            elif eph_frac > 0:
                node_y[idx] = es_m
                node_valid[idx] = 1.0

            if es_real_frac > 0 or eph_frac > 0:
                node_esmr_valid[idx] = 1.0
            if node_valid[idx] == 0 and node_esmr_valid[idx] == 0:
                zero_ctx[idx] = 1.0

    nems_nodes = node_nems_valid == 1.0
    um_fill = float(np.median(node_feats[nems_nodes, 3])) if nems_nodes.any() else 0.05

    node_mask = np.zeros(N, dtype=np.float32)
    if training:
        if supervised:
            nems_vi = np.where(node_nems_valid == 1.0)[0]
            if len(nems_vi):
                node_mask[nems_vi] = 1.0
                _hide_node_features(node_feats, nems_vi, um_fill)
        else:
            nems_vi = np.where(node_nems_valid == 1.0)[0]
            if len(nems_vi):
                mi = np.random.choice(nems_vi, max(1, int(len(nems_vi) * mask_ratio)), replace=False)
                node_mask[mi] = 1.0
                _hide_node_features(node_feats, mi, um_fill)

    return Data(
        x=torch.from_numpy(node_feats),
        edge_index=get_edge_index(nR, nC),
        y=torch.from_numpy(node_y).unsqueeze(1),
        mask=torch.from_numpy(node_mask).unsqueeze(1),
        valid=torch.from_numpy(node_valid).unsqueeze(1),
        zero_ctx=torch.from_numpy(zero_ctx).unsqueeze(1),
        prior_y=torch.from_numpy(prior_y).unsqueeze(1),
    )

In [ ]:
# =============================================================================
# 2d. Residual GraphSAGE (Eq. sage), loss (Eq. loss) and inpainting helpers
# =============================================================================
class ResGraphSAGE(nn.Module):
    """Residual GraphSAGE. With residual=True the head returns a bounded correction
    (tanh x MAX_CORRECTION) to the baseline; otherwise a SIC fraction (sigmoid)."""

    def __init__(self, in_ch, hidden=HIDDEN_DIM, layers=NUM_LAYERS, residual=RESIDUAL_ON_BASELINE):
        super().__init__()
        self.convs = nn.ModuleList()
        self.projs = nn.ModuleList()
        self.norms = nn.ModuleList()
        ic = in_ch
        for _ in range(layers):
            self.convs.append(SAGEConv(ic, hidden))
            self.projs.append(nn.Linear(ic, hidden, bias=False) if ic != hidden else nn.Identity())
            self.norms.append(nn.LayerNorm(hidden))
            ic = hidden
        self.residual = residual
        n_out = N_BASES + 1 if residual else 1
        self.head = nn.Sequential(nn.Linear(hidden, hidden // 2), nn.ReLU(), nn.Linear(hidden // 2, n_out))

    def forward(self, x, ei):
        """residual: (mixing weights (N, N_BASES), correction (N, 1)); else SIC fraction (N, 1)."""
        for conv, proj, norm in zip(self.convs, self.projs, self.norms):
            x = norm(F.relu(conv(x, ei)) + proj(x))
        out = self.head(x)
        if not self.residual:
            return torch.sigmoid(out)
        return torch.softmax(out[:, :N_BASES], dim=1), torch.tanh(out[:, N_BASES:]) * MAX_CORRECTION


def node_prediction(model, b):
    """Block-mean SIC fraction predicted for each node of a batch."""
    if not model.residual:
        return model(b.x, b.edge_index)
    w, delta = model(b.x, b.edge_index)
    return (w * b.bases).sum(dim=1, keepdim=True) + delta


def masked_mae(pred, target, mask):
    return (torch.abs(pred - target) * mask).sum() / (mask.sum() + 1e-8)


def masked_mae_with_physics(pred, target, mask, zero_ctx, prior_y,
                             physics_weight=PHYSICS_PENALTY_WEIGHT):
    """At zero-context nodes (no NEMS, no ESMR anywhere in the block), the
    prediction is anchored toward the boundary prior -- the nearest valid
    SIC value -- which regularises deep-void predictions without a
    hard-coded concentration floor that would be wrong near the ice edge."""
    mae_term = masked_mae(pred, target, mask)
    if zero_ctx.sum() > 0:
        # residual model: prior_y is the baseline, so this pulls the correction to zero
        phys_term = (torch.abs(pred - prior_y) * zero_ctx).sum() / (zero_ctx.sum() + 1e-8)
    else:
        phys_term = torch.tensor(0.0, device=pred.device)
    return mae_term + physics_weight * phys_term


def _edt_inpaint(arr, valid_mask):
    """Nearest-valid EDT fill of all NaN pixels -- guarantees zero holes."""
    filled = arr.copy()
    nan_mask = ~valid_mask
    if not nan_mask.any():
        return filled
    _, idx = distance_transform_edt(nan_mask, return_indices=True)
    filled[nan_mask] = arr[idx[0][nan_mask], idx[1][nan_mask]]
    return filled


def _neighbourhood_mean(arr, radius=5):
    """Mean of valid (non-NaN) pixels within `radius` at each pixel."""
    valid = (~np.isnan(arr)).astype(np.float32)
    arr_z = np.where(np.isnan(arr), 0.0, arr)
    size = radius * 2 + 1
    sum_v = uniform_filter(arr_z, size=size, mode='nearest') * size ** 2
    cnt_v = uniform_filter(valid, size=size, mode='nearest') * size ** 2
    with np.errstate(invalid='ignore'):
        mean_v = np.where(cnt_v > 0, sum_v / cnt_v, np.nan)
    return mean_v.astype(np.float32)

## Part 3 — Masked-reconstruction training and hold-out validation

Blocks of the composite are withheld *before* the graph is built, and every derived field is rebuilt from what survives. Targets come from the intact composite (Methods, "Masked-reconstruction training"). Periods are split 8/2/3 in interleaved order. The test periods are scored against four baseline interpolators at gap sizes of 300–2400 km. These are the numbers in Table 2 of the paper.

In [ ]:
# =============================================================================
# 3a. Post-processing of node predictions, composite cache, temporal features
# =============================================================================
def postprocess(pp, comp, unobs, fringe_src):
    """Grid-level clean-up of the bilinearly resampled node predictions [%].

    1  discard predictions within FRINGE_DILATION_PX of never-observed ocean
    2  neighbourhood clamp: reconstructed values < 30 % inside ice >= 30 % -> local mean
    3  nearest-valid fill so no hole survives
    4  13 x 13 median filter over the reconstructed region (de-blocking)
    5  interior-pack clamp: reconstructed blobs > 12 pp below a >= 85 % pack -> local mean
    """
    pp = pp.copy()
    r = FRINGE_DILATION_PX
    fringe = binary_dilation(fringe_src, structure=np.ones((2 * r + 1, 2 * r + 1), bool))
    pp[fringe] = np.nan

    fill_px = np.isnan(comp) & ~unobs & ~fringe
    neigh = _neighbourhood_mean(np.where(fill_px, pp, comp), radius=5)
    low = fill_px & (pp < GNN_CLAMP_THRESHOLD) & (neigh >= GNN_CLAMP_THRESHOLD)
    pp[low] = neigh[low]

    ok = fill_px & ~np.isnan(pp)
    merged = comp.copy()
    merged[ok] = pp[ok]
    pp = _edt_inpaint(merged, (~np.isnan(comp) & ~unobs) | ok)

    fill_region = np.isnan(comp) & ~unobs
    if fill_region.any():
        sm = median_filter(pp, size=BLOCK_SIZE + 1, mode="nearest")
        pp[fill_region] = sm[fill_region]
        wide = _neighbourhood_mean(np.where(fill_region, pp, comp), radius=15)
        blob = fill_region & (wide >= 85.0) & (pp < wide - 12.0)
        if blob.any():
            pp[blob] = wide[blob]
            blur = gaussian_filter(pp, sigma=3.0)
            soft = binary_dilation(blob, structure=np.ones((7, 7), bool)) & fill_region
            pp[soft] = 0.5 * pp[soft] + 0.5 * blur[soft]
    return pp


class CompositeCache:
    """Memoised composite SIC per period, for the previous/next-period features."""

    def __init__(self, periods, nems, esmr, unobs):
        self.periods, self.nems, self.esmr, self.unobs = periods, nems, esmr, unobs
        self.keys = sorted(periods)
        self._c = {}

    def sic(self, pk):
        if pk not in self._c:
            self._c[pk] = build_composite(self.periods[pk], self.nems, self.esmr, self.unobs)[0]
        return self._c[pk]

    def neighbours(self, pk):
        """Composites of the preceding and following NEMS periods. These are the
        adjacent periods WITH data, which can be up to 8 weeks away (e.g. 17-30 June)."""
        i = self.keys.index(pk)
        prev = self.sic(self.keys[i - 1]) if i > 0 else None
        nxt = self.sic(self.keys[i + 1]) if i < len(self.keys) - 1 else None
        return prev, nxt


def temporal_block_features(prev_sic, next_sic, shape):
    """Block mean [0-1] and defined fraction of the neighbouring composites -> (N, 4)."""
    B = BLOCK_SIZE
    nR, nC = shape[0] // B, shape[1] // B
    out = np.zeros((nR * nC, N_TEMPORAL), np.float32)
    for slot, field in ((0, prev_sic), (2, next_sic)):
        if field is None:
            continue
        f = field[:nR * B, :nC * B].reshape(nR, B, nC, B).transpose(0, 2, 1, 3).reshape(nR * nC, B * B)
        valid = ~np.isnan(f)
        n = valid.sum(axis=1)
        mean = np.where(n > 0, np.where(valid, f, 0).sum(axis=1) / np.maximum(n, 1), 0.0)
        out[:, slot] = mean / 100.0
        out[:, slot + 1] = n / (B * B)
    return out


def baseline_fields(comp, unobs, prev_sic, next_sic, esmr_field=None):
    """Pixel-level baselines [%], stacked (N_BASES, H, W): nearest valid value; mean of
    the neighbouring periods (nearest value where neither is defined); their blend, the
    temporal weight growing as local observation density falls; and ESMR for the window
    (interpolated in time across an outage; nearest value where undefined). Also returns
    the density (fraction of observed cells within 5 cells)."""
    near = _boundary_prior(comp) * 100.0
    dens = np.nan_to_num(_neighbourhood_mean((~np.isnan(comp) & ~unobs).astype(np.float32), 5))
    tint = np.full(comp.shape, np.nan, np.float32)
    stack = [f for f in (prev_sic, next_sic) if f is not None]
    if stack:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            tint = np.nanmean(np.stack(stack, 0), axis=0)
    tint = np.where(np.isnan(tint), near, tint)
    w = np.clip(1.0 - dens, 0.0, 1.0)
    blend = (1 - w) * near + w * tint
    esmr_b = near if esmr_field is None else np.where(np.isfinite(esmr_field), esmr_field, near)
    return np.stack([near, tint, blend, esmr_b]).astype(np.float32), dens.astype(np.float32)


def block_means(field, B=None):
    """NaN-aware B x B block means -> (N,)."""
    B = B or BLOCK_SIZE
    nR, nC = field.shape[0] // B, field.shape[1] // B
    f = field[:nR * B, :nC * B].reshape(nR, B, nC, B).transpose(0, 2, 1, 3).reshape(nR * nC, B * B)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        return np.nan_to_num(np.nanmean(f, axis=1)).astype(np.float32)


def graph_inputs(comp, dc, std, em, em_ph, um, us, es_m, es_s, es_vf, bp, unobs, prev_sic, next_sic,
                 esmr_ctx=None, training=False):
    """Graph with spatial, temporal, baseline and ESMR node features, plus the baseline
    fields. esmr_ctx = (ESMR field for the window [%], distance in days) or None."""
    g = make_graph(comp, dc, std, em, em_ph, um, us, es_m, es_s, es_vf, bp, training=training)
    g = augment_graph(g, prev_sic, next_sic, comp.shape)
    e_field, e_dist = esmr_ctx if esmr_ctx is not None else (None, np.nan)
    bases, dens = baseline_fields(comp, unobs, prev_sic, next_sic, e_field)
    bmeans = np.stack([block_means(b) for b in bases], 1) / 100.0            # (N, N_BASES)
    dist = np.full(g.num_nodes, 1.0 if not np.isfinite(e_dist) else min(e_dist / 60.0, 1.0), np.float32)
    # Distance from each cell to the nearest defined composite cell: unlike the local
    # density, it tells a 600 km gap from a 2400 km one, which decides whether the
    # nearest value or the temporal interpolation is to be trusted.
    gap_d = np.minimum(distance_transform_edt(np.isnan(comp)) * 25.0 / GAP_DIST_SCALE_KM, 1.0).astype(np.float32)
    extra = np.stack([bmeans[:, 2], block_means(dens), bmeans[:, 3], dist, block_means(gap_d)], 1).astype(np.float32)
    g.x = torch.cat([g.x, torch.from_numpy(extra)], dim=1)
    g.bases = torch.from_numpy(bmeans.astype(np.float32))
    if RESIDUAL_ON_BASELINE:
        g.prior_y = g.bases[:, 2:3].clone()   # regularise deep voids toward the blend
    return g, bases


def augment_graph(g, prev_sic, next_sic, shape):
    g.x = torch.cat([g.x, torch.from_numpy(temporal_block_features(prev_sic, next_sic, shape))], dim=1)
    return g

In [ ]:
# =============================================================================
# 3b. Training graphs with composite-level block hold-out
# =============================================================================
def _hole_composite(comp, dc, std, um, esmr_mask, unobs, k, hold_frac, rng):
    """Withhold k x k groups of blocks that contain NEMS retrievals and rebuild every
    derived field from what survives (k = 1 for training, k > 1 in the gap sweep)."""
    B = BLOCK_SIZE
    eligible = ~np.isnan(comp) & ~esmr_mask & ~unobs
    nR, nC = comp.shape[0] // B, comp.shape[1] // B
    cand = [(bi, bj) for bi in range(0, nR - k + 1, k) for bj in range(0, nC - k + 1, k)
            if eligible[bi * B:(bi + k) * B, bj * B:(bj + k) * B].any()]
    if not cand:
        return None
    sel = [cand[i] for i in rng.choice(len(cand), size=max(1, int(len(cand) * hold_frac)), replace=False)]
    hold = np.zeros(comp.shape, bool)
    for bi, bj in sel:
        hold[bi * B:(bi + k) * B, bj * B:(bj + k) * B] = True
    hold &= eligible

    comp_h = comp.copy(); comp_h[hold] = np.nan
    dc_h = dc.copy(); dc_h[hold] = 0
    std_h = std.copy(); std_h[hold] = np.nan
    um_h = um.copy()
    um_h[hold] = float(np.nanmean(um)) if np.isfinite(np.nanmean(um)) else 0.05
    return comp_h, dc_h, std_h, um_h, _boundary_prior(comp_h), hold, sel


def make_training_graph(comp, dc, std, em, em_ph, um, us, es_m, es_s, es_vf, unobs,
                        prev_sic, next_sic, hold_frac, rng, k=1, esmr_ctx=None):
    """Inputs from the holed composite (k x k-block holes); targets (block means) from the intact one."""
    B = BLOCK_SIZE
    holed = _hole_composite(comp, dc, std, um, em, unobs, k, hold_frac, rng)
    if holed is None:
        return None
    comp_h, dc_h, std_h, um_h, bp_h, hold, sel = holed
    g, bases = graph_inputs(comp_h, dc_h, std_h, em, em_ph, um_h, us, es_m, es_s, es_vf, bp_h, unobs,
                            prev_sic, next_sic, esmr_ctx)
    nC = comp.shape[1] // B
    y = np.zeros(g.num_nodes, np.float32)
    m = np.zeros(g.num_nodes, np.float32)
    bmeans = g.bases.numpy().copy()
    blocks = [(bi + di, bj + dj) for bi, bj in sel for di in range(k) for dj in range(k)]
    for bi, bj in blocks:
        sl = (slice(bi * B, (bi + 1) * B), slice(bj * B, (bj + 1) * B))
        cells = hold[sl]
        if cells.any():
            idx = bi * nC + bj
            y[idx] = comp[sl][cells].mean() / 100.0
            bmeans[idx] = bases[:, sl[0], sl[1]][:, cells].mean(axis=1) / 100.0   # over the withheld cells
            m[idx] = 1.0
    g.y = torch.from_numpy(y).unsqueeze(1)
    g.mask = torch.from_numpy(m).unsqueeze(1)
    g.bases = torch.from_numpy(bmeans)
    return g


class HoldOutDataset(torch.utils.data.Dataset):
    """Training (fresh hold-out every epoch; with probability OUTAGE_SIM_PROB the period's
    ESMR is removed within a random margin, simulating a hot-load outage) or evaluation
    (fixed hold-out; outage_days > 0 scores each period as a simulated outage)."""

    def __init__(self, keys, periods, nems, esmr, unobs, hold_frac, fixed=False, seed=SEED, outage_days=0):
        self.keys = list(keys)
        self.periods, self.nems, self.esmr, self.unobs = periods, nems, esmr, unobs
        self.hold_frac, self.fixed, self.seed, self.outage_days = hold_frac, fixed, seed, outage_days
        self.rng = np.random.default_rng(seed)
        self.cache = CompositeCache(periods, nems, esmr, unobs)
        self._normal = {}                   # composite and ESMR context with the real ESMR

    def __len__(self):
        return len(self.keys)

    def _inputs(self, pk, esmr):
        return (build_composite(self.periods[pk], self.nems, esmr, self.unobs), esmr_context(pk, esmr))

    def __getitem__(self, i):
        pk = self.keys[i]
        rng = np.random.default_rng(self.seed + i) if self.fixed else self.rng
        if self.fixed and self.outage_days:
            esmr_v = esmr_without(self.esmr, pk, self.outage_days, self.outage_days)
            composite, ctx = self._inputs(pk, esmr_v)
        elif not self.fixed and rng.random() < OUTAGE_SIM_PROB:
            gb, ga = rng.integers(0, OUTAGE_SIM_MAX_DAYS + 1, size=2)
            composite, ctx = self._inputs(pk, esmr_without(self.esmr, pk, gb, ga))
        else:
            if pk not in self._normal:
                self._normal[pk] = self._inputs(pk, self.esmr)
            composite, ctx = self._normal[pk]
        comp, dc, std, em, em_ph, um, us, es_m, es_s, es_vf, bp, _, _ = composite
        prev, nxt = self.cache.neighbours(pk)
        k = int(rng.choice(TRAIN_HOLD_SIZES))
        g = make_training_graph(comp, dc, std, em, em_ph, um, us, es_m, es_s, es_vf, self.unobs,
                                prev, nxt, self.hold_frac, rng, k=k, esmr_ctx=ctx)
        if g is None:                       # no eligible block in this period: nothing to score
            g, _ = graph_inputs(comp, dc, std, em, em_ph, um, us, es_m, es_s, es_vf, bp, self.unobs,
                                prev, nxt, ctx)
            g.y = torch.zeros(g.num_nodes, 1)
            g.mask = torch.zeros(g.num_nodes, 1)
        return g


def interleaved_split(keys, test_every=4, val_every=7):
    """8/2/3 for 13 periods, each subset spanning the seasonal cycle."""
    keys = list(keys)
    test = [k for i, k in enumerate(keys) if i % test_every == 2]
    val = [k for i, k in enumerate(keys) if i % val_every == 5 and k not in test]
    return [k for k in keys if k not in test + val], val, test

In [ ]:
# =============================================================================
# 3c. Training, prediction and hold-out validation against baseline interpolators
# =============================================================================
def train_model(hemi, train_k, val_k, periods, nems, esmr, unobs, save_path, epochs=NUM_EPOCHS):
    seed_everything(SEED)
    tl = PyGDataLoader(HoldOutDataset(train_k, periods, nems, esmr, unobs, HOLD_FRAC_TRAIN),
                       batch_size=BATCH_SIZE, shuffle=True)
    vl = PyGDataLoader(torch.utils.data.ConcatDataset([
        HoldOutDataset(val_k, periods, nems, esmr, unobs, HOLD_FRAC_EVAL, fixed=True),
        HoldOutDataset(val_k, periods, nems, esmr, unobs, HOLD_FRAC_EVAL, fixed=True, outage_days=VAL_OUTAGE_DAYS)]),
        batch_size=BATCH_SIZE, shuffle=False)
    model = ResGraphSAGE(in_ch=N_FEATURES_V4, hidden=HIDDEN_DIM, layers=NUM_LAYERS).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=5, factor=0.5, min_lr=1e-5)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"[{hemi}] parameters {n_params:,}; train {train_k}; val {val_k}")

    best, hist = np.inf, []
    for ep in range(epochs):
        model.train(); t_loss = 0.0
        for b in tl:
            b = b.to(DEVICE); opt.zero_grad()
            loss = masked_mae_with_physics(node_prediction(model, b), b.y, b.mask, b.zero_ctx, b.prior_y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); t_loss += loss.item()
        model.eval(); v_loss = 0.0
        with torch.no_grad():
            for b in vl:
                b = b.to(DEVICE)
                v_loss += masked_mae(node_prediction(model, b), b.y, b.mask).item()
        t_loss /= max(len(tl), 1); v_loss /= max(len(vl), 1)
        hist.append((t_loss, v_loss))
        sched.step(v_loss)
        if v_loss < best:
            best = v_loss
            torch.save(model.state_dict(), save_path)
        if ep % 10 == 0 or ep == epochs - 1:
            print(f"  epoch {ep + 1:3d}  train {t_loss:.4f}  val {v_loss:.4f}  lr {opt.param_groups[0]['lr']:.1e}")
    model.load_state_dict(torch.load(save_path, map_location=DEVICE))
    model.eval()
    print(f"  best validation MAE {best:.4f}")
    return model, hist, n_params


def predict(comp, dc, std, em, em_ph, um, us, es_m, es_s, es_vf, bp, model, masks, prev, nxt, post=None,
            esmr_ctx=None):
    """Grid prediction [%]. post: "none" = raw field; "light" = merge into the composite
    and fill what is left from the nearest valid cell; "full" = postprocess();
    "fade:L" = postprocess() next to observations, fading to the raw field over a
    distance L [km]. The default (None) is the variant selected on the validation
    periods (POST_MODE)."""
    post = POST_MODE if post is None and "POST_MODE" in globals() else post
    B = BLOCK_SIZE
    H, W = comp.shape
    g, bases = graph_inputs(comp, dc, std, em, em_ph, um, us, es_m, es_s, es_vf, bp, masks["unobs"], prev, nxt,
                            esmr_ctx)
    g = g.to(DEVICE)
    up = lambda a: zoom(a.reshape(H // B, W // B), zoom=B, order=1)[:H, :W]
    with torch.no_grad():
        out = model(g.x, g.edge_index)
    if model.residual:
        w, delta = (t.cpu().numpy() for t in out)
        wg = np.stack([up(w[:, k]) for k in range(N_BASES)])
        wg /= wg.sum(axis=0, keepdims=True)
        raw = np.clip((wg * bases).sum(axis=0) + 100.0 * up(delta[:, 0]), 0, 100)
    else:
        raw = np.clip(100.0 * up(out.squeeze(1).cpu().numpy()), 0, 100)
    if post in (None, "none"):
        return raw
    if post == "full":
        return postprocess(raw, comp, masks["unobs"], masks["fringe_src"])
    if str(post).startswith("fade:"):
        # full post-processing next to observations, fading to the raw field with
        # distance d into the gap: weight exp(-d / L)
        L = float(post.split(":")[1])
        full = postprocess(raw, comp, masks["unobs"], masks["fringe_src"])
        w = np.exp(-distance_transform_edt(np.isnan(comp)) * 25.0 / L)
        out = np.where(np.isnan(comp) & ~masks["unobs"], w * full + (1 - w) * raw, full)
        return np.clip(out, 0, 100)
    fill = np.isnan(comp) & ~masks["unobs"]
    out = comp.copy()
    out[fill] = raw[fill]
    return np.clip(_edt_inpaint(out, ~np.isnan(out)), 0, 100)


def gap_sweep(hemi, test_k, periods, nems, esmr, masks, model, ks=(1, 2, 4, 8), hold_frac=HOLD_FRAC_EVAL,
              post=None, quiet=False, outage_days=0):
    """Withhold k x k blocks (300-2400 km) of NEMS retrievals in the test periods and
    score the network and four baselines on the withheld cells."""
    unobs = masks["unobs"]
    cache = CompositeCache(periods, nems, esmr, unobs)
    rows = []
    for k in ks:
        rng = np.random.default_rng(SEED)
        acc = defaultdict(lambda: ([], []))
        for pk in test_k:
            esmr_v = esmr_without(esmr, pk, outage_days, outage_days) if outage_days else esmr
            comp, dc, std, em, em_ph, um, us, es_m, es_s, es_vf, bp, _, _ = build_composite(
                periods[pk], nems, esmr_v, unobs)
            ctx = esmr_context(pk, esmr_v)
            holed = _hole_composite(comp, dc, std, um, em, unobs, k, hold_frac, rng)
            if holed is None or holed[5].sum() < 100:
                continue
            comp_h, dc_h, std_h, um_h, bp_h, hold, _ = holed
            prev, nxt = cache.neighbours(pk)
            args = (comp_h, dc_h, std_h, em, em_ph, um_h, us, es_m, es_s, es_vf, bp_h, model, masks, prev, nxt)
            near = bp_h * 100.0
            tint = None
            stack = [f for f in (prev, nxt) if f is not None]
            if stack:
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore", RuntimeWarning)
                    tint = np.nanmean(np.stack(stack, 0), axis=0)
            blend = None
            if tint is not None:
                dens = np.nan_to_num(_neighbourhood_mean((~np.isnan(comp_h) & ~unobs).astype(np.float32), 5))
                w = np.clip(1.0 - dens, 0.0, 1.0)
                blend = np.where(np.isnan(tint), near, (1 - w) * near + w * tint)
            e_interp = None if ctx[0] is None else np.where(np.isfinite(ctx[0]), ctx[0], near)
            fields = {"network": predict(*args, post=post, esmr_ctx=ctx),
                      "network (raw)": predict(*args, post="none", esmr_ctx=ctx),
                      "esmr-interp": e_interp, "nearest-valid": near, "persistence": prev, "temporal-interp": tint, "blended": blend}
            truth = comp[hold]
            for name, field in fields.items():
                if field is None:
                    continue
                v = field[hold]
                ok = np.isfinite(v) & np.isfinite(truth)
                acc[name][0].append(truth[ok]); acc[name][1].append(v[ok])
        for name, (ts, ps) in acc.items():
            t, p = np.concatenate(ts), np.concatenate(ps)
            rows.append(dict(hemi=hemi, gap_km=k * BLOCK_SIZE * 25, method=name, n=t.size,
                             mae=mean_absolute_error(t, p), rmse=float(np.sqrt(np.mean((t - p) ** 2))),
                             bias=float(p.mean() - t.mean()), r2=r2_score(t, p)))
    res = pd.DataFrame(rows)
    piv = res.pivot(index="gap_km", columns="method", values="mae")
    base = piv[[c for c in BASELINES if c in piv]].min(axis=1)
    if not quiet:
        print(f"\n{hemi}: MAE [pp] on withheld NEMS cells, periods {test_k}")
        print(piv.round(2).to_string())
        print("network / best baseline:", (piv["network"] / base).round(2).to_dict())
    return res


# Post-processing variant chosen on the VALIDATION periods only; the test periods are
# scored once, with that choice.
POST_OPTIONS = ("none", "light", "full", "fade:50", "fade:100", "fade:200", "fade:400")
GNN, VAL_MAE = {}, {}
for hemi, periods, nems, esmr in (("N", nh_periods, nh_nems, nh_esmr), ("S", sh_periods, sh_nems, sh_esmr)):
    tr, va, te = interleaved_split(list(periods))
    if RETRAIN_GNN or not (MODEL_DIR / f"gnn_{hemi}H.pth").exists():
        model, hist, n_params = train_model(hemi, tr, va, periods, nems, esmr, MASKS[hemi]["unobs"],
                                            MODEL_DIR / f"gnn_{hemi}H.pth")
    else:                                    # reload the trained weights (same code and seed)
        model = ResGraphSAGE(in_ch=N_FEATURES_V4, hidden=HIDDEN_DIM, layers=NUM_LAYERS).to(DEVICE)
        model.load_state_dict(torch.load(MODEL_DIR / f"gnn_{hemi}H.pth", map_location=DEVICE))
        model.eval()
        hist, n_params = [], sum(p.numel() for p in model.parameters())
        print(f"[{hemi}] weights reloaded from {MODEL_DIR / f'gnn_{hemi}H.pth'} ({n_params:,} parameters)")
    # validation score = mean over gap sizes of the normal and the simulated-outage evaluation
    val_mae = {p: np.mean([gap_sweep(hemi, va, periods, nems, esmr, MASKS[hemi], model, post=p, quiet=True,
                                     outage_days=od).query("method == 'network'")["mae"].mean()
                           for od in (0, VAL_OUTAGE_DAYS)]) for p in POST_OPTIONS}
    VAL_MAE[hemi] = val_mae
    print(f"[{hemi}] validation MAE [pp] by post-processing: " + ", ".join(f"{k} {v:.3f}" for k, v in val_mae.items()))
    GNN[hemi] = dict(model=model, hist=hist, n_params=n_params, split=(tr, va, te))
# One post-processing choice for both hemispheres: the lower summed validation MAE
POST_MODE = min(POST_OPTIONS, key=lambda p: VAL_MAE["N"][p] + VAL_MAE["S"][p])
print(f"post-processing chosen on validation: {POST_MODE}")
(VAL_DIR / "post_mode.txt").write_text(POST_MODE)
# Test periods, scored once: as they are, and as simulated outages (ESMR removed
# VAL_OUTAGE_DAYS either side), which is the situation the network is used for.
for hemi, periods, nems, esmr in (("N", nh_periods, nh_nems, nh_esmr), ("S", sh_periods, sh_nems, sh_esmr)):
    sweep = gap_sweep(hemi, GNN[hemi]["split"][2], periods, nems, esmr, MASKS[hemi], GNN[hemi]["model"])
    sweep.to_csv(VAL_DIR / f"gap_sweep_{hemi}H.csv", index=False)
    GNN[hemi]["sweep"] = sweep
    sweep_o = gap_sweep(hemi, GNN[hemi]["split"][2], periods, nems, esmr, MASKS[hemi], GNN[hemi]["model"],
                        outage_days=VAL_OUTAGE_DAYS)
    sweep_o.to_csv(VAL_DIR / f"gap_sweep_outage_{hemi}H.csv", index=False)
    GNN[hemi]["sweep_outage"] = sweep_o

In [ ]:
# =============================================================================
# 3d. Robustness of the comparison in Table 2: random initialisation, and the
#     dependence of the ranking on which periods are withheld
# =============================================================================
# Source of the two robustness figures quoted in Results and discussion. It retrains
# the network from three seeds per hemisphere, so it is off by default; set
# RUN_ROBUSTNESS = True (or NEMS_ROBUSTNESS=1 in the environment) to reproduce them.
import json

RUN_ROBUSTNESS = os.environ.get("NEMS_ROBUSTNESS", "0") == "1"
N_SEEDS_ROBUST = 3

if RUN_ROBUSTNESS:
    SEED_KEEP = SEED
    SPREAD = {}
    for hemi, periods, nems, esmr in (("N", nh_periods, nh_nems, nh_esmr), ("S", sh_periods, sh_nems, sh_esmr)):
        tr, va, te = GNN[hemi]["split"]
        per_seed = []
        for s in range(N_SEEDS_ROBUST):
            globals()["SEED"] = s
            m, _, _ = train_model(hemi, tr, va, periods, nems, esmr, MASKS[hemi]["unobs"],
                                  MODEL_DIR / f"robust_{hemi}_{s}.pth")
            # validation score of this seed, and of every simple method on the same cells:
            # mean over the four gap sizes of the normal and the simulated-outage evaluation
            frames = [gap_sweep(hemi, va, periods, nems, esmr, MASKS[hemi], m, post=POST_MODE,
                                quiet=True, outage_days=od) for od in (0, VAL_OUTAGE_DAYS)]
            mae = pd.concat(frames).groupby("method")["mae"].mean()
            per_seed.append(mae)
            print(f"[{hemi}] seed {s}: network {mae['network']:.3f} pp", flush=True)
        globals()["SEED"] = SEED_KEEP
        net = [m["network"] for m in per_seed]
        simple = {b: float(per_seed[0][b]) for b in BASELINES if b in per_seed[0]}
        best_simple = min(simple, key=simple.get)
        SPREAD[hemi] = dict(network=[float(v) for v in net], spread=float(max(net) - min(net)),
                            simple=simple, best_simple=best_simple)
        print(f"[{hemi}] validation MAE over {N_SEEDS_ROBUST} initialisations: "
              f"{', '.join(f'{v:.3f}' for v in net)} pp -> spread {max(net) - min(net):.2f} pp")
        print(f"[{hemi}] best simple method on the VALIDATION periods: {best_simple} "
              f"{simple[best_simple]:.3f} pp (network, best seed {min(net):.3f} pp); "
              f"on the test periods it is {GNN[hemi]['sweep'].query('method == @best_simple')['mae'].mean():.3f} pp")
    (VAL_DIR / "robustness.json").write_text(json.dumps(SPREAD, indent=1))
    print("\nQuoted in the paper: initialisation spread "
          f"{SPREAD['N']['spread']:.2f} pp (Arctic) and {SPREAD['S']['spread']:.2f} pp (Antarctic); "
          f"persistence on the validation periods {SPREAD['N']['simple'].get('persistence', float('nan')):.2f} pp "
          f"against the network's {min(SPREAD['N']['network']):.2f} pp in the Arctic.")
else:
    print("Robustness check skipped (RUN_ROBUSTNESS = False). "
          "It retrains the network three times per hemisphere.")

## Part 4 — Inference and the 14-day product (Dataset 2)

Each period file carries the gap-filled SIC, the NEMS-only composite (`raw_ice_conc_values`), a per-cell `total_standard_error`, the number of NEMS days per cell (`nems_day_count`), and a `provenance_flag` that separates NEMS retrievals from ESMR fill, network reconstruction and pole-hole fill.

In [ ]:
# =============================================================================
# 4a. Error of ESMR interpolated in time across an outage (pole-hole uncertainty)
# =============================================================================
# During the hot-load outages the Arctic pole hole is filled with ESMR interpolated
# in time from up to POLE_ESMR_MAX_GAP_DAYS away (esmr_time_field). The error this
# adds is measured on the windows where ESMR IS available: ESMR is removed from the
# window and from a margin of m days on either side, the window is rebuilt by the
# same time interpolation, and the result is compared with the true ESMR window mean
# over the pole-hole cells. The RMS error as a function of the distance in time to
# the ESMR used (the `dist` returned by esmr_time_field) is the temporal term of the
# pole-hole uncertainty in Part 4.
TIME_INTERP_MARGINS = (0, 7, 14, 21, 28, 42, 56)
_rows = []
_cap = MASKS["N"]["pole_cap"]
for pk in sorted(nh_periods):
    w0 = datetime.strptime(pk, "%Y%m%d")
    inside = [d for d in nh_esmr if w0 <= datetime.strptime(d, "%Y%m%d") <= w0 + timedelta(days=PERIOD_DAYS - 1)]
    if len(inside) < PERIOD_DAYS - 2:                   # a window ESMR covers (almost) fully
        continue
    truth = _esmr_mean(inside, nh_esmr)
    for m in TIME_INTERP_MARGINS:
        field, desc, dist = esmr_time_field(pk, esmr_without(nh_esmr, pk, m, m))
        if field is None or "one side" in desc:
            continue
        ok = _cap & np.isfinite(truth) & np.isfinite(field)
        if ok.sum() < 100:
            continue
        d = field[ok] - truth[ok]
        _rows.append(dict(window=pk, margin=m, dist_days=dist, n=int(ok.sum()),
                          bias=float(d.mean()), rmse=float(np.sqrt(np.mean(d ** 2)))))
TIME_INTERP_ERR = pd.DataFrame(_rows)
TIME_INTERP_ERR.to_csv(VAL_DIR / "esmr_time_interpolation_error.csv", index=False)
_tbl = TIME_INTERP_ERR.groupby("margin").agg(dist_days=("dist_days", "mean"), rmse=("rmse", lambda r: float(np.sqrt(np.mean(np.square(r))))),
                                             bias=("bias", "mean"), windows=("window", "size"))
print("ESMR interpolated in time over the pole hole, against the true ESMR window mean [pp]")
print(_tbl.round(2).to_string())
_d = np.r_[0.0, _tbl["dist_days"].to_numpy()]
_e = np.r_[0.0, _tbl["rmse"].to_numpy()]
_order = np.argsort(_d)
_d, _e = _d[_order], np.maximum.accumulate(_e[_order])   # non-decreasing with distance


def time_interp_error_pp(dist_days):
    """RMS error [pp] of ESMR interpolated in time, at a distance of dist_days from the
    ESMR used; held constant beyond the largest distance tested."""
    return np.interp(dist_days, _d, _e)

In [ ]:
# =============================================================================
# 4. Inference, pole-hole fill, provenance, uncertainty and NetCDF export
# =============================================================================
# Pole hole (never-observed ocean around the North Pole; there is none in the
# Antarctic): ESMR of the same window when it has >= POLE_ESMR_MIN_DAYS days,
# otherwise ESMR interpolated in time between the nearest ESMR days before and after
# the window (up to POLE_ESMR_MAX_GAP_DAYS away), feathered into a harmonic
# interpolation from the rim. No second land-spillover correction is applied: the
# daily retrievals (Part 1) and the ESMR product are already corrected.
PROVENANCE = {
    1: "NEMS retrieval (14-day mean)",
    2: "ESMR period mean",
    3: "graph neural network reconstruction",
    4: "pole hole: ESMR, same window",
    5: "pole hole: ESMR, interpolated in time",
    6: "pole hole: interpolated from the rim",
    7: "nearest-neighbour fill (unobserved ocean outside the pole hole, or prediction fringe)",
}
PROVENANCE_NAMES = {1: "nems_retrieval", 2: "esmr_period_mean", 3: "gnn_reconstruction",
                    4: "pole_hole_esmr_same_window", 5: "pole_hole_esmr_time_interpolated",
                    6: "pole_hole_rim_interpolation", 7: "nearest_neighbour_fill"}

def esmr_pole_field(pk, esmr):
    field, desc, _ = esmr_time_field(pk, esmr)
    return field, desc


def _harmonic_fill(field, fill, domain, n_iter=5000, tol=0.01):
    """Smooth fill of `fill` cells from surrounding `domain` values (land = no-flux)."""
    out = np.asarray(field, np.float64).copy()
    known = domain & ~fill & np.isfinite(out)
    if not fill.any() or not known.any():
        return out
    out[fill] = _edt_inpaint(np.where(known, out, np.nan), known)[fill]
    w = domain.astype(np.float64)
    vals = np.where(domain, np.nan_to_num(out), 0.0)
    for _ in range(n_iter):
        p, pw = np.pad(vals * w, 1), np.pad(w, 1)
        s = p[:-2, 1:-1] + p[2:, 1:-1] + p[1:-1, :-2] + p[1:-1, 2:]
        n = pw[:-2, 1:-1] + pw[2:, 1:-1] + pw[1:-1, :-2] + pw[1:-1, 2:]
        new = np.where(n > 0, s / np.maximum(n, 1), vals)
        delta = np.max(np.abs(new[fill] - vals[fill]))
        vals[fill] = new[fill]
        if delta < tol:
            break
    out[fill] = vals[fill]
    return out


def fill_pole_hole(filled, cap, ocean, esmr_field, esmr_desc=""):
    """Replace the nearest-neighbour pole-hole fill with ESMR, feathered over
    POLE_FEATHER_PX cells at the rim into a harmonic interpolation of the rim values.
    Returns (field, description, mask of pole cells that carry ESMR)."""
    used = np.zeros(cap.shape, bool)
    if not cap.any():
        return filled, "no ocean pole hole", used
    rows, cols = np.where(binary_dilation(cap, iterations=POLE_FEATHER_PX + 3))
    r0, r1, c0, c1 = rows.min(), rows.max() + 1, cols.min(), cols.max() + 1   # work on a crop
    sub_cap, sub_ocean = cap[r0:r1, c0:c1], ocean[r0:r1, c0:c1]
    L = _harmonic_fill(filled[r0:r1, c0:c1], sub_cap, sub_ocean)

    out = filled.astype(np.float32).copy()
    sub = out[r0:r1, c0:c1]
    if esmr_field is None:
        sub[sub_cap] = L[sub_cap]
        desc = "smooth interpolation from the rim (no ESMR)"
    else:
        E = np.asarray(esmr_field, np.float64)[r0:r1, c0:c1]
        w = np.clip((distance_transform_edt(sub_cap) - 0.5) / POLE_FEATHER_PX, 0.0, 1.0)
        blend = np.where(np.isfinite(E), w * E + (1 - w) * L, L)
        sub[sub_cap] = blend[sub_cap]
        used[r0:r1, c0:c1] = sub_cap & np.isfinite(E)
        desc = esmr_desc
    out[r0:r1, c0:c1] = np.clip(sub, 0.0, 100.0)
    return out, desc, used



def fill_uncertainty_model(sweep, method="network"):
    """Reconstruction error [pp] as a function of the distance [km] to the nearest
    defined composite cell, from the RMSE of the gap-filler in the hold-out sweep: a
    withheld square of side G has cells up to G/2 from its edge."""
    net = sweep[sweep["method"] == method].sort_values("gap_km")
    d = np.r_[0.0, net["gap_km"].to_numpy() / 2.0]
    e = np.r_[net["rmse"].to_numpy()[0], net["rmse"].to_numpy()]
    return lambda dist_km: np.interp(dist_km, d, e)


def period_product(pk, hemi, periods, nems, esmr, model, cache, fill_unc):
    masks = MASKS[hemi]
    unobs, ocean, cap = masks["unobs"], masks["ocean"], masks["pole_cap"]
    comp, dc, std, em, em_ph, um, us, es_m, es_s, es_vf, bp, _, _ = build_composite(
        periods[pk], nems, esmr, unobs)
    prev, nxt = cache.neighbours(pk)
    filled = predict(comp, dc, std, em, em_ph, um, us, es_m, es_s, es_vf, bp, model, masks, prev, nxt,
                     esmr_ctx=esmr_context(pk, esmr))
    e_field, e_desc = esmr_pole_field(pk, esmr) if cap.any() else (None, "")
    filled, pole_src, pole_esmr = fill_pole_hole(filled, cap, ocean, e_field, e_desc)

    r = FRINGE_DILATION_PX
    fringe = binary_dilation(masks["fringe_src"], structure=np.ones((2 * r + 1, 2 * r + 1), bool))
    nems_cell = (dc > 0) & ~unobs
    esmr_cell = (em | em_ph) & ~cap
    prov = np.zeros(comp.shape, np.int8)
    prov[ocean & np.isnan(comp) & ~unobs & ~fringe] = 3
    prov[ocean & ((np.isnan(comp) & ~unobs & fringe) | (unobs & np.isnan(comp) & ~cap))] = 7
    prov[ocean & esmr_cell] = 2
    prov[ocean & nems_cell] = 1
    if cap.any():
        same = pole_src.startswith("ESMR, same window")
        prov[cap & pole_esmr] = 4 if same else 5
        prov[cap & ~pole_esmr] = 6

    esmr_days = [d for d in periods[pk] if d in esmr]
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        esmr_unc = (np.nanmean(np.stack([load_esmr_fill_unc(esmr[d]) for d in esmr_days]), axis=0)
                    if esmr_days else np.full(comp.shape, np.nan, np.float32))
    # Uncertainty of the defined composite cells (NEMS period mean, or ESMR where ESMR
    # filled the composite), as a fraction.
    src_unc = np.where(nems_cell, um, np.where(np.isfinite(esmr_unc), esmr_unc, um))
    defined = ~np.isnan(comp)
    dist_px, (iy, ix) = distance_transform_edt(~defined, return_indices=True)
    dist_km = dist_px * 25.0
    # Filled cells (network reconstruction, nearest-neighbour fill, rim interpolation):
    # the hold-out RMSE at the cell's distance to the nearest defined cell, combined in
    # quadrature with the uncertainty of that nearest defined cell, which the fill
    # inherits. The first term is the reconstruction error, the second the error of the
    # values it starts from.
    unc = np.hypot(fill_unc(dist_km) / 100.0, np.nan_to_num(src_unc[iy, ix]))
    unc = np.where(prov == 1, um, unc)
    unc = np.where((prov == 2) | (prov == 4), esmr_unc, unc)
    if (prov == 5).any():
        # Pole hole filled with ESMR interpolated in time: the ESMR uncertainty of the
        # days used, combined with the time-interpolation error at that distance in
        # time (cell 4a).
        e_unc, _, e_dist = esmr_time_field(pk, {d: f for d, f in esmr.items()}, loader=load_esmr_fill_unc)
        t_err = time_interp_error_pp(e_dist) / 100.0
        unc = np.where(prov == 5, np.hypot(np.nan_to_num(e_unc), t_err), unc)
    unc = np.where(prov == 0, np.nan, unc).astype(np.float32)

    nems_only = np.where(nems_cell, comp, np.nan).astype(np.float32)
    return dict(filled=filled, nems_only=nems_only, unc=unc, prov=prov, day_count=dc,
                pole_source=pole_src)


def write_period(pk, hemi, res, smask, mask_path):
    start = datetime.strptime(pk, "%Y%m%d")
    stack = _apply_masks(np.dstack([res["filled"], res["unc"] * 100.0]), mask_path,
                         ["ice_conc", "total_standard_error"], month=None)
    sic, unc = stack[:, :, 0], stack[:, :, 1] / 100.0
    nan = np.full(sic.shape, np.nan, np.float32)
    path = write_netcdf({"ice_conc": sic / 100.0, "raw_ice_conc_values": res["nems_only"] / 100.0,
                         "total_standard_error": unc, "algorithm_standard_error": nan,
                         "smearing_standard_error": nan, "status_flag": make_status_flag(sic, smask),
                         "tb_22": nan, "tb_31": nan, "tb_22_corr": nan, "tb_31_corr": nan},
                        start, hemi, AREA_DEF[hemi], build_save_dir(GNN_FILLED_DIR, hemi, start))
    with nc.Dataset(path, "a") as f:                 # variables create_NetCDF does not write
        dims = f.variables["ice_conc"].dimensions
        p = f.createVariable("provenance_flag", "i1", dims, zlib=True, fill_value=-1)
        p[:] = res["prov"][None]
        p.long_name = "source of the ice_conc value"
        p.flag_values = np.array([0] + sorted(PROVENANCE), dtype="i1")
        p.flag_meanings = " ".join(["not_ocean"] + [PROVENANCE_NAMES[k] for k in sorted(PROVENANCE)])
        p.comment = "; ".join(f"{k}: {v}" for k, v in PROVENANCE.items()) + "; 0: land, lake or coast"
        n = f.createVariable("nems_day_count", "i2", dims, zlib=True)
        n[:] = res["day_count"][None]
        n.long_name = "number of days with a NEMS retrieval in the 14-day period"
        f.variables["total_standard_error"].comment = (
            "NEMS cells: period mean of the daily total uncertainty; ESMR cells: period mean of the "
            "ESMR total standard error; reconstructed cells: hold-out RMSE of the reconstruction at the "
            "cell's distance to the nearest defined composite cell")
        end = start + timedelta(days=PERIOD_DAYS)
        tv = f.variables["time"]
        if "time_bnds" in f.variables:
            tb = f.variables["time_bnds"]
            tb[0, :] = nc.date2num([start, end], getattr(tb, "units", tv.units),
                                   getattr(tv, "calendar", "standard"))
        f.time_coverage_end = f"{end - timedelta(seconds=1):%Y-%m-%dT%H:%M:%SZ}"
        f.time_coverage_duration = f.time_coverage_resolution = f"P{PERIOD_DAYS}D"
        f.period_start = f"{start:%Y-%m-%d}"
        f.period_end = f"{start + timedelta(days=PERIOD_DAYS - 1):%Y-%m-%d}"
        f.period_midpoint = f"{start + timedelta(days=PERIOD_DAYS // 2):%Y-%m-%d}"
    return path


if "GNN" not in globals():                         # Part 4 run without Part 3: load checkpoints
    GNN = {}
    for _h in ("N", "S"):
        _m = ResGraphSAGE(in_ch=N_FEATURES_V4, hidden=HIDDEN_DIM, layers=NUM_LAYERS).to(DEVICE)
        _m.load_state_dict(torch.load(MODEL_DIR / f"gnn_{_h}H.pth", map_location=DEVICE))
        GNN[_h] = dict(model=_m.eval(), sweep=pd.read_csv(VAL_DIR / f"gap_sweep_{_h}H.csv"))
    POST_MODE = (VAL_DIR / "post_mode.txt").read_text().strip()
    for _h in ("N", "S"):
        GNN[_h]["sweep_outage"] = pd.read_csv(VAL_DIR / f"gap_sweep_outage_{_h}H.csv")

PERIOD_PRODUCTS = {}
for hemi, periods, nems, esmr, smask, mpath in (
        ("N", nh_periods, nh_nems, nh_esmr, arctic_smask, ARCTIC_LANDMASK_PATH),
        ("S", sh_periods, sh_nems, sh_esmr, antarctic_smask, ANTARCTIC_LANDMASK_PATH)):
    cache = CompositeCache(periods, nems, esmr, MASKS[hemi]["unobs"])
    # The network's reconstruction error from the test sweep: the simulated-outage sweep
    # for windows inside an ESMR outage, the normal one otherwise.
    unc_normal = fill_uncertainty_model(GNN[hemi]["sweep"])
    unc_outage = fill_uncertainty_model(GNN[hemi]["sweep_outage"])
    for pk in sorted(periods):
        n_esmr = sum(1 for d in periods[pk] if d in esmr)
        fill_unc = unc_outage if n_esmr < ESMR_MIN_DAYS else unc_normal
        res = period_product(pk, hemi, periods, nems, esmr, GNN[hemi]["model"], cache, fill_unc)
        res["path"] = write_period(pk, hemi, res, smask, mpath)
        PERIOD_PRODUCTS[(hemi, pk)] = res
        counts = {k: int((res["prov"] == k).sum()) for k in PROVENANCE}
        print(f"{hemi}H {pk}: pole hole <- {res['pole_source'] or 'n/a'}; cells by provenance {counts}")
print(f"Wrote {len(PERIOD_PRODUCTS)} period files to {GNN_FILLED_DIR}")

## Part 5 — Validation, figures and manuscript numbers

All cells read the NetCDF products written in Parts 1 and 4. Every comparison uses the same 14-day windows as the NEMS periods. The windows run from 17 December 1972, extent is the area of ocean cells with SIC ≥ 30 % on the EASE2 grid, and a window counts as an ESMR outage when it has fewer than `ESMR_MIN_DAYS` ESMR days.

In [ ]:
# =============================================================================
# 5a. Ocean-masked distribution copies of the 14-day product
# =============================================================================
def hemisphere_of(path):
    name = Path(path).name.upper()
    return "N" if ("_NH-" in name or "EASE2_NH" in name) else ("S" if ("_SH-" in name or "EASE2_SH" in name) else None)


_files = [p for p in GNN_FILLED_DIR.rglob("*.nc") if GNN_FILLED_MASKED_DIR not in p.parents]
for path in _files:
    hemi = hemisphere_of(path)
    with xr.open_dataset(path) as d:
        d = d.load()
    for v in ("ice_conc", "total_standard_error"):
        d[v] = d[v].where(OCEAN[hemi])
    d["ice_conc"] = d["ice_conc"].where((d["ice_conc"] >= 0) & (d["ice_conc"] <= 100))
    d["ice_conc"].attrs["comment"] = (d["ice_conc"].attrs.get("comment", "") +
                                      " Masked to ocean (lmask = 0 in the ESA CCI LandOceanLakeMask v2).").strip()
    d.to_netcdf(GNN_FILLED_MASKED_DIR / f"{path.stem}_masked.nc")
print(f"{len(_files)} ocean-masked copies in {GNN_FILLED_MASKED_DIR}")

In [ ]:
# =============================================================================
# 5b. NIC weekly ice charts on the EASE2 grid
# =============================================================================
def parse_nic_date(fname):
    m = re.search(r"nic_weekly_(\d{4})_(\d{2})_(\d{2})", fname)
    return datetime(int(m.group(1)), int(m.group(2)), int(m.group(3))) if m else None


def read_nic_bin(path, grid_size=361):
    return np.fromfile(path, dtype=np.uint8).reshape(grid_size, grid_size)


def nic_to_sic(grid):
    sic = grid.astype(np.float32).copy()
    sic[grid == 108] = 100.0
    sic[np.isin(grid, [157, 253, 254])] = np.nan
    return sic


def nic_land_mask(grid):
    return grid == 254


nic_bin_files_all = sorted(glob.glob(str(NIC_ROOT / "**" / "nic_weekly_*.bin"), recursive=True))
nic_file_dates_all = [(f, parse_nic_date(os.path.basename(f))) for f in nic_bin_files_all]
nic_file_dates_all = [(f, d) for f, d in nic_file_dates_all if d is not None]
nic_file_dates_all.sort(key=lambda x: x[1])
print(f"Found {len(nic_file_dates_all)} NIC weekly ice-chart file(s) under {NIC_ROOT}.")


# NIC charts (NSIDC G02172) are on the NA25 EASE-Grid (v1): 361 x 361 cells of
# 25.067525 km, Lambert azimuthal equal-area on a 6371.228 km sphere, pole at
# row/col 180. NEMS and ESMR are on EASE2 (EPSG:6931, 25 km, 432 x 432). Charts are
# moved to EASE2 by nearest-neighbour lookup, which keeps the chart classes; EASE2
# cells outside the chart domain are NaN.
from pyproj import Transformer

NA25_CELL_M = 25067.525
NA25_CENTER = 180
NA25_SIZE = 361
_EASE1_NL = "+proj=laea +lat_0=90 +lon_0=0 +x_0=0 +y_0=0 +R=6371228 +units=m +no_defs"
_NA25_INDEX = None


def _ease2_nh_xy_m():
    """Cell-centre x, y [m] of the EASE2 NH grid the NEMS/ESMR files use."""
    if "nh_esmr" in globals() and nh_esmr:
        with xr.open_dataset(nh_esmr[sorted(nh_esmr)[0]]) as ds:
            xn = "xc" if "xc" in ds.coords or "xc" in ds else "x"
            yn = "yc" if "yc" in ds.coords or "yc" in ds else "y"
            x, y = ds[xn].values.astype(float), ds[yn].values.astype(float)
        scale = 1.0 if np.nanmax(np.abs(x)) > 1e5 else 1000.0   # km -> m
        return x * scale, y * scale
    print("  [nic_to_ease2] no ESMR file to read xc/yc from -- using the "
          "standard 432 x 432, +/-5400 km EASE2 NH grid")
    c = (np.arange(432) - 215.5) * 25000.0
    return c, -c


def _na25_index():
    global _NA25_INDEX
    if _NA25_INDEX is None:
        x, y = _ease2_nh_xy_m()
        X, Y = np.meshgrid(x, y)
        to_ease1 = Transformer.from_crs("EPSG:6931", _EASE1_NL, always_xy=True)
        xa, ya = to_ease1.transform(X, Y)
        col = np.rint(xa / NA25_CELL_M + NA25_CENTER).astype(int)
        row = np.rint(NA25_CENTER - ya / NA25_CELL_M).astype(int)
        inside = (row >= 0) & (row < NA25_SIZE) & (col >= 0) & (col < NA25_SIZE)
        _NA25_INDEX = (np.clip(row, 0, NA25_SIZE - 1), np.clip(col, 0, NA25_SIZE - 1), inside)
    return _NA25_INDEX


def nic_to_ease2(nic_field):
    """Any 361 x 361 NA25 field (SIC from nic_to_sic, or a boolean mask) -> EASE2 NH."""
    row, col, inside = _na25_index()
    src = np.asarray(nic_field)
    if src.dtype == bool:
        out = src[row, col].copy()
        out[~inside] = False
        return out
    out = src.astype(np.float32)[row, col]
    out[~inside] = np.nan
    return out


def nic_sic_ease2(path):
    """Read one NIC weekly chart straight onto EASE2 NH (SIC %, NaN = land / not covered)."""
    return nic_to_ease2(nic_to_sic(read_nic_bin(path)))


# Charts already on EASE2 (*_NH_NIC-BIN.nc), if present, are used instead of the
# regridded .bin charts. The array orientation is checked once against the EASE2
# land mask, so a file stored with the y-axis reversed cannot shift the chart.
NIC_EASE2_PATTERN = "*_NH_NIC-BIN.nc"
USE_NIC_EASE2_FILES = NIC_EASE2_DIR.exists() and any(NIC_EASE2_DIR.rglob(NIC_EASE2_PATTERN))

_NIC_LAND_CODES = (254,)
_NIC_NODATA_CODES = (157, 253, 254, 255)
_NIC_ORIENT = None
_ORIENTATIONS = {
    "as stored": lambda a: a,
    "flipped up-down": np.flipud,
    "flipped left-right": np.fliplr,
    "transposed": lambda a: a.T,
}


def _nic_nc_varname(ds):
    names = [n for n in ds.data_vars if ds[n].ndim >= 2 and np.issubdtype(ds[n].dtype, np.number)]
    for key in ("ice_conc", "sic", "conc", "nic", "ice"):
        for n in names:
            if key in n.lower():
                return n
    if names:
        return names[0]
    raise KeyError(f"no 2-D numeric variable in {list(ds.data_vars)}")


def _nic_nc_read(path):
    """-> (SIC [%] with NaN for land/no data, land mask), as stored in the file."""
    with xr.open_dataset(path) as ds:
        a = np.squeeze(ds[_nic_nc_varname(ds)].values).astype(np.float32)
    if a.ndim != 2:
        raise ValueError(f"{Path(path).name}: expected a 2-D chart, got shape {a.shape}")
    land = np.isin(a, _NIC_LAND_CODES)
    sic = a.copy()
    sic[a == 108] = 100.0
    sic[np.isin(a, _NIC_NODATA_CODES)] = np.nan
    sic[(sic < 0) | (sic > 100)] = np.nan
    finite = sic[np.isfinite(sic)]
    if finite.size and finite.max() <= 1.5:
        sic = sic * 100.0
    return sic, land


def _nic_orientation(sample_path):
    """Pick the orientation whose NIC land best matches the EASE2 land mask."""
    sic, land = _nic_nc_read(sample_path)
    ref = (np.asarray(arctic_smask) != 0) if "arctic_smask" in globals() else None
    if ref is not None and sic.shape != ref.shape:
        raise ValueError(f"NIC file grid {sic.shape} != EASE2 grid {ref.shape} -- not the NEMS EASE2 grid")
    if ref is None or not land.any():
        print("  [NIC EASE2] no land code in the file (or no land mask loaded); using the orientation as stored")
        return "as stored"
    covered = np.isfinite(sic) | land
    scores = {}
    for name, f in _ORIENTATIONS.items():
        l, c = f(land), f(covered)
        scores[name] = (l & ref & c).sum() / max(((l | ref) & c).sum(), 1)
    best = max(scores, key=scores.get)
    print("  [NIC EASE2] land overlap with the EASE2 mask: "
          + ", ".join(f"{k} {v:.2f}" for k, v in sorted(scores.items(), key=lambda kv: -kv[1])))
    if best != "as stored":
        print(f"  [NIC EASE2] NOTE: files are {best} relative to the NEMS grid -- corrected on read")
    if scores[best] < 0.5:
        print("  [NIC EASE2] WARNING: poor land agreement for every orientation -- check the grid")
    return best


def nic_sic_ease2_nc(path):
    global _NIC_ORIENT
    if _NIC_ORIENT is None:
        _NIC_ORIENT = _nic_orientation(path)
    return _ORIENTATIONS[_NIC_ORIENT](_nic_nc_read(path)[0])


if USE_NIC_EASE2_FILES:
    nic_file_dates_all = []
    for f in sorted(NIC_EASE2_DIR.rglob(NIC_EASE2_PATTERN)):
        m = re.search(r"(\d{8})", f.name)
        if m:
            nic_file_dates_all.append((str(f), datetime.strptime(m.group(1), "%Y%m%d")))
    nic_file_dates_all.sort(key=lambda x: x[1])
    nic_sic_ease2 = nic_sic_ease2_nc          # replaces the NA25 regrid everywhere downstream
    _probe = nic_sic_ease2(nic_file_dates_all[0][0])
    print(f"Using {len(nic_file_dates_all)} EASE2 NIC charts from {NIC_EASE2_DIR} "
          f"({nic_file_dates_all[0][1].date()} -> {nic_file_dates_all[-1][1].date()}); "
          f"grid {_probe.shape}, {int(np.isfinite(_probe).sum()):,} cells with chart data, "
          f"SIC range {np.nanmin(_probe):.0f}-{np.nanmax(_probe):.0f} %")
else:
    print(f"Using {len(nic_file_dates_all)} NA25 .bin NIC charts, regridded to EASE2.")

In [ ]:
# =============================================================================
# 5c. Extent and area on the NEMS 14-day windows: NEMS, ESMR and NIC
# =============================================================================
# Columns: NEMS (gap-filled), NEMS_direct_cells (cells with a NEMS retrieval only),
# ESMR, NIC (Arctic only), *_area (SIC-weighted), and NEMS_common / ESMR_common:
# both instruments counted over the SAME cells (NEMS retrieved and ESMR defined).
# The *_common pair is the only comparison here that is independent of the ESMR
# fill. Gap-filled NEMS in ESMR-covered windows contains ESMR by construction.
ESMR_MIN_VALID_FRAC = 0.05       # an ESMR day is usable if >= 5 % of ocean cells are defined
if "nh_esmr" not in globals():
    nh_nems, nh_esmr, nh_dates = discover("N")
    sh_nems, sh_esmr, sh_dates = discover("S")
WINDOWS = pd.date_range(FIRST_WINDOW, LAST_WINDOW_START, freq=f"{PERIOD_DAYS}D")


def extent_area(field, ocean, sel=None):
    """(extent, area) in million km2 of ocean cells with SIC >= SIE_THRESHOLD."""
    if field is None:
        return np.nan, np.nan
    f = np.where(ocean & np.isfinite(field) & (True if sel is None else sel), field, np.nan)
    ice = f >= SIE_THRESHOLD
    return ice.sum() * PIXEL_AREA_KM2 / 1e6, np.nansum(np.where(ice, f / 100.0, 0)) * PIXEL_AREA_KM2 / 1e6


def window_mean(fields):
    if not fields:
        return None
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        return np.nanmean(np.stack(fields, 0), axis=0)


def period_files(hemi):
    root = GNN_FILLED_MASKED_DIR if any(GNN_FILLED_MASKED_DIR.rglob("*.nc")) else GNN_FILLED_DIR
    tag = "_NH-" if hemi == "N" else "_SH-"
    return {file_date(p): p for p in root.rglob("*.nc") if tag in p.name}


rows = []
for hemi, esmr, smask in (("N", nh_esmr, arctic_smask), ("S", sh_esmr, antarctic_smask)):
    ocean = OCEAN[hemi]
    nems_files = period_files(hemi)
    esmr_dt = {d: datetime.strptime(d, "%Y%m%d") for d in esmr}
    for w0 in WINDOWS:
        w1 = w0 + timedelta(days=PERIOD_DAYS - 1)
        row = dict(hemi="NH" if hemi == "N" else "SH", window_start=w0, midpoint=w0 + timedelta(days=PERIOD_DAYS // 2))
        nems_full = nems_raw = None
        if w0.strftime("%Y%m%d") in nems_files:
            with xr.open_dataset(nems_files[w0.strftime("%Y%m%d")]) as d:
                nems_full = to_percent(d["ice_conc"].values[0])
                nems_raw = to_percent(d["raw_ice_conc_values"].values[0])
        esmr_fields = []
        for d, t in esmr_dt.items():
            if w0 <= t <= w1:
                a = load_ice(esmr[d])
                if np.isfinite(a[ocean]).mean() >= ESMR_MIN_VALID_FRAC:
                    esmr_fields.append(a)
        row["n_esmr_days"] = len(esmr_fields)
        esmr_mean = window_mean(esmr_fields) if len(esmr_fields) >= ESMR_MIN_DAYS else None
        nic_mean = None
        if hemi == "N":
            charts = [f for f, t in nic_file_dates_all if w0 <= t <= w1 + timedelta(hours=23)]
            row["n_nic_charts"] = len(charts)
            nic_mean = window_mean([nic_sic_ease2(f) for f in charts])
        row["NEMS"], row["NEMS_area"] = extent_area(nems_full, ocean)
        row["NEMS_direct_cells"], _ = extent_area(nems_raw, ocean)
        row["ESMR"], row["ESMR_area"] = extent_area(esmr_mean, ocean)
        row["NIC"], row["NIC_area"] = extent_area(nic_mean, ocean)
        if nems_raw is not None and esmr_mean is not None:
            common = np.isfinite(nems_raw) & np.isfinite(esmr_mean)
            row["common_cells"] = int((common & ocean).sum())
            row["NEMS_common"], _ = extent_area(nems_raw, ocean, common)
            row["ESMR_common"], _ = extent_area(esmr_mean, ocean, common)
        rows.append(row)
val_reg_df = pd.DataFrame(rows)
val_reg_df.to_csv(VAL_DIR / "registered_extent_series.csv", index=False)


def agreement(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    ok = np.isfinite(a) & np.isfinite(b)
    d = a[ok] - b[ok]
    return dict(n=int(ok.sum()), bias=float(d.mean()) if ok.any() else np.nan,
                rmse=float(np.sqrt(np.mean(d ** 2))) if ok.any() else np.nan,
                r=float(np.corrcoef(a[ok], b[ok])[0, 1]) if ok.sum() > 2 else np.nan)


EXTENT_AGREEMENT = {}
for h in ("NH", "SH"):
    d = val_reg_df[val_reg_df.hemi == h]
    pairs = [("NEMS", "ESMR"), ("NEMS_common", "ESMR_common")] + ([("NEMS", "NIC")] if h == "NH" else [])
    for a, b in pairs:
        EXTENT_AGREEMENT[(h, a, b)] = agreement(d[a], d[b])
    outage = d.NEMS.notna() & d.ESMR.isna()
    EXTENT_AGREEMENT[(h, "NEMS-only windows", "")] = dict(n=int(outage.sum()),
                                                          windows=list(d.window_start[outage].dt.strftime("%Y-%m-%d")))
for k, v in EXTENT_AGREEMENT.items():
    print(k, {kk: (round(vv, 3) if isinstance(vv, float) else vv) for kk, vv in v.items()})
print(val_reg_df[val_reg_df.NEMS.notna()].to_string(index=False, float_format=lambda v: f"{v:.2f}"))

In [ ]:
# =============================================================================
# 5d. Figure: sea ice extent 1972-1973, NEMS vs ESMR and ice charts
# =============================================================================
# Dots: gap-filled NEMS extent. Shading: windows with NEMS but fewer than
# ESMR_MIN_DAYS usable ESMR days. SHOW_RETRIEVED_ONLY adds the extent from
# NEMS-retrieved cells only (open dots on stems); it is off in the paper figure.
SHOW_RETRIEVED_ONLY = False
X_START, X_END = pd.Timestamp("1972-12-01"), pd.Timestamp("1973-12-10")
reg = val_reg_df.copy() if "val_reg_df" in globals() else pd.read_csv(
    VAL_DIR / "registered_extent_series.csv", parse_dates=["window_start", "midpoint"])
reg["window_start"] = pd.to_datetime(reg["window_start"])
reg["midpoint"] = pd.to_datetime(reg["midpoint"])


def _read_g00917(path):
    toks = Path(path).read_text().split()
    rows, i = {}, 0
    while i + 36 < len(toks):
        yy, mm = int(toks[i][:2]), int(toks[i][2:])
        end = pd.Timestamp(1900 + yy if yy > 50 else 2000 + yy, mm, 1) + pd.offsets.MonthEnd(0)
        rows[end] = np.array(toks[i + 1:i + 37], dtype=float).sum() / 1000.0   # 1000 km2 -> million km2
        i += 37
    return pd.DataFrame({"total": pd.Series(rows)}).sort_index()



try:
    g00917 = {"NH": _read_g00917(G00917_DIR / "arctic.dat"), "SH": _read_g00917(G00917_DIR / "antarc.dat")}
except OSError:
    g00917 = None
    print(f"G00917 files not found in {G00917_DIR}; figure drawn without the CAC charts.")
chart = g00917

# --- Style (same palette and sizes as Fig. 6)
INK, FRAME, MUTED, GRID_COL = "#1A1A1A", "#3C3C3C", "#5A5A5A", "#E4E6E9"
C_NEMS, C_ESMR, C_CHART, C_GAP = "#F58220", "#3498C3", "#2A3550", "#ECA458"
C_STEM = "#F9BA84"          # light NEMS orange for the stems (no alpha, so the legend matches)
FS_TITLE, FS_LETTER, FS_LABEL, FS_TICK, FS_LEGEND = 10.5, 11, 9.5, 8.5, 9
plt.rcParams.update({
    "font.family": "sans-serif", "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 9, "axes.edgecolor": FRAME, "axes.linewidth": 0.5, "axes.labelcolor": INK,
    "xtick.color": INK, "ytick.color": INK, "text.color": INK,
    "figure.facecolor": "white", "savefig.facecolor": "white", "pdf.fonttype": 42, "ps.fonttype": 42,
})


def _outage_spans(d):
    """Contiguous runs of NEMS windows without usable ESMR -> [(start, end), ...]."""
    spans, run = [], None
    for _, r in d.iterrows():
        gap = np.isfinite(r["NEMS"]) and not np.isfinite(r["ESMR"])
        if gap:
            end = r["window_start"] + pd.Timedelta(days=PERIOD_DAYS)
            run = (run[0], end) if run else (r["window_start"], end)
        elif run:
            spans.append(run); run = None
    if run:
        spans.append(run)
    return spans


def _month_label(x, pos=None):
    d = mdates.num2date(x)
    return f"{d:%b}\n{d.year}" if (d.month == 1 or pos == 0) else f"{d:%b}"


def _panel(ax, d, hemi, letter, title):
    for s0, s1 in _outage_spans(d):
        ax.axvspan(s0, s1, color=C_GAP, alpha=0.25, lw=0, zorder=0)

    # Drawing order (bottom -> top): NEMS stems and dots, NIC, ESMR, CAC. NEMS is
    # the largest marker and sits underneath, so where ESMR/NIC have (almost) the
    # same value their small markers show INSIDE the NEMS dot instead of hiding.
    has = d.NEMS.notna()
    if SHOW_RETRIEVED_ONLY and "NEMS_direct_cells" in d:
        ax.vlines(d.midpoint[has], d.NEMS_direct_cells[has], d.NEMS[has], color=C_STEM, lw=1.3, zorder=2)
        ax.scatter(d.midpoint[has], d.NEMS_direct_cells[has], s=22, facecolors="white",
                   edgecolors=C_NEMS, linewidths=1.1, zorder=3)
    ax.scatter(d.midpoint[has], d.NEMS[has], s=70, color=C_NEMS, edgecolors="white", linewidths=0.7, zorder=4)

    if hemi == "NH" and d["NIC"].notna().any():
        ax.plot(d.midpoint, d.NIC, color=C_CHART, lw=1.2, marker="s", ms=3.4, mec="white", mew=0.5, zorder=5)
    ax.plot(d.midpoint, d.ESMR, color=C_ESMR, lw=1.4, marker="o", ms=3.6, mec="white", mew=0.6, zorder=6)
    if chart is not None:
        ch = chart[hemi].loc[X_START:X_END, "total"]
        ax.plot(ch.index, ch.values, ls="none", marker="D", ms=4.6, mfc="none", mec=C_CHART, mew=1.1, zorder=7)

    vals = [d[c].to_numpy() for c in ("NEMS", "ESMR", "NIC") if c in d]
    if chart is not None:
        vals.append(chart[hemi].loc[X_START:X_END, "total"].to_numpy())
    top = np.nanmax(np.concatenate(vals))
    ax.set_ylim(0, 5 * np.ceil(top / 5) + (1 if top % 5 > 4 else 0))
    ax.yaxis.set_major_locator(plt.MultipleLocator(5))
    ax.yaxis.set_minor_locator(plt.MultipleLocator(2.5))
    ax.set_ylabel("Sea ice extent [10$^6$ km$^2$]", fontsize=FS_LABEL)
    ax.grid(axis="y", which="major", color=GRID_COL, lw=0.5)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.tick_params(labelsize=FS_TICK, length=3, width=0.6)
    ax.tick_params(which="minor", length=1.5, width=0.4)
    ax.text(0.0, 1.03, letter, transform=ax.transAxes, fontsize=FS_LETTER, fontweight="bold",
            va="bottom", ha="left")
    ax.text(0.028, 1.03, title, transform=ax.transAxes, fontsize=FS_TITLE, va="bottom", ha="left")


fig = plt.figure(figsize=(182 * MM, 150 * MM))
gs = fig.add_gridspec(3, 1, height_ratios=[1, 1, 0.2], hspace=0.3,
                      left=0.085, right=0.975, top=0.95, bottom=0.02)
ax_nh = fig.add_subplot(gs[0])
ax_sh = fig.add_subplot(gs[1], sharex=ax_nh)
_panel(ax_nh, reg[reg.hemi == "NH"].sort_values("midpoint"), "NH", "a", "Northern Hemisphere")
_panel(ax_sh, reg[reg.hemi == "SH"].sort_values("midpoint"), "SH", "b", "Southern Hemisphere")
ax_sh.set_xlim(X_START, X_END)
ax_sh.xaxis.set_major_locator(mdates.MonthLocator())
ax_sh.xaxis.set_major_formatter(plt.FuncFormatter(_month_label))
plt.setp(ax_nh.get_xticklabels(), visible=False)

handles = [
    Line2D([], [], ls="none", marker="o", ms=8.3, mfc=C_NEMS, mec="white", mew=0.7, label="NEMS, gap-filled"),
    Line2D([], [], color=C_ESMR, lw=1.4, marker="o", ms=3.6, mec="white", mew=0.6, label="ESMR"),
    Line2D([], [], color=C_CHART, lw=1.2, marker="s", ms=3.4, mec="white", mew=0.5, label="NIC weekly charts"),
]
if chart is not None:
    handles.append(Line2D([], [], ls="none", marker="D", ms=4.6, mfc="none", mec=C_CHART, mew=1.1,
                          label="CAC charts, month end"))
handles.append(Patch(facecolor=C_GAP, alpha=0.25, lw=0, label="ESMR outage"))
if SHOW_RETRIEVED_ONLY:
    handles.insert(1, Line2D([], [], ls="-", lw=1.3, color=C_STEM, marker="o", ms=4.8, mfc="white", mec=C_NEMS,
                             mew=1.1, label="NEMS, retrieved cells only"))
leg_ax = fig.add_subplot(gs[2]); leg_ax.axis("off")         # legend beneath the panels
leg_ax.legend(handles=handles, loc="center", ncol=3, frameon=False, fontsize=FS_LEGEND,
              handlelength=2.2, handletextpad=0.6, columnspacing=2.2, labelspacing=0.7, borderaxespad=0.0)

saved = []
for ext in ("pdf", "png"):
    out = PLOT_DIR / f"fig_sea_ice_extent_1973.{ext}"
    fig.savefig(out, dpi=300)
    saved.append(str(out))
print("Saved: " + ", ".join(saved))
plt.show()

In [ ]:
# =============================================================================
# 5e. Table 2: monthly mean sea ice extent
# =============================================================================
# Monthly value = mean of the 14-day windows whose midpoint falls in the month
# (1-3 windows; single-window months are marked with a dagger). CAC = NOAA/NMC/CAC
# end-of-month chart extent (NSIDC G00917: ice > 1/8 on 1 x 2.5 degree boxes
# poleward of 50 degrees), a lower threshold than 30 %.
MONTHS = pd.period_range("1972-12", "1973-11", freq="M")
# False: each series uses every window it has (a month can then compare NEMS
#        and ESMR from different windows, e.g. when ESMR is out in one of them).
# True:  per hemisphere, only windows where ALL series exist (like-for-like rows).
JOINT_WINDOWS = False

reg = val_reg_df.copy() if "val_reg_df" in globals() else pd.read_csv(
    VAL_DIR / "registered_extent_series.csv", parse_dates=["window_start", "midpoint"])
reg["midpoint"] = pd.to_datetime(reg["midpoint"])
reg["month"] = reg["midpoint"].dt.to_period("M")
chart = None
if globals().get("g00917") is not None:
    chart = {h: df["total"].set_axis(df["total"].index.to_period("M")) for h, df in g00917.items()}

# --- Monthly means
SERIES = [("NH_NEMS", "NH", "NEMS"), ("NH_ESMR", "NH", "ESMR"), ("NH_NIC", "NH", "NIC"),
          ("SH_NEMS", "SH", "NEMS"), ("SH_ESMR", "SH", "ESMR")]
_need_cols = {"NH": ["NEMS", "ESMR", "NIC"], "SH": ["NEMS", "ESMR"]}
means, counts = {}, {}
for name, hemi, col in SERIES:
    d = reg.loc[reg.hemi == hemi]
    d = d.dropna(subset=_need_cols[hemi] if JOINT_WINDOWS else [col])
    s = d.groupby("month")[col]
    means[name] = s.mean().reindex(MONTHS)
    counts["n_" + name] = s.size().reindex(MONTHS).fillna(0).astype(int)
monthly_sie_means = pd.DataFrame(means)
if chart is not None:
    monthly_sie_means.insert(3, "NH_CAC", chart["NH"].reindex(MONTHS).to_numpy())
    monthly_sie_means["SH_CAC"] = chart["SH"].reindex(MONTHS).to_numpy()
monthly_sie_counts = pd.DataFrame(counts)
monthly_sie = pd.concat([monthly_sie_means, monthly_sie_counts], axis=1)
monthly_sie.index.name = "month"

# Months where every satellite/NIC series has at least one window
_need = {"NH": ["NH_NEMS", "NH_ESMR", "NH_NIC"], "SH": ["SH_NEMS", "SH_ESMR"]}
all_series = {h: monthly_sie_means[c].notna().all(axis=1) for h, c in _need.items()}
monthly_sie["all_NH"] = all_series["NH"]
monthly_sie["all_SH"] = all_series["SH"]

print(f"=== Monthly mean sea ice extent (SIC >= {SIE_THRESHOLD:.0f} %, EASE2), million km2 ===")
print(f"    {len(MONTHS)} months, from the registered 14-day windows of Part 7c"
      + (" (joint windows only)" if JOINT_WINDOWS else "") + "\n")
print(monthly_sie.to_string(float_format=lambda v: f"{v:6.2f}", na_rep="    --"))
for h in ("NH", "SH"):
    print(f"  {h}: month has {', '.join(c.split('_')[1] for c in _need[h])} (not necessarily the same windows "
          f"unless JOINT_WINDOWS): "
          f"{', '.join(p.strftime('%b') for p in MONTHS[all_series[h].to_numpy()])}")


# --- Markdown (for the manuscript draft)
def _md(v, n=None):
    if not np.isfinite(v):
        return "-"
    return f"{v:.2f}" + ("*" if n == 1 else "")


cols = list(monthly_sie_means.columns)
md = ["| Month | " + " | ".join(c.replace("_", " ") for c in cols) + " |",
      "|---|" + "---:|" * len(cols)]
for p in MONTHS:
    md.append(f"| {p.strftime('%b %Y')} | " + " | ".join(
        _md(monthly_sie_means.at[p, c], monthly_sie_counts.at[p, "n_" + c] if "n_" + c in monthly_sie_counts else None)
        for c in cols) + " |")
md.append("\n\\* single 14-day window in that month.  CAC: end-of-month chart, ice > 1/8.")
print("\n" + "\n".join(md))


# --- LaTeX
def _tex(v, n=None, w=5):
    if not np.isfinite(v):
        return f"{'--':>{w}}"
    return f"{v:{w}.2f}" + (r"\rlap{\textsuperscript{\dag}}" if n == 1 else "")


_cac = chart is not None
_nh = ["NH_NEMS", "NH_ESMR", "NH_NIC"] + (["NH_CAC"] if _cac else [])
_sh = ["SH_NEMS", "SH_ESMR"] + (["SH_CAC"] if _cac else [])
tex = [r"\begin{table}[ht]", r"\centering",
       r"\caption{Mean monthly sea ice extent [$10^6$\,km$^2$]. NEMS, ESMR and NIC (SIC $\geq$ 30\,\%, "
       r"EASE2 grid) are averaged over the 14-day NEMS windows centred in each month; months covered by a "
       r"single window are marked (\dag)."
       + (r" CAC is the NOAA/NMC/CAC end-of-month chart extent (ice $>$ 1/8 on 1$^\circ\times$2.5$^\circ$ "
          r"boxes poleward of 50$^\circ$; Ropelewski, 1983) and is expected to exceed the 30\,\% extents."
          if _cac else "") + "}",
       r"\label{tab:sie-monthly}", r"\begin{tabular}{l" + "r" * (len(_nh) + len(_sh)) + "}", r"\toprule",
       rf" & \multicolumn{{{len(_nh)}}}{{c}}{{Arctic}} & \multicolumn{{{len(_sh)}}}{{c}}{{Antarctic}} \\",
       rf"\cmidrule(lr){{2-{1 + len(_nh)}}}\cmidrule(lr){{{2 + len(_nh)}-{1 + len(_nh) + len(_sh)}}}",
       "Month & " + " & ".join(c.split("_")[1] for c in _nh + _sh) + r" \\", r"\midrule"]
for p in MONTHS:
    tex.append(p.strftime("%b %Y") + " & " + " & ".join(
        _tex(monthly_sie_means.at[p, c], monthly_sie_counts.at[p, "n_" + c] if "n_" + c in monthly_sie_counts else None)
        for c in _nh + _sh) + r" \\")
tex += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
val_table1_tex = "\n".join(tex)
print("\n" + val_table1_tex)

# --- Save
VAL_DIR.mkdir(parents=True, exist_ok=True)
_csv_path = VAL_DIR / "sie_monthly_means.csv"
monthly_sie.to_csv(_csv_path, float_format="%.4f")
(VAL_DIR / "table1_sie_monthly.tex").write_text(val_table1_tex)
(VAL_DIR / "table1_sie_monthly.md").write_text("\n".join(md))
print(f"\nSaved: {_csv_path}, {VAL_DIR / 'table1_sie_monthly.tex'}, {VAL_DIR / 'table1_sie_monthly.md'}")

In [ ]:
# =============================================================================
# 5f. Fig. 6 inputs: one 14-day period per hemisphere
# =============================================================================
# Everything is for the same 14-day window: retrieved and gap-filled SIC from the
# period file, total uncertainty as the period mean of the daily product's
# total_standard_error, 31.4 GHz Tb and tie-point samples gridded from the
# observations in the window, and the NIC charts dated inside the window.
FIG6_PERIODS = {"N": "19730225", "S": "19730812"}
FIG6_REGION = {"N": "arctic", "S": "antarctic"}


def load_period_file(hemi, pk):
    """(gap-filled SIC [%], NEMS-only SIC [%], land mask, provenance) from Part 4's file."""
    tag = "_NH-" if hemi == "N" else "_SH-"
    hits = sorted(p for p in GNN_FILLED_DIR.rglob(f"*{tag}{pk}*.nc") if GNN_FILLED_MASKED_DIR not in p.parents)
    if not hits:
        raise FileNotFoundError(f"No {hemi}H period file for {pk} in {GNN_FILLED_DIR}")
    with xr.open_dataset(hits[0]) as d:
        filled = to_percent(d["ice_conc"].values[0])
        raw = to_percent(d["raw_ice_conc_values"].values[0])
        land = (d["status_flag"].values[0].astype(np.int32) & 3) > 0
        prov = d["provenance_flag"].values[0] if "provenance_flag" in d else None
    return filled, raw, land, prov


def period_uncertainty_pct(hemi, pk):
    """Period mean of the daily total uncertainty [pp] (what the product carries)."""
    nems = nh_nems if hemi == "N" else sh_nems
    periods = nh_periods if hemi == "N" else sh_periods
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        return 100.0 * np.nanmean(np.stack([load_unc(nems[d]) for d in periods[pk]]), axis=0)


def nic_for_window(t0, t1):
    """Mean of the NIC charts dated t0..t1 on EASE2 [%], or the chart nearest the
    window midpoint if none falls inside. Returns (field, list of chart dates)."""
    inside = [(f, d) for f, d in nic_file_dates_all if t0 <= d <= t1 + timedelta(hours=23)]
    if not inside and nic_file_dates_all:
        mid = t0 + (t1 - t0) / 2
        inside = [min(nic_file_dates_all, key=lambda fd: abs(fd[1] - mid))]
    if not inside:
        return None, []
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        field = np.nanmean(np.stack([nic_sic_ease2(f) for f, _ in inside]), axis=0).astype(np.float32)
    return field, [d for _, d in inside]


def _period_dynamic_tiepoint_flags(record, period_start, period_end, min_c_ice=0.95,
                                    col_c_ice='c_ice_hybrid', col_22='tb_22_phys_corrected',
                                    col_31='tb_31_phys_corrected', iqr_multiplier=1.0,
                                    ow_min_sst=271, ow_max_sst=275, ow_outlier_sigma=1.0):
    dt = pd.to_datetime(record['datetime'])
    period_sel = (dt >= period_start) & (dt < period_end + timedelta(days=1))
    sub = record[period_sel]

    ice_eligible = sub[col_c_ice] >= min_c_ice
    ice_df = sub[ice_eligible].dropna(subset=[col_22, col_31])
    fyi_idx, myi_idx = pd.Index([]), pd.Index([])
    if len(ice_df) > 0:
        clean = remove_outliers_multivariate(ice_df, columns=[col_22, col_31],
                                              method='iqr', iqr_multiplier=iqr_multiplier)
        if len(clean) > 0:
            n = max(1, int(len(clean) * 0.10))
            fyi_idx = clean.nlargest(n, col_31).index
            myi_idx = clean.nsmallest(n, col_31).index

    ow_eligible = ((sub['Ts'] >= ow_min_sst) & (sub['Ts'] <= ow_max_sst) & (sub[col_c_ice] <= 0.0))
    ow_df = sub[ow_eligible].dropna(subset=[col_22, col_31])
    ow_idx = pd.Index([])
    if len(ow_df) > 0:
        z22 = np.abs((ow_df[col_22] - ow_df[col_22].mean()) / ow_df[col_22].std())
        z31 = np.abs((ow_df[col_31] - ow_df[col_31].mean()) / ow_df[col_31].std())
        ow_idx = ow_df[(z22 < ow_outlier_sigma) & (z31 < ow_outlier_sigma)].index

    return record.index.isin(fyi_idx), record.index.isin(myi_idx), record.index.isin(ow_idx)


def _period_tb_tiepoint_grid_v2(record, mask_path, region_name, period_key, period_days=PERIOD_DAYS):
    if period_key is None:
        return None, None
    period_start = datetime.strptime(period_key, '%Y%m%d')
    period_end = period_start + timedelta(days=period_days - 1)
    fyi_flag, myi_flag, ow_flag = _period_dynamic_tiepoint_flags(record, period_start, period_end)

    dt = pd.to_datetime(record['datetime'])
    period_sel = (dt >= period_start) & (dt < period_end + timedelta(days=1))
    period_df = record[period_sel].copy()
    period_df['fyi_flag'] = fyi_flag[period_sel].astype(float)
    period_df['myi_flag'] = myi_flag[period_sel].astype(float)
    period_df['ow_flag'] = ow_flag[period_sel].astype(float)
    if period_df.empty:
        print(f"  [{region_name}] no observations in window {period_start.date()}-{period_end.date()}.")
        return None, None

    resampled = resample_to_ease2(period_df, region_name,
                                   ['tb_31_phys_corrected', 'fyi_flag', 'myi_flag', 'ow_flag'],
                                   mask_path=mask_path, month=None)
    fyi_frac, myi_frac, ow_frac = resampled[:, :, 1], resampled[:, :, 2], resampled[:, :, 3]
    tp_class = np.zeros(fyi_frac.shape, dtype=np.int8)
    tp_class[fyi_frac >= 0.5] = 1
    tp_class[myi_frac >= 0.5] = 2
    tp_class[ow_frac >= 0.5] = 3
    return resampled[:, :, 0], tp_class


FIG6 = {}
for hemi, record in (("N", arctic_record), ("S", antarctic_record)):
    pk = FIG6_PERIODS[hemi]
    mask_path = ARCTIC_LANDMASK_PATH if hemi == "N" else ANTARCTIC_LANDMASK_PATH
    filled, raw, land, prov = load_period_file(hemi, pk)
    tb31, tpc = _period_tb_tiepoint_grid_v2(record, mask_path, FIG6_REGION[hemi], pk)
    t0 = datetime.strptime(pk, "%Y%m%d")
    nic, nic_dates = nic_for_window(t0, t0 + timedelta(days=PERIOD_DAYS - 1)) if hemi == "N" else (None, [])
    FIG6[hemi] = dict(pk=pk, label=period_label(pk), filled=filled, raw=raw, land=land, prov=prov,
                      unc=period_uncertainty_pct(hemi, pk), tb31=tb31, tpc=tpc, nic=nic, nic_dates=nic_dates)
    print(f"[{hemi}H] {period_label(pk)}: reconstructed (not NEMS-retrieved) ocean cells "
          f"{int((np.isnan(raw) & np.isfinite(filled) & ~land).sum()):,}"
          + (f"; NIC charts {[d.strftime('%Y-%m-%d') for d in nic_dates]}" if hemi == "N" else ""))

In [ ]:
# =============================================================================
# 5g. Fig. 6: NEMS diagnostics for one 14-day period per hemisphere
# =============================================================================
# a-c / g-i: retrieved SIC, 31.4 GHz Tb, tie-point samples; d-f / j-k: total
# uncertainty (product), gap-filled SIC, NEMS - NIC (mean of the charts inside the
# window). The pole hole is excluded from NEMS - NIC.

# --- Colours and sizes
MM = 1 / 25.4
INK, FRAME, MUTED, LAND_COL = "#1A1A1A", "#3C3C3C", "#5A5A5A", "#B0B0B0"
BLUE_DEEP, BLUE_DARK, BLUE, BLUE_LIGHT = "#16324D", "#1E5079", "#3498C3", "#7ABBD6"
ORANGE_LIGHT, ORANGE, ORANGE_DARK = "#ECA458", "#F58220", "#A94A16"
ICE_NORM = mcolors.Normalize(vmin=0, vmax=100)
CMAP_TB = mcolors.LinearSegmentedColormap.from_list(
    "tb_seq", [BLUE_DEEP, BLUE_DARK, BLUE, BLUE_LIGHT, ORANGE_LIGHT, ORANGE, ORANGE_DARK])
TB_NORM = mcolors.Normalize(vmin=130, vmax=260)
CMAP_ERR = mcolors.LinearSegmentedColormap.from_list("err_seq", [ORANGE_LIGHT, ORANGE, ORANGE_DARK])
ERR_NORM = mcolors.Normalize(vmin=0, vmax=30)
DIFF_NORM = mcolors.Normalize(vmin=-50, vmax=50)


def _rgba(data, cmap, norm, land):
    """RGBA image: land grey, NaN (not land) black, values through cmap."""
    rgba = np.zeros((*data.shape, 4))
    rgba[land] = mcolors.to_rgba(LAND_COL)
    nodata = np.isnan(data) & ~land
    rgba[nodata] = mcolors.to_rgba("black")
    ok = ~np.isnan(data) & ~land
    rgba[ok] = cmap(norm(data[ok]))
    return rgba


# --- Common Arctic map window: the NIC chart domain (NSIDC NA25 EASE-Grid
# subset, 361 x 361 cells of 25.067525 km, i.e. +/- 4525 km from the pole)
NH_MAP_HALF_WIDTH_M = 180.5 * 25067.525


def _nh_map_window(shape, cell_m=25000.0):
    def _one(n):
        x = (np.arange(n) - (n - 1) / 2) * cell_m
        keep = np.flatnonzero(np.abs(x) <= NH_MAP_HALF_WIDTH_M)
        return slice(int(keep[0]), int(keep[-1]) + 1)
    return _one(shape[0]), _one(shape[1])


# --- NEMS - NIC for the Arctic period (pole hole excluded: not NEMS data)
_n = FIG6["N"]
nh_pole_hole = MASKS["N"]["pole_cap"] if "MASKS" in globals() else np.zeros(_n["land"].shape, bool)
nh_nems_nic_diff = None
if _n["nic"] is not None:
    _both = np.isfinite(_n["nic"]) & np.isfinite(_n["filled"]) & ~_n["land"] & ~nh_pole_hole
    nh_nems_nic_diff = np.where(_both, _n["filled"] - _n["nic"], np.nan).astype(np.float32)
FIG6["N"]["diff"], FIG6["S"]["diff"] = nh_nems_nic_diff, None
_nic_days = ", ".join(f"{d.day} {d:%b}" for d in _n["nic_dates"])

# --- Figure: layout, titles and colours
# Drawn at print size (FIG_WIDTH_MM ~ 0.95 \linewidth), so the font sizes below
# are the sizes on the page. SIC uses a scale that runs to white at 100 %, so
# contrasts inside the pack are visible. A labelled rule under each block
# separates the hemispheres.
FS_TITLE, FS_LETTER, FS_CBAR, FS_TICK, FS_HEMI = 8.5, 10, 8, 7, 9.5

CMAP_SIC_FIG = mcolors.LinearSegmentedColormap.from_list(
    "ice_to_white", [BLUE_DEEP, BLUE_DARK, BLUE, BLUE_LIGHT, "#FFFFFF"])
# NEMS - NIC: orange <-> light neutral <-> blue, with the two ends matched in
# lightness (OKLab L 0.41 / 0.39) so +x and -x pp look equally strong.
CMAP_DIFF_FIG = mcolors.LinearSegmentedColormap.from_list(
    "diff_balanced", ["#7A2E0B", "#C9601A", "#EFA86A", "#F2F2F2", "#86BEDC", "#2F7FB0", "#16486E"])


def _period_text(label):
    """'1973-02-25 to 1973-03-10' -> '25 Feb to 10 Mar 1973'."""
    try:
        a, b = [datetime.strptime(s.strip(), "%Y-%m-%d") for s in label.split("to")]
        return f"{a.day} {a:%b} to {b.day} {b:%b %Y}"
    except Exception:
        return label


# Fixed grid in millimetres: every map is an exact square with its own colorbar
# slot beside it, so nothing floats and the letters sit on the map corners.
MAP_MM, CB_MM, GAP_MM, TITLE_MM, RULE_MM = 43.0, 2.5, 14.0, 7.0, 7.0
L_MM, R_MM, TOP_MM, BOT_MM = 5.0, 14.0, 1.0, 2.0


def _label(ax, letter, title):
    """Bold panel letter at the map's top-left corner, title left-aligned beside it."""
    ax.text(0.0, 1.02, letter, transform=ax.transAxes, fontsize=FS_LETTER, fontweight="bold",
            color=INK, va="bottom", ha="left")
    ax.text(0.075, 1.02, title, transform=ax.transAxes, fontsize=FS_TITLE, color=INK,
            va="bottom", ha="left")


def _map(ax, cax, data, cmap, norm, land, letter, title, cbar_label, ticks=None):
    ax.imshow(_rgba(data, cmap, norm, land), origin="upper", interpolation="nearest", aspect="auto")
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_edgecolor(FRAME); sp.set_linewidth(0.5)
    _label(ax, letter, title)
    cb = fig.colorbar(plt.cm.ScalarMappable(cmap=cmap, norm=norm), cax=cax, ticks=ticks)
    cb.set_label(cbar_label, fontsize=FS_CBAR, color=INK, labelpad=3)
    cb.ax.tick_params(labelsize=FS_TICK, colors=INK, width=0.4, length=2)
    cb.outline.set_edgecolor(FRAME); cb.outline.set_linewidth(0.4)
    return ax


def _hemisphere_rule(ax, text):
    ax.axis("off")
    ax.plot([0.0, 1.0], [0.35, 0.35], transform=ax.transAxes, color=FRAME, lw=0.7, clip_on=False)
    ax.text(0.5, 0.35, text, transform=ax.transAxes, ha="center", va="center", fontsize=FS_HEMI,
            fontweight="bold", color=INK, bbox=dict(facecolor="white", edgecolor="none", pad=4))


_wr = [MAP_MM, CB_MM, GAP_MM, MAP_MM, CB_MM, GAP_MM, MAP_MM, CB_MM]
_hr = [RULE_MM, TITLE_MM, MAP_MM, TITLE_MM, MAP_MM,      # Northern Hemisphere block
       RULE_MM + 5, TITLE_MM, MAP_MM, TITLE_MM, MAP_MM]  # Southern Hemisphere block
_W = L_MM + sum(_wr) + R_MM
_H = TOP_MM + sum(_hr) + BOT_MM
fig = plt.figure(figsize=(_W * MM, _H * MM))
gs = fig.add_gridspec(len(_hr), len(_wr), width_ratios=_wr, height_ratios=_hr, wspace=0, hspace=0,
                      left=L_MM / _W, right=1 - R_MM / _W, top=1 - TOP_MM / _H, bottom=BOT_MM / _H)
_MAP_ROWS = {"NH": (2, 4), "SH": (7, 9)}
_RULE_ROW = {"NH": 0, "SH": 5}


def _slot(row, col):
    """(map axes, colorbar axes) for panel column col (0-2) in grid row `row`."""
    return fig.add_subplot(gs[row, 3 * col]), fig.add_subplot(gs[row, 3 * col + 1])


for hemi, name, letters in (("N", "Northern Hemisphere", "abcdef"), ("S", "Southern Hemisphere", "ghijkl")):
    B = FIG6[hemi]
    rA, rB = _MAP_ROWS["NH" if hemi == "N" else "SH"]
    L = letters
    _hemisphere_rule(fig.add_subplot(gs[_RULE_ROW["NH" if hemi == "N" else "SH"], :]),
                     f"{name}, {_period_text(B['label'])}")
    land = B["land"]
    tb_tp = (np.where(B["tpc"] > 0, B["tb31"], np.nan) if B["tpc"] is not None
             else np.full(land.shape, np.nan, np.float32))
    tb31 = B["tb31"] if B["tb31"] is not None else np.full(land.shape, np.nan, np.float32)
    fields = dict(raw=B["raw"], tb31=tb31, tp=tb_tp, unc=B["unc"], filled=B["filled"], diff=B["diff"])
    if hemi == "N":                      # Arctic panels show the NIC chart domain
        win = _nh_map_window(land.shape)
        fields = {k: (None if v is None else np.asarray(v)[win]) for k, v in fields.items()}
        land = land[win]

    _map(*_slot(rA, 0), fields["raw"], CMAP_SIC_FIG, ICE_NORM, land, L[0],
         "NEMS SIC, retrieved", "SIC [%]", ticks=[0, 25, 50, 75, 100])
    _map(*_slot(rA, 1), fields["tb31"], CMAP_TB, TB_NORM, land, L[1], "NEMS T$_B$, 31.4 GHz", "T$_B$ [K]")
    _map(*_slot(rA, 2), fields["tp"], CMAP_TB, TB_NORM, land, L[2], "Tie points (FYI, MYI, OW)", "T$_B$ [K]")
    _map(*_slot(rB, 0), fields["unc"], CMAP_ERR, ERR_NORM, land, L[3], "Total SIC uncertainty", "Uncertainty [pp]")
    _map(*_slot(rB, 1), fields["filled"], CMAP_SIC_FIG, ICE_NORM, land, L[4],
         "NEMS SIC, gap-filled", "SIC [%]", ticks=[0, 25, 50, 75, 100])
    ax_f, cax_f = _slot(rB, 2)
    if fields["diff"] is not None:
        _map(ax_f, cax_f, fields["diff"], CMAP_DIFF_FIG, DIFF_NORM, land, L[5],
             f"NEMS - NIC ({_nic_days})", "Difference [pp]", ticks=[-50, -25, 0, 25, 50])
    else:                                # no NIC charts for the Antarctic: the slot holds the key
        ax_f.axis("off"); cax_f.axis("off")
        ax_f.legend(handles=[
            mpatches.Patch(facecolor=LAND_COL, edgecolor=FRAME, linewidth=0.3, label="Land"),
            mpatches.Patch(facecolor="black", edgecolor=FRAME, linewidth=0.3, label="No data"),
        ], loc="upper left", fontsize=FS_CBAR, frameon=False, handlelength=1.6, borderaxespad=0.0)
        ax_f.text(0.0, 0.08, "Panels a-f show the NIC chart\narea. There are no NIC charts\nfor the Southern Hemisphere\nin 1973.",
                  transform=ax_f.transAxes, fontsize=FS_TICK, color=MUTED, va="bottom")

for ext in ("pdf", "png"):
    fig.savefig(PLOT_DIR / f"fig6_nems_diagnostics.{ext}", dpi=300)
print(f"Saved {PLOT_DIR / 'fig6_nems_diagnostics.pdf'}")
plt.show()

In [ ]:
# =============================================================================
# 5h. NEMS - NIC statistics for the Fig. 6 period, and the consistency of the
#     stated uncertainty with that comparison
# =============================================================================
diff = nh_nems_nic_diff
nic = FIG6["N"]["nic"]
nems = FIG6["N"]["filled"]
ok = np.isfinite(diff)
v = diff[ok]
NIC_STATS = dict(
    n=int(ok.sum()), frac_grid=ok.mean(), mean=v.mean(), median=np.median(v), std=v.std(),
    rms=np.sqrt(np.mean(v ** 2)), p5=np.percentile(v, 5), p95=np.percentile(v, 95),
    within_10pp=np.mean(np.abs(v) <= 10), below_50=np.mean(v <= -50), above_50=np.mean(v >= 50),
    below_50_area_mkm2=(v <= -50).sum() * PIXEL_AREA_KM2 / 1e6,
    chart_dates=[d.strftime("%Y-%m-%d") for d in FIG6["N"]["nic_dates"]])

# Distance to the NIC 30 % ice edge
ice30 = np.where(np.isfinite(nic), nic >= SIE_THRESHOLD, False)
edge = ice30 & binary_dilation(~ice30 & np.isfinite(nic))   # ice cells touching open water
dist_km = distance_transform_edt(~edge) * 25.0
near = dist_km <= 100.0
NIC_STATS["below_50_near_edge"] = np.mean(near[ok][v <= -50]) if (v <= -50).any() else np.nan
NIC_STATS["all_near_edge"] = np.mean(near[ok])

# NEMS - NIC by NIC concentration class, per region
REGIONS = {"Labrador Sea": dict(lat=(50, 62), lon=(-65, -45)),
           "White/Barents Sea": dict(lat=(66, 80), lon=(20, 60))}
NIC_BINS = [0, 15, 40, 70, 90, 100.01]
lon_g, lat_g = _get_area_definition("arctic").get_lonlats()
REGION_BINS = {}
for name, rng in REGIONS.items():
    m = ok & (lat_g >= rng["lat"][0]) & (lat_g <= rng["lat"][1]) & (lon_g >= rng["lon"][0]) & (lon_g <= rng["lon"][1])
    cls = pd.cut(nic[m], NIC_BINS, right=False)
    REGION_BINS[name] = pd.Series(diff[m]).groupby(cls, observed=False).agg(["mean", "size"])
    print(f"\n{name}: NEMS - NIC [pp] by NIC concentration class\n{REGION_BINS[name].round(1)}")

print("\nNEMS - NIC over the whole chart domain:")
for k, val in NIC_STATS.items():
    print(f"  {k:<22}{val:.4f}" if isinstance(val, (float, np.floating)) else f"  {k:<22}{val}")

# ---------------------------------------------------------------------------
# Is the stated per-cell uncertainty consistent with the chart comparison?
# Near the ice edge the difference is dominated by the footprint, so that part of
# the domain does not test the retrieval uncertainty. Away from the edge it does,
# up to the chart's own error.
unc = FIG6["N"]["unc"]
EDGE_BANDS = {}
for lo, hi in [(0.0, 100.0), (100.0, 300.0), (300.0, np.inf)]:
    m = ok & (dist_km > lo) & (dist_km <= hi)
    if not m.any():
        continue
    m = m & np.isfinite(unc)
    d, u = diff[m], unc[m]
    EDGE_BANDS[f"{lo:.0f}-{hi:.0f} km"] = dict(
        n=int(m.sum()), mean=float(d.mean()), rms=float(np.sqrt(np.mean(d ** 2))),
        stated_unc=float(np.sqrt(np.mean(u ** 2))) if u.size else float("nan"))

far = ok & (dist_km > 100.0) & np.isfinite(unc)
d_far, u_far = diff[far], unc[far]
NIC_STATS["rms_beyond_100km"] = float(np.sqrt(np.mean(d_far ** 2)))
# The per-cell uncertainty is strongly skewed (zero over the climatological ice-free
# ocean, largest at the ice edge), so the quantity to combine with the chart error is
# its root mean square, not its mean.
NIC_STATS["unc_beyond_100km"] = float(np.sqrt(np.mean(u_far ** 2)))
NIC_STATS["unc_mean_beyond_100km"] = float(u_far.mean())
# If the NEMS uncertainty and the charts' own +-5 to +-10 pp error are independent,
# the difference between the two products should scatter by their sum in quadrature.
for cu in (5.0, 10.0):
    NIC_STATS[f"expected_rms_chart{cu:.0f}pp"] = float(
        np.hypot(NIC_STATS["unc_beyond_100km"], cu))

print("\nNEMS - NIC by distance from the NIC 30 % ice edge:")
for k, b in EDGE_BANDS.items():
    print(f"  {k:<14} n {b['n']:>7,}  mean {b['mean']:+6.2f}  RMS {b['rms']:6.2f}  "
          f"RMS stated uncertainty {b['stated_unc']:5.2f} pp")
print(f"  beyond 100 km: RMS {NIC_STATS['rms_beyond_100km']:.2f} pp against an RMS stated uncertainty of "
      f"{NIC_STATS['unc_beyond_100km']:.2f} pp (mean {NIC_STATS['unc_mean_beyond_100km']:.2f}); with a chart error of 5 to 10 pp the expected spread is "
      f"{NIC_STATS['expected_rms_chart5pp']:.2f} to {NIC_STATS['expected_rms_chart10pp']:.2f} pp")

In [ ]:
# =============================================================================
# 5h2. Reported uncertainty by provenance class, over the whole 14-day product
# =============================================================================
# Answers how the uncertainty of a reconstructed or pole-hole cell compares with
# that of a NEMS retrieval. Restricted to cells with SIC >= 30 % (the extent threshold), because the
# retrieval uncertainty is identically zero outside the climatological maximum
# extent and an average over the full grid is dominated by those zeros.
PROV_LABEL = {0: "land, lake or coast", 1: "NEMS retrieval", 2: "ESMR period mean",
              3: "GNN reconstruction", 4: "pole hole, ESMR same period",
              5: "pole hole, ESMR in time", 6: "pole hole, from the rim",
              7: "nearest-neighbour fill"}
PROV_UNC = {}
for hemi, folder in (("N", "Northern_Hemisphere"), ("S", "Southern_Hemisphere")):
    files = sorted(glob.glob(str(GNN_FILLED_DIR / folder / "**" / "*.nc"), recursive=True))
    if not files:
        print(f"[{hemi}] no 14-day files found; run Part 4 first")
        continue
    unc, prov, sic = [], [], []
    for f in files:
        with xr.open_dataset(f) as d:
            unc.append(np.asarray(d["total_standard_error"]).ravel())
            prov.append(np.asarray(d["provenance_flag"]).ravel())
            sic.append(np.asarray(d["ice_conc"]).ravel())
    unc, prov, sic = np.concatenate(unc), np.concatenate(prov), np.concatenate(sic)
    ice = np.isfinite(unc) & (sic >= SIE_THRESHOLD)     # SIC >= 30 %
    rows = []
    for k in sorted(PROV_LABEL):
        m = ice & (prov == k)
        if not m.any():
            continue
        u = unc[m]
        rows.append(dict(flag=k, source=PROV_LABEL[k], cells=int(m.sum()),
                         mean=float(u.mean()), median=float(np.median(u)),
                         p95=float(np.percentile(u, 95))))
    PROV_UNC[hemi] = pd.DataFrame(rows).set_index("flag")
    print(f"\n[{hemi}] reported uncertainty [pp] over cells with SIC >= 30 %, by provenance")
    print(PROV_UNC[hemi].round(2).to_string())
    (VAL_DIR / f"uncertainty_by_provenance_{hemi}H.csv").write_text(PROV_UNC[hemi].to_csv())

if len(PROV_UNC) == 2:
    for hemi in ("N", "S"):
        t = PROV_UNC[hemi]
        if 1 in t.index and 3 in t.index:
            print(f"[{hemi}] NEMS retrieval {t.loc[1, 'mean']:.1f} pp against GNN reconstruction "
                  f"{t.loc[3, 'mean']:.1f} pp (spread of the reconstruction value, 95th minus median: "
                  f"{t.loc[3, 'p95'] - t.loc[3, 'median']:.2f} pp)")

In [ ]:
# =============================================================================
# 5l. Daily NEMS against same-day ESMR, cell by cell
# =============================================================================
# NEMS and ESMR flew on the same platform, so on the days both recorded data
# (December 1972 - February 1973, June, October 1973) their daily fields are
# near-simultaneous. This is the only pixel-level check of the daily product
# (Dataset 1), the only pixel-level check in the Antarctic (there are no charts),
# and a test of the reported retrieval uncertainty: if the NEMS and ESMR
# uncertainties are realistic and independent, the NEMS - ESMR differences should
# scatter by their sum in quadrature, and about 68 % of them should fall inside it.
#
# Cells: ocean (lmask = 0), both products defined on the same day. "Ice cells" are
# those where either product holds >= 30 % ice (SIE_THRESHOLD); elsewhere both products are usually
# open water (often zero by the climatological mask) and agree trivially.
ICE_STRAT = SIE_THRESHOLD              # 30 %, the extent threshold used throughout (comparability with ESMR/SCAMS papers)


def _diff_stats(d, u=None):
    """Summary of a difference sample d [pp]; u = combined stated uncertainty [pp]."""
    d = np.asarray(d, np.float64)
    if d.size == 0:
        return dict(n=0)
    s = dict(n=int(d.size), mean=float(d.mean()), median=float(np.median(d)), std=float(d.std()),
             rms=float(np.sqrt(np.mean(d ** 2))), p5=float(np.percentile(d, 5)),
             p95=float(np.percentile(d, 95)), within_10pp=float(np.mean(np.abs(d) <= 10)))
    if u is not None:
        u = np.asarray(u, np.float64)
        s["unc_rms"] = float(np.sqrt(np.mean(u ** 2)))
        s["within_1sigma"] = float(np.mean(np.abs(d) <= u))
    return s


def _edge_distance_km(field):
    """Distance [km] to the 30 % ice edge of `field` (ice cells touching open water)."""
    ice = np.where(np.isfinite(field), field >= SIE_THRESHOLD, False)
    edge = ice & binary_dilation(~ice & np.isfinite(field))
    if not edge.any():
        return np.full(field.shape, np.inf)
    return distance_transform_edt(~edge) * 25.0


NEMS_ESMR = {}
NEMS_ESMR_DAILY = []
for hemi, nems, esmr in (("N", nh_nems, nh_esmr), ("S", sh_nems, sh_esmr)):
    ocean = OCEAN[hemi]
    pole = MASKS[hemi]["pole_cap"]
    acc = defaultdict(list)
    for day in sorted(set(nems) & set(esmr)):
        e = load_ice(esmr[day])
        if np.isfinite(e[ocean]).mean() < ESMR_MIN_VALID_FRAC:      # unusable ESMR day
            continue
        n = load_ice(nems[day])
        un, ue = 100.0 * load_unc(nems[day]), 100.0 * load_unc(esmr[day])
        both = ocean & ~pole & np.isfinite(n) & np.isfinite(e)
        if not both.any():
            continue
        ice = both & ((n >= ICE_STRAT) | (e >= ICE_STRAT))
        dist = _edge_distance_km(np.where(ocean, e, np.nan))
        ucomb = np.hypot(un, ue)
        d = n - e
        acc["d_all"].append(d[both])
        acc["d_ice"].append(d[ice])
        acc["u_ice"].append(ucomb[ice])
        acc["un_ice"].append(un[ice])
        acc["ue_ice"].append(ue[ice])
        acc["dist_ice"].append(dist[ice])
        acc["e_ice"].append(e[ice])
        acc["month"].append(np.full(int(ice.sum()), pd.Timestamp(day).month))
        NEMS_ESMR_DAILY.append(dict(hemi=hemi, date=pd.Timestamp(day), cells=int(both.sum()),
                                    ice_cells=int(ice.sum()),
                                    bias_ice=float(d[ice].mean()) if ice.any() else np.nan,
                                    rms_ice=float(np.sqrt(np.mean(d[ice] ** 2))) if ice.any() else np.nan))
    cat = {k: np.concatenate(v) for k, v in acc.items()}
    ok_u = np.isfinite(cat["u_ice"])
    res = dict(days=len([r for r in NEMS_ESMR_DAILY if r["hemi"] == hemi]),
               all=_diff_stats(cat["d_all"]),
               ice=_diff_stats(cat["d_ice"][ok_u], cat["u_ice"][ok_u]))
    res["ice"]["unc_nems_rms"] = float(np.sqrt(np.nanmean(cat["un_ice"] ** 2)))
    res["ice"]["unc_esmr_rms"] = float(np.sqrt(np.nanmean(cat["ue_ice"] ** 2)))
    near = cat["dist_ice"] <= 100.0                  # includes the edge cells themselves
    res["ice_0-100km"] = _diff_stats(cat["d_ice"][ok_u & near], cat["u_ice"][ok_u & near])
    res["ice_100-infkm"] = _diff_stats(cat["d_ice"][ok_u & ~near], cat["u_ice"][ok_u & ~near])
    # by ESMR concentration class: is a bias confined to the edge or present in the pack?
    for lo, hi in [(0, 30), (30, 70), (70, 90), (90, 101)]:
        m = ok_u & (cat["e_ice"] >= lo) & (cat["e_ice"] < hi)
        res[f"ice_esmr{lo}-{min(hi, 100)}"] = _diff_stats(cat["d_ice"][m], cat["u_ice"][m])
    for label, months in (("Dec-Feb", (12, 1, 2)), ("Jun", (6,)), ("Oct", (10,))):
        m = ok_u & np.isin(cat["month"], months)
        res[f"ice_{label}"] = _diff_stats(cat["d_ice"][m], cat["u_ice"][m])
    NEMS_ESMR[hemi] = res

NEMS_ESMR_DAILY = pd.DataFrame(NEMS_ESMR_DAILY)
NEMS_ESMR_DAILY.to_csv(VAL_DIR / "nems_esmr_daily.csv", index=False)
for hemi, res in NEMS_ESMR.items():
    print(f"\n[{hemi}H] NEMS - ESMR, same day, {res['days']} days")
    for k, s in res.items():
        if isinstance(s, dict) and s.get("n"):
            extra = (f"  stated RMS {s['unc_rms']:5.1f}  inside 1 sigma {100 * s['within_1sigma']:4.0f} %"
                     if "unc_rms" in s else "")
            print(f"  {k:<16} n {s['n']:>9,}  mean {s['mean']:+6.2f}  median {s['median']:+6.2f}  "
                  f"RMS {s['rms']:6.2f}  within 10 pp {100 * s['within_10pp']:5.1f} %{extra}")
    print(f"  stated RMS uncertainty over ice cells: NEMS {res['ice']['unc_nems_rms']:.1f}, "
          f"ESMR {res['ice']['unc_esmr_rms']:.1f} pp")

In [ ]:
# =============================================================================
# 5m. NEMS - NIC in every 14-day window, and in the ESMR outages in particular
# =============================================================================
# The Fig. 6 statistics (5h) cover one window, 25 Feb - 10 Mar, and over the whole
# chart domain they are dominated by open water that both products agree on. Here
# the same comparison is made for every window with NIC charts, over all cells and
# over the cells where either product holds >= 30 % ice, for the gap-filled field
# and for the NEMS-retrieved cells alone (provenance 1). Pole hole excluded, as in 5h.
NIC_WINDOWS = []
NIC_WINDOW_BANDS = {}
_pfiles = {file_date(p): p for p in GNN_FILLED_DIR.rglob("*_NH-*.nc") if GNN_FILLED_MASKED_DIR not in p.parents}
_reg = val_reg_df[val_reg_df.hemi == "NH"].set_index(val_reg_df[val_reg_df.hemi == "NH"].window_start.dt.strftime("%Y%m%d"))
pole = MASKS["N"]["pole_cap"]
for pk in sorted(nh_periods):
    filled, raw, land, prov = load_period_file("N", pk)
    t0 = datetime.strptime(pk, "%Y%m%d")
    nic, nic_dates = nic_for_window(t0, t0 + timedelta(days=PERIOD_DAYS - 1))
    if nic is None:
        continue
    both = np.isfinite(nic) & np.isfinite(filled) & ~land & ~pole      # same cells as 5h
    d = filled - nic
    ice = both & ((filled >= ICE_STRAT) | (nic >= ICE_STRAT))
    ret = ice & (prov == 1)
    unc = period_uncertainty_pct("N", pk)
    dist = _edge_distance_km(np.where(np.isfinite(nic), nic, np.nan))
    n_esmr = int(_reg.loc[pk, "n_esmr_days"]) if pk in _reg.index else 0
    row = dict(window=pk, label=period_label(pk), outage=n_esmr < ESMR_MIN_DAYS, esmr_days=n_esmr,
               charts=", ".join(f"{x:%d %b}" for x in nic_dates), nems_days=len(nh_periods[pk]))
    for tag, m in (("all", both), ("ice", ice), ("ice_nems", ret)):
        s = _diff_stats(d[m])
        row.update({f"{tag}_{k}": v for k, v in s.items()})
    # share of the ice cells that NEMS itself retrieved, and that the network reconstructed
    row["ice_frac_nems"] = float(ret.sum() / max(ice.sum(), 1))
    row["ice_frac_gnn"] = float((ice & (prov == 3)).sum() / max(ice.sum(), 1))
    # stated vs observed spread away from the chart ice edge (NEMS-retrieved cells)
    far = ret & (dist > 100.0) & np.isfinite(unc)
    near = ret & (dist <= 100.0) & np.isfinite(unc)
    for tag, m in (("far", far), ("near", near)):
        row[f"{tag}_n"] = int(m.sum())
        row[f"{tag}_rms"] = float(np.sqrt(np.mean(d[m] ** 2))) if m.any() else np.nan
        row[f"{tag}_unc_rms"] = float(np.sqrt(np.mean(unc[m] ** 2))) if m.any() else np.nan
    NIC_WINDOWS.append(row)
NIC_WINDOWS = pd.DataFrame(NIC_WINDOWS)
NIC_WINDOWS.to_csv(VAL_DIR / "nems_nic_by_window.csv", index=False)

_show = ["label", "outage", "charts", "all_n", "all_within_10pp", "ice_n", "ice_mean", "ice_median",
         "ice_rms", "ice_within_10pp", "ice_nems_n", "ice_nems_mean", "ice_nems_rms", "ice_nems_within_10pp",
         "ice_frac_gnn", "far_rms", "far_unc_rms", "near_rms", "near_unc_rms"]
with pd.option_context("display.width", 250, "display.max_columns", 30):
    print(NIC_WINDOWS[_show].to_string(index=False, float_format=lambda v: f"{v:.2f}"))

_o = NIC_WINDOWS[NIC_WINDOWS.outage]
_c = NIC_WINDOWS[~NIC_WINDOWS.outage]
print(f"\nOutage windows ({', '.join(_o.window)}): ice-cell RMS {_o.ice_rms.min():.1f}-{_o.ice_rms.max():.1f} pp, "
      f"bias {_o.ice_mean.min():+.1f} to {_o.ice_mean.max():+.1f} pp, within 10 pp "
      f"{100 * _o.ice_within_10pp.min():.0f}-{100 * _o.ice_within_10pp.max():.0f} %")
print(f"Windows with ESMR: ice-cell RMS {_c.ice_rms.min():.1f}-{_c.ice_rms.max():.1f} pp, "
      f"bias {_c.ice_mean.min():+.1f} to {_c.ice_mean.max():+.1f} pp, within 10 pp "
      f"{100 * _c.ice_within_10pp.min():.0f}-{100 * _c.ice_within_10pp.max():.0f} %")

In [ ]:
# =============================================================================
# 5n. Where the extent of each 14-day window comes from (provenance), and the
#     extent counted over NEMS-retrieved cells only
# =============================================================================
# The gap-filled extent mixes NEMS retrievals with ESMR fill, network
# reconstruction and, in the Arctic, the pole hole (ESMR of the same window, or
# ESMR interpolated in time across an outage). This splits every window's extent
# (SIC >= 30 %) by provenance, so the outage extents can be read for what NEMS
# itself contributes.
PROV_GROUPS = {"nems": (1,), "esmr": (2,), "gnn": (3,), "pole_same": (4,), "pole_interp": (5, 6), "nn": (7,)}
EXTENT_BY_PROV = []
for hemi in ("N", "S"):
    periods = nh_periods if hemi == "N" else sh_periods
    for pk in sorted(periods):
        filled, raw, land, prov = load_period_file(hemi, pk)
        ocean = OCEAN[hemi] & np.isfinite(filled)
        ice = ocean & (filled >= SIE_THRESHOLD)
        row = dict(hemi=hemi, window=pk, total=ice.sum() * PIXEL_AREA_KM2 / 1e6,
                   gnn_cells_frac=float((prov == 3).sum() / prov.size))
        for name, flags in PROV_GROUPS.items():
            row[name] = (ice & np.isin(prov, flags)).sum() * PIXEL_AREA_KM2 / 1e6
        pole_desc = PERIOD_PRODUCTS[(hemi, pk)]["pole_source"] if "PERIOD_PRODUCTS" in globals() else ""
        row["pole_source"] = pole_desc
        EXTENT_BY_PROV.append(row)
EXTENT_BY_PROV = pd.DataFrame(EXTENT_BY_PROV)
_reg2 = val_reg_df.assign(window=val_reg_df.window_start.dt.strftime("%Y%m%d"),
                          hemi=val_reg_df.hemi.str[0])[["hemi", "window", "n_esmr_days", "NIC"]]
EXTENT_BY_PROV = EXTENT_BY_PROV.merge(_reg2, on=["hemi", "window"], how="left")
EXTENT_BY_PROV["outage"] = EXTENT_BY_PROV.n_esmr_days < ESMR_MIN_DAYS
for k in PROV_GROUPS:
    EXTENT_BY_PROV[f"{k}_pct"] = 100 * EXTENT_BY_PROV[k] / EXTENT_BY_PROV.total
EXTENT_BY_PROV.to_csv(VAL_DIR / "extent_by_provenance.csv", index=False)
with pd.option_context("display.width", 250, "display.max_columns", 30):
    print(EXTENT_BY_PROV[["hemi", "window", "outage", "total", "nems", "esmr", "gnn", "pole_same", "pole_interp",
                          "nn", "nems_pct", "gnn_pct", "pole_same_pct", "pole_interp_pct", "NIC", "gnn_cells_frac"]]
          .to_string(index=False, float_format=lambda v: f"{v:.2f}"))

# NEMS-only extent against the charts over the same cells (Arctic): the part of
# the outage-window extent that NEMS observed, checked against the charts there.
NIC_SAME_CELLS = []
for pk in sorted(nh_periods):
    filled, raw, land, prov = load_period_file("N", pk)
    t0 = datetime.strptime(pk, "%Y%m%d")
    nic, _ = nic_for_window(t0, t0 + timedelta(days=PERIOD_DAYS - 1))
    m = OCEAN["N"] & (prov == 1) & np.isfinite(nic) & np.isfinite(filled)
    NIC_SAME_CELLS.append(dict(window=pk,
                               nems=(m & (filled >= SIE_THRESHOLD)).sum() * PIXEL_AREA_KM2 / 1e6,
                               nic=(m & (nic >= SIE_THRESHOLD)).sum() * PIXEL_AREA_KM2 / 1e6))
NIC_SAME_CELLS = pd.DataFrame(NIC_SAME_CELLS)
NIC_SAME_CELLS["diff"] = NIC_SAME_CELLS.nems - NIC_SAME_CELLS.nic
NIC_SAME_CELLS = NIC_SAME_CELLS.merge(EXTENT_BY_PROV[EXTENT_BY_PROV.hemi == "N"][["window", "outage"]], on="window")
print("\nArctic extent over NEMS-retrieved cells, NEMS against NIC on the same cells [million km2]")
print(NIC_SAME_CELLS.to_string(index=False, float_format=lambda v: f"{v:.2f}"))
_a = agreement(NIC_SAME_CELLS.nems, NIC_SAME_CELLS.nic)
print(f"all windows: bias {_a['bias']:+.2f}, RMSE {_a['rmse']:.2f}; outage windows: "
      + ", ".join(f"{r.window} {r['diff']:+.2f}" for _, r in NIC_SAME_CELLS[NIC_SAME_CELLS.outage].iterrows()))

In [ ]:
# =============================================================================
# 5o. Does the climatological mask clip 1973 ice?
# =============================================================================
# The daily retrieval sets SIC to zero outside the monthly climatological maximum
# extent of the ESA CCI LandOceanLakeMask v2 (variable `climatology`, value 0 =
# open water; derived from the modern passive-microwave record and expanded by
# 150 km). 1973 may have had ice beyond a modern maximum, so count the chart and
# ESMR ice that falls in the masked area.
CLIM_CHECK = []
for hemi, path in (("N", ARCTIC_LANDMASK_PATH), ("S", ANTARCTIC_LANDMASK_PATH)):
    with xr.open_dataset(path) as _m:
        clim = _m["climatology"].values
    esmr = nh_esmr if hemi == "N" else sh_esmr
    for month in range(1, 13):
        masked = OCEAN[hemi] & (clim[month - 1] == 0)
        e_days = [d for d in esmr if int(d[4:6]) == month and d.startswith("1973")]
        e_ice = e_tot = 0.0
        for d in e_days:
            e = load_ice(esmr[d])
            e_ice += float((masked & (e >= SIE_THRESHOLD)).sum())
            e_tot += float((OCEAN[hemi] & (e >= SIE_THRESHOLD)).sum())
        row = dict(hemi=hemi, month=month, esmr_days=len(e_days),
                   esmr_ice_outside_pct=100 * e_ice / e_tot if e_tot else np.nan,
                   esmr_ice_outside_km2=e_ice / max(len(e_days), 1) * PIXEL_AREA_KM2)
        if hemi == "N":
            charts = [f for f, t in nic_file_dates_all if t.year == 1973 and t.month == month]
            n_ice = n_tot = 0.0
            for f in charts:
                c = nic_sic_ease2(f)
                n_ice += float((masked & (c >= SIE_THRESHOLD)).sum())
                n_tot += float((OCEAN[hemi] & (c >= SIE_THRESHOLD)).sum())
            row.update(nic_charts=len(charts), nic_ice_outside_pct=100 * n_ice / n_tot if n_tot else np.nan,
                       nic_ice_outside_km2=n_ice / max(len(charts), 1) * PIXEL_AREA_KM2)
        CLIM_CHECK.append(row)
CLIM_CHECK = pd.DataFrame(CLIM_CHECK)
CLIM_CHECK.to_csv(VAL_DIR / "climatology_mask_check.csv", index=False)
print(CLIM_CHECK.to_string(index=False, float_format=lambda v: f"{v:.2f}"))
for h in ("N", "S"):
    t = CLIM_CHECK[CLIM_CHECK.hemi == h]
    msg = f"[{h}H] ESMR ice (>= 30 %) outside the mask: max {t.esmr_ice_outside_pct.max():.2f} % of ESMR extent"
    if h == "N":
        msg += f"; NIC: max {t.nic_ice_outside_pct.max():.2f} % ({t.nic_ice_outside_km2.max() / 1e3:.0f} thousand km2)"
    print(msg)

In [ ]:
# =============================================================================
# 5p. Footprint-scale reference from same-day ESMR, for the observation-level checks
# =============================================================================
# On the days both instruments recorded, each NEMS observation is placed on the EASE2
# grid and given the ESMR concentration of that day over a 9 x 9-cell window
# (225 km, the footprint scale used for the land-spillover correction): the window
# mean, minimum and maximum. This classifies observations as clean open water
# (no ESMR ice anywhere in the window) or consolidated ice (ESMR >= 90 % everywhere
# in the window) without using ERA5, so it is independent of the RTM correction and
# of the tie-point selection, both of which use ERA5 SIC.
FOOT_PX = SPILLOVER_WINDOW_PX
from scipy.ndimage import maximum_filter, minimum_filter


def _esmr_footprint(e):
    fin = np.isfinite(e)
    cover = uniform_filter(fin.astype(np.float32), FOOT_PX, mode="constant")
    mean = uniform_filter(np.where(fin, e, 0).astype(np.float32), FOOT_PX, mode="constant") / np.maximum(cover, 1e-6)
    mx = maximum_filter(np.where(fin, e, -np.inf), FOOT_PX, mode="constant", cval=-np.inf)
    mn = minimum_filter(np.where(fin, e, np.inf), FOOT_PX, mode="constant", cval=np.inf)
    ok = cover >= 0.5
    return np.where(ok, mean, np.nan), np.where(ok, mx, np.nan), np.where(ok, mn, np.nan), cover


def attach_esmr_footprint(record, hemi):
    esmr = nh_esmr if hemi == "N" else sh_esmr
    area = AREA_DEF[hemi]
    rec = record.copy()
    for c in ("esmr_fp_mean", "esmr_fp_max", "esmr_fp_min", "esmr_fp_cover"):
        rec[c] = np.nan
    day = pd.to_datetime(rec["datetime"]).dt.strftime("%Y%m%d")
    for d in sorted(set(day) & set(esmr)):
        e = load_ice(esmr[d])
        if np.isfinite(e[OCEAN[hemi]]).mean() < ESMR_MIN_VALID_FRAC:
            continue
        mean, mx, mn, cover = _esmr_footprint(e)
        sel = (day == d).to_numpy()
        cols, rows = area.get_array_indices_from_lonlat(rec.loc[sel, "lon"].to_numpy(), rec.loc[sel, "lat"].to_numpy())
        cols, rows = np.ma.filled(cols, -1), np.ma.filled(rows, -1)
        inside = (cols >= 0) & (rows >= 0)
        idx = np.flatnonzero(sel)[inside]
        rec.loc[rec.index[idx], "esmr_fp_mean"] = mean[rows[inside], cols[inside]]
        rec.loc[rec.index[idx], "esmr_fp_max"] = mx[rows[inside], cols[inside]]
        rec.loc[rec.index[idx], "esmr_fp_min"] = mn[rows[inside], cols[inside]]
        rec.loc[rec.index[idx], "esmr_fp_cover"] = cover[rows[inside], cols[inside]]
    return rec


FP_RECORD = {"N": attach_esmr_footprint(arctic_record, "N"), "S": attach_esmr_footprint(antarctic_record, "S")}
for h, r in FP_RECORD.items():
    print(f"[{h}H] observations with a same-day ESMR footprint value: {r.esmr_fp_mean.notna().sum():,} of {len(r):,}")

In [ ]:
# =============================================================================
# 5q. What the RTM correction does, judged against an ESMR reference
# =============================================================================
# Spread of the retrieved SIC before (measured Tb, first-pass tie points) and after
# (corrected Tb, second-pass tie points) the correction, over clean open water and
# consolidated ice as defined by ESMR in 5p. The concentrations are recomputed here
# WITHOUT the [0, 1] clipping the product applies, because a clipped value cannot show
# its spread: over open water the one-channel 22 GHz SIC is the relevant estimate
# (the hybrid uses it below 40 %), over consolidated ice the two-channel estimate.
# The one-channel 31.4 GHz estimate is computed alongside, to test the statement in
# Methods that 22.235 GHz gives the less noisy one-channel open-water SIC.
def _sic1(tb, fy, my, ow):
    return 100.0 * (tb - ow) / ((fy + my) / 2 - ow)


def _sic2(tb31, tb22, fy22, my22, ow22, fy31, my31, ow31):
    af = (fy31 - my31) / (fy22 - my22)
    bf = my31 - af * my22
    qf = (tb31 - ow31) / (tb22 - ow22)
    wf = ow31 - qf * ow22
    ti = (bf - wf) / (qf - af)
    return 100.0 * (tb22 - ow22) / (ti - ow22)


RTM_EFFECT = {}
for h, r in FP_RECORD.items():
    s = {"before": "", "after": "_corrected"}
    tb = {"before": ("tb_22_measured", "tb_31_measured"), "after": ("tb_22_phys_corrected", "tb_31_phys_corrected")}
    ow = (r.esmr_fp_max <= 0.0)
    ice = (r.esmr_fp_min >= 90.0)
    month = pd.to_datetime(r["datetime"]).dt.month
    res = {}
    for k in ("before", "after"):
        t22, t31 = r[tb[k][0]], r[tb[k][1]]
        g = lambda n: r[f"{n}{s[k]}"]
        sic1_22 = _sic1(t22, g("FYI_tp22"), g("MYI_tp22"), g("OW_tp22"))
        sic1_31 = _sic1(t31, g("FYI_tp31"), g("MYI_tp31"), g("OW_tp31"))
        sic2 = _sic2(t31, t22, g("FYI_tp22"), g("MYI_tp22"), g("OW_tp22"), g("FYI_tp31"), g("MYI_tp31"), g("OW_tp31"))
        for name, v, m in (("ow_1ch22", sic1_22, ow), ("ow_1ch31", sic1_31, ow), ("ice_2ch", sic2, ice),
                           ("ice_1ch22", sic1_22, ice)):
            x = v[m & np.isfinite(v)]
            res[(k, name)] = dict(n=int(x.size), mean=float(x.mean()), std=float(x.std()))
        for label, months in (("Dec-Feb", (12, 1, 2)), ("Jun", (6,)), ("Oct", (10,))):
            mm = np.isin(month, months)
            for name, v, m in (("ow_1ch22", sic1_22, ow), ("ice_2ch", sic2, ice)):
                x = v[m & mm & np.isfinite(v)]
                res[(k, f"{name}_{label}")] = dict(n=int(x.size), mean=float(x.mean()) if x.size else np.nan,
                                                   std=float(x.std()) if x.size else np.nan)
    RTM_EFFECT[h] = res
    print(f"\n[{h}H] SIC over ESMR-defined surfaces, unclipped [%]: mean / standard deviation (n)")
    for key in sorted({kk for _, kk in res}):
        b, a = res[("before", key)], res[("after", key)]
        print(f"  {key:<18} before {b['mean']:+7.1f} / {b['std']:5.1f}   after {a['mean']:+7.1f} / {a['std']:5.1f}   (n {a['n']:,})")

In [ ]:
# =============================================================================
# 5r. Footprint contamination of the tie points, and its effect on the retrieval
# =============================================================================
# The tie-point candidates are selected from the value at the footprint centre:
# ERA5 SIC = 0 with SST 271-275 K for open water (both passes), ERA5 SIC >= 95 %
# (first pass) or first-pass hybrid SIC >= 95 % (second pass) for ice. A 200 x 300 km
# footprint near the ice edge can still hold ice (open-water candidates) or open
# water (ice candidates). On the days with ESMR, the candidates are checked against
# the ESMR footprint values from 5p, the tie points are recomputed per 14-day window
# from the candidates ESMR confirms as clean (no ESMR ice in the window for open
# water; ESMR >= 90 % everywhere in the window for ice) with the same procedure
# (IQR screening, upper / lower 10 % for FYI / MYI, z-score < 1 for open water), and
# the effect of the change on the retrieved SIC is evaluated on the same observations.
def _ice_tp(df, c22, c31):
    clean = remove_outliers_multivariate(df, columns=[c22, c31], method="iqr", iqr_multiplier=TP_IQR_MULTIPLIER)
    out = {}
    for ch, col in (("22", c22), ("31", c31)):
        v = np.sort(clean[col].to_numpy())
        n = max(1, int(len(v) * 0.10))
        out[f"FYI_tp{ch}"], out[f"MYI_tp{ch}"] = v[-n:].mean(), v[:n].mean()
    return out


def _ow_tp(df, c22, c31):
    z22 = np.abs((df[c22] - df[c22].mean()) / df[c22].std())
    z31 = np.abs((df[c31] - df[c31].mean()) / df[c31].std())
    core = df[(z22 < 1) & (z31 < 1)]
    return {"OW_tp22": core[c22].mean(), "OW_tp31": core[c31].mean()}


TP_CONTAM = {}
TP_SENS_ROWS = []
for h, r in FP_RECORD.items():
    c22, c31 = "tb_22_phys_corrected", "tb_31_phys_corrected"
    has = r.esmr_fp_mean.notna()
    ow_cand = has & (r.c_ice <= 0.0) & r.Ts.between(*OW_TP_SST)
    ice_cand = has & (r.c_ice_hybrid >= ICE_TP_MIN_SIC)
    TP_CONTAM[h] = dict(
        ow_n=int(ow_cand.sum()), ow_any_ice=float((r.esmr_fp_max[ow_cand] >= SIE_THRESHOLD / 2).mean()),
        ow_fp_mean=float(r.esmr_fp_mean[ow_cand].mean()),
        ice_n=int(ice_cand.sum()), ice_not_pack=float((r.esmr_fp_min[ice_cand] < 90.0).mean()),
        ice_fp_mean=float(r.esmr_fp_mean[ice_cand].mean()))
    # Tie points are pooled over each overlap block (Dec-Feb, Jun, Oct) rather than per
    # 14-day window, because clean open-water candidates are too few per window. Both the
    # standard and the clean set are pooled the same way, so their difference isolates
    # the effect of footprint contamination. "Clean" also requires the footprint window
    # to be at least 90 % ocean, so that coastal (land-contaminated) footprints are not
    # mistaken for clean open water.
    mon = pd.to_datetime(r["datetime"]).dt.month
    for label, months in (("Dec-Feb", (12, 1, 2)), ("Jun", (6,)), ("Oct", (10,))):
        inw = np.isin(mon, months)
        std_ice, std_ow = r[ice_cand & inw], r[ow_cand & inw]
        cln_ice = r[ice_cand & inw & (r.esmr_fp_min >= 90.0) & (r.esmr_fp_cover >= 0.9)]
        cln_ow = r[ow_cand & inw & (r.esmr_fp_max <= 0.0) & (r.esmr_fp_cover >= 0.9)]
        if min(len(std_ice), len(cln_ice), len(std_ow), len(cln_ow)) < 30:
            continue
        tp_std = {**_ice_tp(std_ice, c22, c31), **_ow_tp(std_ow, c22, c31)}
        tp_cln = {**_ice_tp(cln_ice, c22, c31), **_ow_tp(cln_ow, c22, c31)}
        obs = r[inw & has]
        args = lambda tp: (obs[c22].to_numpy(), obs[c31].to_numpy(), tp["FYI_tp22"], tp["MYI_tp22"], tp["OW_tp22"],
                           tp["FYI_tp31"], tp["MYI_tp31"], tp["OW_tp31"])
        sic_std, sic_cln = 100 * hybrid(*args(tp_std)), 100 * hybrid(*args(tp_cln))
        pack = ((obs.esmr_fp_min >= 90.0) & (obs.esmr_fp_cover >= 0.9)).to_numpy()
        water = ((obs.esmr_fp_max <= 0.0) & (obs.esmr_fp_cover >= 0.9)).to_numpy()
        TP_SENS_ROWS.append(dict(hemi=h, block=label, n_ow_std=len(std_ow), n_ow_cln=len(cln_ow),
                                 n_ice_std=len(std_ice), n_ice_cln=len(cln_ice),
                                 **{f"d_{k}": tp_cln[k] - tp_std[k] for k in tp_std},
                                 n_pack=int(pack.sum()), n_water=int(water.sum()),
                                 sic_pack_std=float(sic_std[pack].mean()), sic_pack_cln=float(sic_cln[pack].mean()),
                                 sic_water_std=float(sic_std[water].mean()), sic_water_cln=float(sic_cln[water].mean()),
                                 esmr_pack=float(obs.esmr_fp_mean[pack].mean())))
TP_SENS = pd.DataFrame(TP_SENS_ROWS)
TP_SENS["d_sic_pack"] = TP_SENS.sic_pack_cln - TP_SENS.sic_pack_std
TP_SENS["d_sic_water"] = TP_SENS.sic_water_cln - TP_SENS.sic_water_std
TP_SENS.to_csv(VAL_DIR / "tiepoint_footprint_sensitivity.csv", index=False)
for h in ("N", "S"):
    c, t = TP_CONTAM[h], TP_SENS[TP_SENS.hemi == h]
    w = lambda col, wt: float(np.average(t[col], weights=t[wt]))
    print(f"\n[{h}H] open-water candidates {c['ow_n']:,}: {100 * c['ow_any_ice']:.0f} % have ESMR ice >= 15 % in the footprint "
          f"(mean footprint ESMR {c['ow_fp_mean']:.1f} %); ice candidates {c['ice_n']:,}: {100 * c['ice_not_pack']:.0f} % have "
          f"ESMR < 90 % somewhere in the footprint (mean {c['ice_fp_mean']:.1f} %)")
    print(t[["block", "n_ow_std", "n_ow_cln", "n_ice_std", "n_ice_cln"] + [k for k in t.columns if k.startswith("d_")]
            + ["sic_pack_std", "sic_pack_cln", "esmr_pack"]].to_string(index=False, float_format=lambda v: f"{v:+.2f}"))
    print(f"  effect on SIC [pp]: consolidated ice {w('d_sic_pack', 'n_pack'):+.2f} "
          f"(standard {w('sic_pack_std', 'n_pack'):.1f} %, clean {w('sic_pack_cln', 'n_pack'):.1f} %, ESMR {w('esmr_pack', 'n_pack'):.1f} %); "
          f"clean open water {w('d_sic_water', 'n_water'):+.2f} (standard {w('sic_water_std', 'n_water'):.1f} %)")

In [ ]:
# =============================================================================
# 5i. Figs. 7 and 8: regional zoom, NIC / ESMR / NEMS / NEMS - NIC
# =============================================================================
# Labrador Sea and White/Barents Seas for one NEMS window: NIC = mean of the charts
# dated inside the window, ESMR = mean of the ESMR days inside it (none during the
# outage), NEMS = gap-filled product.
REGIONAL_ZOOM_PERIODS = ["19730225", "19730422"]   # Fig. 7 (ESMR available), Fig. 8 (ESMR outage)
nh_land = FIG6["N"]["land"] if "FIG6" in globals() else load_period_file("N", REGIONAL_ZOOM_PERIODS[0])[2]
nh_lon_grid, nh_lat_grid = _get_area_definition("arctic").get_lonlats()
REGIONS_6G = {
    "Labrador Sea": dict(lat=(50, 62), lon=(-65, -45)),
    "White and Barents Seas": dict(lat=(66, 80), lon=(20, 60)),
}

# --- Colours, fonts and sizes (same as Fig. 6)
MM = 1 / 25.4
INK, FRAME, MUTED, LAND_COL = "#1A1A1A", "#3C3C3C", "#5A5A5A", "#B0B0B0"
BLUE_DEEP, BLUE_DARK, BLUE, BLUE_LIGHT = "#16324D", "#1E5079", "#3498C3", "#7ABBD6"
CMAP_SIC_6G = mcolors.LinearSegmentedColormap.from_list(
    "ice_to_white", [BLUE_DEEP, BLUE_DARK, BLUE, BLUE_LIGHT, "#FFFFFF"])
CMAP_DIFF_6G = mcolors.LinearSegmentedColormap.from_list(
    "diff_balanced", ["#7A2E0B", "#C9601A", "#EFA86A", "#F2F2F2", "#86BEDC", "#2F7FB0", "#16486E"])
SIC_NORM_6G = mcolors.Normalize(vmin=0, vmax=100)
DIFF_NORM_6G = mcolors.Normalize(vmin=-50, vmax=50)
FS_TITLE, FS_LETTER, FS_CBAR, FS_TICK, FS_RULE = 8.5, 10, 8, 7, 9.5


def _pct(arr):
    """SIC in percent whether loaded as fractions, percent, or percent x 100."""
    if arr is None:
        return None
    a = np.asarray(arr, dtype=np.float32)
    finite = a[np.isfinite(a)]
    if not finite.size:
        return a
    mx = float(finite.max())
    return a * 100.0 if mx <= 1.5 else (a / 100.0 if mx > 150.0 else a)


def _na25_to_ease2(field_na25, lat, lon):
    """NIC chart on the NSIDC NA25 EASE-Grid (361 x 361) -> the EASE2 grid given by lat/lon."""
    R, C, c0, n = 6371.228, 25.067525, 180, 361
    lat = np.asarray(lat, float); lon = np.asarray(lon, float)
    ok = np.isfinite(lat) & np.isfinite(lon)
    rho = 2 * R / C * np.sin(np.pi / 4 - np.radians(np.where(ok, lat, 90.0)) / 2)
    lam = np.radians(np.where(ok, lon, 0.0))
    col = np.rint(rho * np.sin(lam) + c0).astype(int)
    row = np.rint(rho * np.cos(lam) + c0).astype(int)
    inside = ok & (row >= 0) & (row < n) & (col >= 0) & (col < n)
    out = np.full(lat.shape, np.nan, dtype=np.float32)
    out[inside] = np.asarray(field_na25, dtype=np.float32)[row[inside], col[inside]]
    return out


def _rgba(data, cmap, norm, land):
    rgba = np.zeros((*data.shape, 4))
    rgba[land] = mcolors.to_rgba(LAND_COL)
    rgba[np.isnan(data) & ~land] = mcolors.to_rgba("black")
    ok = ~np.isnan(data) & ~land
    rgba[ok] = cmap(norm(data[ok]))
    return rgba


def _day(d):
    return f"{d.day} {d:%b %Y}" if d is not None else ""


# --- Same period for every source: the NEMS 14-day window
# NIC = mean of the charts dated inside the window, ESMR = mean of the ESMR days
# inside it. Falls back to the nearest chart / the single ESMR file from the 6f
# data cell only when the window has none.


def _period_bounds(label):
    a, b = [datetime.strptime(s.strip(), "%Y-%m-%d") for s in str(label).split("to")]
    return a, b


def _read_nic_ease2(path, shape, land, lat, lon):
    """One NIC chart on the EASE2 grid, SIC in % (NaN = land / not covered)."""
    helper = globals().get("nic_sic_ease2")               # loader from 5b
    if helper is not None:
        return _pct(helper(path))
    p = str(path)
    if p.endswith(".bin"):                                 # NSIDC NA25 chart, 361 x 361
        a = np.fromfile(p, dtype=np.uint8).reshape(361, 361).astype(np.float32)
        a[a == 108] = 100.0
        a[np.isin(a, (157, 253, 254, 255))] = np.nan
        return _na25_to_ease2(a, lat, lon)
    import xarray as xr                                    # chart already on EASE2 (.nc)
    with xr.open_dataset(p) as ds:
        v = [n for n in ds.data_vars if ds[n].ndim >= 2][0]
        a = np.squeeze(ds[v].values).astype(np.float32)
    chart_land = a == 254
    a[a == 108] = 100.0
    a[np.isin(a, (157, 253, 254, 255)) | (a < 0) | (a > 100)] = np.nan
    if chart_land.any() and (np.flipud(chart_land) & land).sum() > (chart_land & land).sum():
        a = np.flipud(a)                                   # file stored with y the other way round
    return _pct(a)


def _nic_for_period(t0, t1, shape, land, lat, lon, fallback=None, fallback_date=None):
    """-> (field, description). Mean of all charts dated t0..t1."""
    charts = [(f, d) for f, d in (globals().get("nic_file_dates_all") or []) if t0 <= d <= t1]
    if charts:
        stack = np.stack([_read_nic_ease2(f, shape, land, lat, lon) for f, _ in charts], 0)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            field = np.nanmean(stack, axis=0).astype(np.float32)
        dates = ", ".join(f"{d.day} {d:%b}" for _, d in charts)
        return field, f"mean of {len(charts)} chart{'s' if len(charts) > 1 else ''} ({dates})"
    near = [(f, d) for f, d in (globals().get("nic_file_dates_all") or [])
            if t0 - timedelta(days=7) <= d <= t1 + timedelta(days=7)]
    if near:
        mid = t0 + (t1 - t0) / 2
        f, d = min(near, key=lambda fd: abs(fd[1] - mid))
        print(f"  NOTE: no NIC chart inside {t0:%d %b} to {t1:%d %b %Y}; using the chart of {d:%d %b}")
        return _read_nic_ease2(f, shape, land, lat, lon), f"chart of {d.day} {d:%b} (outside the period)"
    if fallback is not None:
        when = f"{fallback_date.day} {fallback_date:%b %Y}" if fallback_date is not None else "nearest date"
        print(f"  NOTE: no NIC chart inside {t0:%d %b} to {t1:%d %b %Y}; using the chart of {when}")
        return fallback, f"chart of {when} (outside the period)"
    return None, "no chart"


def _esmr_for_period(t0, t1, fallback=None, fallback_date=None):
    """-> (field, description). Mean of the daily ESMR files t0..t1."""
    esmr_files = globals().get("nh_esmr") or {}
    days = sorted(d for d in esmr_files if t0 <= datetime.strptime(d, "%Y%m%d") <= t1)
    if days:
        reader = globals().get("load_ice")
        if reader is None:
            import xarray as xr

            def reader(path):
                with xr.open_dataset(path) as ds:
                    return ds["ice_conc"].values[0].astype(np.float32)
        stack = np.stack([_pct(reader(esmr_files[d])) for d in days], 0)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            field = np.nanmean(stack, axis=0).astype(np.float32)
        return field, f"mean of {len(days)} day{'s' if len(days) > 1 else ''}"
    if fallback is not None:
        when = f"{fallback_date.day} {fallback_date:%b %Y}" if fallback_date is not None else "nearest date"
        print(f"  NOTE: no ESMR day inside the period; using the ESMR field of {when}")
        return fallback, f"{when} (outside the period)"
    return None, "no ESMR"


def _load_nems_period(pk):
    """Gap-filled NEMS SIC [%] for the period starting pk, from the Part 4 files."""
    import xarray as xr
    roots = [GNN_FILLED_MASKED_DIR, GNN_FILLED_DIR]
    for root in roots:
        hits = sorted(p for p in root.rglob(f"*_NH-{pk}*.nc")
                      if "_polehole_smoothed" not in p.stem and not p.name.endswith(".tmp.nc"))
        if hits:
            with xr.open_dataset(hits[0]) as ds:
                return _pct(ds["ice_conc"].values[0]), hits[0]
    known = sorted(globals().get("nh_periods") or [])
    raise FileNotFoundError(f"No NH period file for {pk} under {[str(r) for r in roots] or 'GNN_FILLED_DIR'}."
                            + (f" NEMS periods: {known}" if known else ""))




def regional_zoom(period_key):
    # --- Fields, all in percent on the EASE2 grid, all for the same period
    land = np.asarray(nh_land, bool)
    nems, _nems_file = _load_nems_period(period_key)
    t0 = datetime.strptime(period_key, "%Y%m%d")
    t1 = t0 + timedelta(days=PERIOD_DAYS - 1)
    nic, nic_desc = _nic_for_period(t0, t1, nems.shape, land, nh_lat_grid, nh_lon_grid)
    esmr, esmr_desc = _esmr_for_period(t0, t1)
    if esmr is None:
        esmr_desc = "none (ESMR outage)"
    period_text = f"{t0.day} {t0:%b} to {t1.day} {t1:%b %Y}"
    diff = None
    if nic is not None:
        _both = np.isfinite(nic) & np.isfinite(nems) & ~land
        diff = np.where(_both, nems - nic, np.nan).astype(np.float32)


    def _square_window(lat_range, lon_range, pad=5, max_shift_frac=0.4):
        """Square row/col window around a lat/lon box, nudged (by up to 40 % of its
        size) to the position with the fewest ocean cells lacking NIC (or ESMR) data."""
        m = ((nh_lat_grid >= lat_range[0]) & (nh_lat_grid <= lat_range[1]) &
             (nh_lon_grid >= lon_range[0]) & (nh_lon_grid <= lon_range[1]))
        rows, cols = np.where(m)
        if rows.size == 0:
            return None
        r0, r1, c0, c1 = rows.min() - pad, rows.max() + 1 + pad, cols.min() - pad, cols.max() + 1 + pad
        side = max(r1 - r0, c1 - c0)
        H, W = m.shape
        side = min(side, H, W)
        rc, cc = (r0 + r1) // 2, (c0 + c1) // 2
        r0 = int(np.clip(rc - side // 2, 0, H - side)); c0 = int(np.clip(cc - side // 2, 0, W - side))
        ocean = ~land
        bad = np.zeros(land.shape, np.float32)
        if nic is not None:
            bad += (np.isnan(nic) & ocean)
        if esmr is not None:
            bad += 0.1 * (np.isnan(esmr) & ocean)
        if not bad.any():
            return slice(r0, r0 + side), slice(c0, c0 + side)
        cum = np.pad(bad.cumsum(0).cumsum(1), ((1, 0), (1, 0)))          # summed-area table
        box = lambda r, c: cum[r + side, c + side] - cum[r, c + side] - cum[r + side, c] + cum[r, c]
        smax = int(side * max_shift_frac)
        best = None
        for dr in range(-smax, smax + 1):
            for dc in range(-smax, smax + 1):
                r, c = r0 + dr, c0 + dc
                if 0 <= r <= H - side and 0 <= c <= W - side:
                    key = (round(float(box(r, c)), 2), dr * dr + dc * dc)
                    if best is None or key < best[0]:
                        best = (key, r, c)
        _, r0, c0 = best
        return slice(r0, r0 + side), slice(c0, c0 + side)


    # --- Layout in millimetres: four square maps per row, one shared SIC colourbar
    # after the third map and one difference colourbar after the fourth
    MAP_MM, CB_MM, SEP_MM, LAB_MM, TITLE_MM, RULE_MM, KEY_MM = 34.0, 2.5, 5.0, 13.0, 7.0, 7.0, 7.0
    L_MM, R_MM, TOP_MM, BOT_MM = 2.0, 13.0, 1.0, 1.0
    _wr = [MAP_MM, SEP_MM, MAP_MM, SEP_MM, MAP_MM, CB_MM, LAB_MM, MAP_MM, CB_MM]
    _hr = [RULE_MM, TITLE_MM, MAP_MM, RULE_MM + 4, TITLE_MM, MAP_MM, KEY_MM]
    _W, _H = L_MM + sum(_wr) + R_MM, TOP_MM + sum(_hr) + BOT_MM
    fig6g = plt.figure(figsize=(_W * MM, _H * MM))
    gs6g = fig6g.add_gridspec(len(_hr), len(_wr), width_ratios=_wr, height_ratios=_hr, wspace=0, hspace=0,
                              left=L_MM / _W, right=1 - R_MM / _W, top=1 - TOP_MM / _H, bottom=BOT_MM / _H)
    _MAP_COLS, _SIC_CB_COL, _DIFF_CB_COL = (0, 2, 4, 7), 5, 8
    _ROWS = [(0, 2), (3, 5)]          # (rule row, map row) per region


    def _label(ax, letter, title):
        ax.text(0.0, 1.03, letter, transform=ax.transAxes, fontsize=FS_LETTER, fontweight="bold",
                color=INK, va="bottom", ha="left")
        ax.text(0.095, 1.03, title, transform=ax.transAxes, fontsize=FS_TITLE, color=INK,
                va="bottom", ha="left")


    def _map(ax, data, cmap, norm, land_c, letter, title, empty_msg):
        ax.set_xticks([]); ax.set_yticks([])
        for sp in ax.spines.values():
            sp.set_edgecolor(FRAME); sp.set_linewidth(0.5)
        _label(ax, letter, title)
        if data is None or not np.isfinite(data[~land_c]).any():
            ax.set_facecolor("white")
            ax.text(0.5, 0.5, empty_msg, ha="center", va="center", fontsize=FS_TICK, color=MUTED,
                    transform=ax.transAxes)
            return
        ax.imshow(_rgba(data, cmap, norm, land_c), origin="upper", interpolation="nearest", aspect="auto")


    def _cbar(cax, cmap, norm, label, ticks):
        cb = fig6g.colorbar(plt.cm.ScalarMappable(cmap=cmap, norm=norm), cax=cax, ticks=ticks)
        cb.set_label(label, fontsize=FS_CBAR, color=INK, labelpad=3)
        cb.ax.tick_params(labelsize=FS_TICK, colors=INK, width=0.4, length=2)
        cb.outline.set_edgecolor(FRAME); cb.outline.set_linewidth(0.4)


    def _rule(ax, text):
        ax.axis("off")
        ax.plot([0.0, 1.0], [0.35, 0.35], transform=ax.transAxes, color=FRAME, lw=0.7, clip_on=False)
        ax.text(0.5, 0.35, text, transform=ax.transAxes, ha="center", va="center", fontsize=FS_RULE,
                fontweight="bold", color=INK, bbox=dict(facecolor="white", edgecolor="none", pad=4))


    titles = ("NIC chart", "ESMR", "NEMS, gap-filled", "NEMS - NIC")
    letters = "abcdefgh"
    for k, ((region, rng), (r_rule, r_map)) in enumerate(zip(REGIONS_6G.items(), _ROWS)):
        _rule(fig6g.add_subplot(gs6g[r_rule, :]), f"{region}, {period_text}")
        win = _square_window(rng["lat"], rng["lon"])
        if win is None:
            ax = fig6g.add_subplot(gs6g[r_map, :]); ax.axis("off")
            ax.text(0.5, 0.5, f"No grid cells in the {region} box", ha="center", va="center",
                    fontsize=FS_TITLE, color=MUTED, transform=ax.transAxes)
            continue
        land_c = land[win]
        crop = lambda a: None if a is None else a[win]
        panels = ((crop(nic), CMAP_SIC_6G, SIC_NORM_6G, "No NIC chart"),
                  (crop(esmr), CMAP_SIC_6G, SIC_NORM_6G, "No ESMR data\n(ESMR outage)"),
                  (crop(nems), CMAP_SIC_6G, SIC_NORM_6G, "No NEMS data"),
                  (crop(diff), CMAP_DIFF_6G, DIFF_NORM_6G, "No NIC chart"))
        for j, (col, (data, cmap, norm, msg)) in enumerate(zip(_MAP_COLS, panels)):
            _map(fig6g.add_subplot(gs6g[r_map, col]), data, cmap, norm, land_c, letters[4 * k + j],
                 titles[j], msg)
        _cbar(fig6g.add_subplot(gs6g[r_map, _SIC_CB_COL]), CMAP_SIC_6G, SIC_NORM_6G, "SIC [%]",
              [0, 25, 50, 75, 100])
        _cbar(fig6g.add_subplot(gs6g[r_map, _DIFF_CB_COL]), CMAP_DIFF_6G, DIFF_NORM_6G, "Difference [pp]",
              [-50, -25, 0, 25, 50])

    key = fig6g.add_subplot(gs6g[len(_hr) - 1, :]); key.axis("off")
    key.legend(handles=[mpatches.Patch(facecolor=LAND_COL, edgecolor=FRAME, linewidth=0.3, label="Land"),
                        mpatches.Patch(facecolor="black", edgecolor=FRAME, linewidth=0.3, label="No data")],
               loc="center", ncol=2, fontsize=FS_CBAR, frameon=False, handlelength=1.6, columnspacing=2.0)

    print(f"Period: {period_text} (NEMS 14-day window)")
    print(f"  NIC:  {nic_desc}")
    print(f"  ESMR: {esmr_desc}")

    saved = []
    for ext in ("pdf", "png"):
        out_path = PLOT_DIR / f"fig_regional_zoom_{period_key}.{ext}"
        fig6g.savefig(out_path, dpi=300)
        saved.append(str(out_path))
    print("Saved: " + ", ".join(saved))
    plt.show()


for _pk in REGIONAL_ZOOM_PERIODS:
    regional_zoom(_pk)

In [ ]:
# =============================================================================
# 5j. Figs. 2 and 3: tie-point locations and screening
# =============================================================================
# fig_tiepoint_locations   sea-ice and open-water tie-point observations by month
#                          (first pass: ERA5 SIC >= 95 % and SST 271-275 K with ERA5 SIC = 0)
# fig_filter_attrition     observations per day surviving each screening step
# fig_filter_removal_<Mon> where each screening step removes observations in one month
# --- Settings ---------------------------------------------------------------
OUT_DIR = PLOT_DIR
SCREEN_MONTH = 3                  # month shown in the screening map
NH_HALF_WIDTH_M = 180.5 * 25067.525    # NIC chart domain (as all Arctic maps)
SH_HALF_WIDTH_M = 216 * 25000.0        # full EASE2 SH grid
X_START, X_END = pd.Timestamp("1972-12-01"), pd.Timestamp("1973-12-10")   # as the extent figure

# --- Style (same as the extent figure and Fig. 6) ---------------------------
MM = 1 / 25.4
FIG_W = 182.0
FS_TITLE, FS_LETTER, FS_LABEL, FS_TICK, FS_LEGEND, FS_MAP = 10.5, 11, 9.5, 8.5, 9, 7.5
INK, FRAME, MUTED, GRID_COL = "#1A1A1A", "#3C3C3C", "#5A5A5A", "#E4E6E9"
BLUE, ORANGE = "#3498C3", "#F58220"
BLUE_DEEP, BLUE_DARK, BLUE_LIGHT = "#16324D", "#1E5079", "#7ABBD6"
ORANGE_DARK, ORANGE_LIGHT, NAVY = "#A94A16", "#ECA458", "#2A3550"
C_GAP = ORANGE_LIGHT              # "NEMS only" shading, as in the extent figure
OCEAN, LAND, COAST, MAP_GRID = "#EBEFF2", "#D6D6D6", "#6E6E6E", "#9AA0A6"

MONTH_COLORS = ["#16324D", "#1E5079", "#2A76A3", "#3498C3", "#59ADD1", "#7ABBD6",
                "#ECA458", "#F58220", "#D9631A", "#A94A16", "#6B3A44", "#2A3550"]
CMAP_MONTH = mcolors.ListedColormap(MONTH_COLORS)
NORM_MONTH = mcolors.BoundaryNorm(np.arange(0.5, 13.5), 12)
MONTH_ABBR = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
MONTH_NAMES = ["January", "February", "March", "April", "May", "June", "July",
               "August", "September", "October", "November", "December"]

TB_TXT = r"$T_\mathrm{B}$"
MIN_ABS_LAT = int(MIN_ABS_LAT)
TB_RANGE = tuple(int(v) for v in TB_RANGE)
DEG = "\u00b0"

plt.rcParams.update({
    "font.family": "sans-serif", "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 9, "mathtext.fontset": "custom", "mathtext.rm": "sans",
    "mathtext.it": "sans:italic", "mathtext.bf": "sans:bold", "mathtext.cal": "sans",
    "axes.edgecolor": FRAME, "axes.linewidth": 0.5, "axes.labelcolor": INK,
    "xtick.color": INK, "ytick.color": INK, "text.color": INK, "legend.frameon": False,
    "figure.facecolor": "white", "savefig.facecolor": "white", "savefig.dpi": 300,
    "pdf.fonttype": 42, "ps.fonttype": 42,
})

EASE2_N = ccrs.LambertAzimuthalEqualArea(central_longitude=0, central_latitude=90,
                                         globe=ccrs.Globe(ellipse="WGS84"))
EASE2_S = ccrs.LambertAzimuthalEqualArea(central_longitude=0, central_latitude=-90,
                                         globe=ccrs.Globe(ellipse="WGS84"))


def _save(fig, stem):
    out = []
    for ext in ("pdf", "png"):
        p = OUT_DIR / f"{stem}.{ext}"
        fig.savefig(p)
        out.append(str(p))
    print("Saved: " + ", ".join(out))


def _ax_mm(fig, x, y_top, w, h, H, **kw):
    """Axes placed in millimetres from the figure's top-left corner."""
    return fig.add_axes([x / FIG_W, 1 - (y_top + h) / H, w / FIG_W, h / H], **kw)


# =============================================================================
# Map panel
# =============================================================================
def _map_panel(ax, pole, letter=None, title=None):
    north = pole == "north"
    h = NH_HALF_WIDTH_M if north else SH_HALF_WIDTH_M
    ax.set_extent([-h, h, -h, h], crs=ax.projection)
    ax.add_feature(cfeature.OCEAN, facecolor=OCEAN, zorder=0)
    ax.add_feature(cfeature.LAND, facecolor=LAND, zorder=10)
    ax.coastlines(resolution="50m", linewidth=0.4, color=COAST, zorder=11)
    sign = 1 if north else -1
    lats = [sign * v for v in (60, 70, 80)]
    ax.gridlines(draw_labels=False, linewidth=0.4, color=MAP_GRID, alpha=0.9,
                 linestyle=(0, (1, 2.5)), xlocs=np.arange(-180, 180, 45), ylocs=lats, zorder=12)
    lab_lon = 180 if north else 0                      # straight up from the pole
    for lat in lats:
        ax.text(lab_lon, lat, f"{abs(lat)}{DEG}{'N' if north else 'S'}", fontsize=FS_MAP,
                color=MUTED, ha="center", va="center", zorder=13, transform=ccrs.PlateCarree(),
                path_effects=[pe.withStroke(linewidth=2.0, foreground="white")])
    ax.spines["geo"].set_linewidth(0.8)
    ax.spines["geo"].set_edgecolor(INK)
    _panel_title(ax, letter, title)


def _panel_title(ax, letter=None, title=None):
    """Bold letter + plain title, left-aligned just above the panel -- the same
    placement and sizes as the extent figure ("a  Northern Hemisphere")."""
    from matplotlib.transforms import offset_copy
    pt = 72 / 25.4                                     # points per mm
    if letter:
        ax.text(0, 1, letter, fontsize=FS_LETTER, fontweight="bold", va="bottom", ha="left",
                transform=offset_copy(ax.transAxes, fig=ax.figure, x=0, y=1.6 * pt, units="points"))
    if title:
        ax.text(0, 1, title, fontsize=FS_TITLE, va="bottom", ha="left",
                transform=offset_copy(ax.transAxes, fig=ax.figure, x=4.8 * pt, y=1.6 * pt, units="points"))


# =============================================================================
# 1. Tie-point locations (selection logic as tiepoints_func)
# =============================================================================
def select_ice_pixels(df, col_22="tb_22_measured", col_31="tb_31_measured", col_c_ice="c_ice",
                      min_c_ice=0.95, min_samples=100, outlier_method="iqr", iqr_multiplier=1.0):
    """Pixels chosen as FYI/MYI tie points (14-day window, top/bottom 10 % per channel)."""
    d = df.copy()
    d["tb_22"], d["tb_31"] = d[col_22], d[col_31]
    d = d.dropna(subset=["tb_22", "tb_31"])
    if col_c_ice in d.columns:
        d = d[d[col_c_ice] >= min_c_ice]
    di = d.sort_values("datetime").set_index("datetime")
    di["_row"] = np.arange(len(di))
    rows = []
    for date in di.index.unique():
        window = di.loc[date - pd.Timedelta(days=14):date].iloc[:-1]
        if len(window) < min_samples:
            continue
        clean = remove_outliers_multivariate(window.copy(), columns=["tb_22", "tb_31"],
                                             method=outlier_method, iqr_multiplier=iqr_multiplier)
        if len(clean) < min_samples:
            continue
        n = max(1, int(len(clean) * 0.10))
        fyi = pd.Index(clean.nlargest(n, "tb_22")["_row"]).union(pd.Index(clean.nlargest(n, "tb_31")["_row"]))
        myi = pd.Index(clean.nsmallest(n, "tb_22")["_row"]).union(pd.Index(clean.nsmallest(n, "tb_31")["_row"]))
        rows += [(r, "FYI") for r in fyi] + [(r, "MYI") for r in myi.difference(fyi)]
    if not rows:
        return pd.DataFrame(columns=["lat", "lon", "datetime", "tp_class"])
    sel = pd.DataFrame(rows, columns=["_row", "tp_class"]).drop_duplicates("_row")
    return di.reset_index()[["lat", "lon", "datetime", "_row"]].merge(sel, on="_row")[
        ["lat", "lon", "datetime", "tp_class"]]


def select_ow_pixels(df, col_22="tb_22_measured", col_31="tb_31_measured",
                     min_sst=271, max_sst=275, max_c_ice=0.0, outlier_sigma=1):
    """Open-water tie-point pixels (SST, ice and z-score filters as analyze_ow_tiepoints)."""
    d = df.copy()
    d["tb_22"], d["tb_31"] = d[col_22], d[col_31]
    d = d.dropna(subset=["tb_22", "tb_31"])
    if "Ts" in d.columns:
        d = d[(d["Ts"] >= min_sst) & (d["Ts"] <= max_sst)]
    if "c_ice" in d.columns:
        d = d[d["c_ice"] <= max_c_ice]
    if len(d):
        z22 = np.abs((d["tb_22"] - d["tb_22"].mean()) / d["tb_22"].std())
        z31 = np.abs((d["tb_31"] - d["tb_31"].mean()) / d["tb_31"].std())
        d = d[(z22 < outlier_sigma) & (z31 < outlier_sigma)]
    out = d[["lat", "lon", "datetime"]].copy()
    out["tp_class"] = "OW"
    return out


def collect(record, kind):
    pts = select_ice_pixels(record) if kind == "ice" else select_ow_pixels(record)
    pts = pts[pts["lat"].abs() >= MIN_ABS_LAT].copy()
    pts["month"] = pd.to_datetime(pts["datetime"]).dt.month
    return pts


def _chrono_months(*frames):
    """Colour bins in time order from the record's first month (Dec 1972 first)."""
    pairs = set()
    for df in frames:
        if df is not None and len(df):
            dt = pd.to_datetime(df["datetime"])
            pairs.update(zip(dt.dt.year, dt.dt.month))
    by_month = {}
    for y, m in pairs:
        by_month.setdefault(m, set()).add(y)
    m0 = min(pairs)[1] if pairs else 1
    order = [((m0 - 1 + k) % 12) + 1 for k in range(12)]
    pos = {m: i + 1 for i, m in enumerate(order)}
    labels, colours = [], []
    first = True
    for m in order:
        if m in by_month:
            yrs = sorted(by_month[m])
            show_year = first or m == 1          # year under the first month and under Jan
            labels.append(f"{MONTH_ABBR[m - 1]}\n{'/'.join(map(str, yrs))}" if show_year else MONTH_ABBR[m - 1])
            colours.append(INK)
            first = False
        else:
            labels.append(MONTH_ABBR[m - 1])
            colours.append("#B0B4B8")              # no observations that month
    for m, ys in by_month.items():
        if len(ys) > 1:
            print(f"  WARNING: {MONTH_ABBR[m - 1]} occurs in {sorted(ys)}; one colour covers both.")
    return pos, labels, colours


def plot_tiepoint_locations(arctic_record, antarctic_record, marker_size=5.0):
    pts = {("ice", "N"): collect(arctic_record, "ice"), ("ice", "S"): collect(antarctic_record, "ice"),
           ("ow", "N"): collect(arctic_record, "water"), ("ow", "S"): collect(antarctic_record, "water")}
    pos, cb_labels, cb_colours = _chrono_months(*pts.values())
    for p in pts.values():
        if len(p):
            p["cpos"] = p["month"].map(pos)

    left, gap, right, top = 2.0, 4.0, 2.0, 7.0
    pw = (FIG_W - left - gap - right) / 2
    rgap, cb_gap, cb_h, bottom = 8.0, 4.0, 4.0, 15.0
    H = top + 2 * pw + rgap + cb_gap + cb_h + bottom
    fig = plt.figure(figsize=(FIG_W * MM, H * MM))

    sc = None
    letters = iter("abcd")
    for r, (kind, row_label) in enumerate((("ice", "Sea ice"), ("ow", "Open water"))):
        for c, (hemi, crs, pole) in enumerate((("N", EASE2_N, "north"), ("S", EASE2_S, "south"))):
            ax = _ax_mm(fig, left + c * (pw + gap), top + r * (pw + rgap), pw, pw, H, projection=crs)
            hemi_name = "Northern Hemisphere" if hemi == "N" else "Southern Hemisphere"
            _map_panel(ax, pole, next(letters), f"{row_label}, {hemi_name}")
            p = pts[(kind, hemi)]
            if len(p):
                sc = ax.scatter(p["lon"].to_numpy(), p["lat"].to_numpy(), s=marker_size, c=p["cpos"].to_numpy(),
                                cmap=CMAP_MONTH, norm=NORM_MONTH, alpha=0.85, linewidths=0,
                                transform=ccrs.PlateCarree(), zorder=5, rasterized=True)

    if sc is not None:
        cax = _ax_mm(fig, left, top + 2 * pw + rgap + cb_gap, FIG_W - left - right, cb_h, H)
        cb = fig.colorbar(sc, cax=cax, orientation="horizontal", ticks=np.arange(1, 13))
        cb.ax.set_xticklabels(cb_labels, fontsize=FS_TICK)
        for t, col in zip(cb.ax.get_xticklabels(), cb_colours):
            t.set_color(col)
        cb.ax.tick_params(length=0, pad=3)
        cb.set_label("Month of observation", fontsize=FS_LABEL, labelpad=4)
        cb.outline.set_linewidth(0.5)
        cb.outline.set_edgecolor(FRAME)
        cb.solids.set_alpha(1.0)

    _save(fig, "fig_tiepoint_locations")
    n = {k: len(v) for k, v in pts.items()}
    print(f"\nTie-point pixels: sea ice {n[('ice', 'N')]:,} (NH) / {n[('ice', 'S')]:,} (SH), "
          f"open water {n[('ow', 'N')]:,} (NH) / {n[('ow', 'S')]:,} (SH)")
    print("Caption: Locations of the NEMS observations selected as tie points, coloured by month of "
          "observation: (a, b) sea ice (first-year and multiyear ice) and (c, d) open water, for the "
          "Northern and Southern Hemisphere. Months without observations are shown in grey on the "
          "colour scale. Maps are on the EASE2 grid.")
    plt.show()
    return pts


# =============================================================================
# 2. Screening: attrition timeline
# =============================================================================
def build_raw_frames(ds):
    """Per-frame extraction as in 1a, but without the Tb filter."""
    frames = []
    for frame in range(14):
        V = ds["tcwv"].isel(n16_frames=frame)
        L = ds["tcw"].isel(n16_frames=frame) - V
        W = np.sqrt(ds["v10"].isel(n16_frames=frame) ** 2 + ds["u10"].isel(n16_frames=frame) ** 2)
        frames.append(pd.DataFrame({
            "lat": ds["LAT"].isel(n16_frames=frame).values.ravel(),
            "lon": ds["LON"].isel(n16_frames=frame).values.ravel(),
            "V": V.values.ravel(),
            "Ta": ds["t2m"].isel(n16_frames=frame).values.ravel(),
            "Ts": ds["sst"].isel(n16_frames=frame).values.ravel(),
            "c_ice": ds["siconc"].isel(n16_frames=frame).values.ravel(),
            "tcw": ds["tcw"].isel(n16_frames=frame).values.ravel(),
            "L": L.values.ravel(), "W": W.values.ravel(),
            "tb_22_measured": ds["TBNEMS"].isel(n5_channels=0, n14_frames=frame).values.ravel(),
            "tb_31_measured": ds["TBNEMS"].isel(n5_channels=1, n14_frames=frame).values.ravel(),
            "time": pd.to_datetime(ds["Time"].isel(n16_frames=frame).values)}))
    raw = pd.concat(frames, ignore_index=True)
    raw["datetime"] = raw["time"]
    raw["day"] = raw["datetime"].dt.floor("D")
    return raw


def _in_range(d):
    lo, hi = TB_RANGE
    return ((d["tb_22_measured"] >= lo) & (d["tb_22_measured"] <= hi) &
            (d["tb_31_measured"] >= lo) & (d["tb_31_measured"] <= hi))


STAGES = [  # (column, legend label, style)
    ("raw", "All extracted pixels", dict(color="#B4B8BC", marker="o", ls="-")),
    ("tb", f"{TB_TXT} within {TB_RANGE[0]}-{TB_RANGE[1]} K", dict(color=BLUE_LIGHT, marker="s", ls="-")),
    ("complete", "Complete records", dict(color=BLUE_DEEP, marker="D", ls="-")),
    ("nh", f"Poleward of {MIN_ABS_LAT}{DEG}N", dict(color=BLUE, marker="o", ls="-")),
    ("sh", f"Poleward of {MIN_ABS_LAT}{DEG}S", dict(color=ORANGE, marker="s", ls="--")),
]

D = pd.Timestamp
SPANS = {   # instrument availability, first and last day with data (inclusive)
    "NEMS": [(D("1972-12-17"), D("1973-03-07")), (D("1973-04-15"), D("1973-05-01")),
             (D("1973-06-17"), D("1973-06-30")), (D("1973-08-12"), D("1973-08-31")),
             (D("1973-10-14"), D("1973-10-31"))],
    "ESMR": [(D("1972-12-11"), D("1973-02-28")), (D("1973-05-29"), D("1973-08-10")),
             (D("1973-09-06"), D("1977-05-16"))],   # ESMR files: no data 1 Mar-28 May, 11 Aug-5 Sep 1973
}


def _spans_excl(name):
    """SPANS as half-open intervals [first day, last day + 1), so the last day is drawn in full."""
    return [(a, b + pd.Timedelta(days=1)) for a, b in SPANS[name]]


def _nems_only():
    out = []
    for a0, a1 in _spans_excl("NEMS"):
        cur = [(a0, a1)]
        for b0, b1 in _spans_excl("ESMR"):
            nxt = []
            for c0, c1 in cur:
                if b1 <= c0 or b0 >= c1:
                    nxt.append((c0, c1)); continue
                if c0 < b0:
                    nxt.append((c0, b0))
                if b1 < c1:
                    nxt.append((b1, c1))
            cur = nxt
        out += cur
    return sorted(out)


def _month_label(x, pos=None):
    d = mdates.num2date(x)
    return f"{d:%b}\n{d.year}" if (d.month == 1 or pos == 0) else f"{d:%b}"


def compute_stage_counts(raw):
    s1 = raw[_in_range(raw)]
    s2 = s1.dropna()
    per_day = lambda d: d.groupby("day").size()
    counts = pd.DataFrame({"raw": per_day(raw), "tb": per_day(s1), "complete": per_day(s2),
                           "nh": per_day(s2[s2["lat"] >= MIN_ABS_LAT]),
                           "sh": per_day(s2[s2["lat"] <= -MIN_ABS_LAT])}).sort_index()
    if len(counts):
        counts = counts.reindex(pd.date_range(counts.index.min(), counts.index.max(), freq="D"))
        has = counts["raw"].notna()
        counts.loc[has] = counts.loc[has].fillna(0)        # data that day, nothing left -> 0
    return counts


def plot_filter_attrition(raw):
    counts = compute_stage_counts(raw)
    H = 105.0
    fig = plt.figure(figsize=(FIG_W * MM, H * MM))
    gs = fig.add_gridspec(2, 1, height_ratios=[1, 0.17], hspace=0.12,
                          left=0.095, right=0.975, top=0.97, bottom=0.235)
    ax = fig.add_subplot(gs[0])
    ax_cov = fig.add_subplot(gs[1], sharex=ax)

    for s0, s1 in _nems_only():
        ax.axvspan(s0, s1, color=C_GAP, alpha=0.25, lw=0, zorder=0)
    for col, label, st in STAGES:
        ax.plot(counts.index, counts[col], lw=1.1, ms=2.6, mew=0, label=label, zorder=3, **st)

    top = np.nanmax(counts.to_numpy(dtype=float))
    ax.set_ylim(0, top * 1.06)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:,.0f}"))
    ax.set_ylabel("Pixels per day", fontsize=FS_LABEL)
    ax.grid(axis="y", color=GRID_COL, lw=0.5)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.tick_params(labelsize=FS_TICK, length=3, width=0.6)
    ax.tick_params(axis="x", labelbottom=False)

    # availability strip
    h = 0.55
    for i, name in enumerate(("NEMS", "ESMR")):
        y = 1 - i
        for t0, t1 in _spans_excl(name):
            ax_cov.broken_barh([(mdates.date2num(t0), mdates.date2num(t1) - mdates.date2num(t0))],
                               (y, h), facecolors="#4E5257", edgecolor="none")
    ax_cov.set_yticks([1 + h / 2, h / 2])
    ax_cov.set_yticklabels(["NEMS", "ESMR"], fontsize=FS_TICK)
    ax_cov.set_ylim(-0.2, 1 + h + 0.2)
    ax_cov.tick_params(axis="y", length=0, pad=3)
    ax_cov.tick_params(axis="x", labelsize=FS_TICK, length=3, width=0.6)
    for side in ("top", "right", "left"):
        ax_cov.spines[side].set_visible(False)
    ax_cov.set_xlim(X_START, X_END)
    ax_cov.xaxis.set_major_locator(mdates.MonthLocator())
    ax_cov.xaxis.set_major_formatter(plt.FuncFormatter(_month_label))

    handles = [Line2D([], [], lw=1.4, ms=4, mew=0, **st) for _, _, st in STAGES]
    labels = [lab for _, lab, _ in STAGES]
    handles.append(Patch(facecolor=C_GAP, alpha=0.25, lw=0)); labels.append("NEMS only (ESMR outage)")
    fig.legend(handles, labels, loc="lower center", bbox_to_anchor=(0.5, 0.015), ncol=3,
               fontsize=FS_LEGEND, handlelength=2.0, handletextpad=0.5, columnspacing=2.0,
               labelspacing=0.55, borderaxespad=0.0)

    _save(fig, "fig_filter_attrition")
    tot = counts.sum()
    print(f"\nTotal pixels by stage (raw = {tot['raw']:,.0f}):")
    for col, label, _ in STAGES:
        print(f"  {col:<9}{tot[col]:>12,.0f}   {100 * tot[col] / tot['raw']:5.1f} % of raw")
    lost = tot["complete"] - tot["nh"] - tot["sh"]
    print(f"  {'|lat|<' + str(MIN_ABS_LAT):<9}{lost:>12,.0f}   {100 * lost / tot['raw']:5.1f} % of raw (removed)")
    print("Caption: Number of NEMS pixels per day remaining after each screening step: all extracted "
          f"pixels, brightness temperatures within {TB_RANGE[0]}-{TB_RANGE[1]} K in both channels, complete "
          f"records, and the Northern and Southern Hemisphere subsets poleward of {MIN_ABS_LAT} deg. Days "
          "without data are left as gaps. The strip below shows when NEMS and ESMR data are available; "
          "shading marks periods with NEMS but no ESMR data (ESMR hot-load anomaly).")
    plt.show()
    return counts


# =============================================================================
# 3. Screening: where pixels are removed (one month)
# =============================================================================
REMOVAL = [  # key, label, colour (drawn in this order; retained underneath)
    ("kept", "Retained", "#C8D2D8"),
    ("latitude", f"|lat| < {MIN_ABS_LAT}{DEG}", BLUE_LIGHT),
    ("nan", "Incomplete record", ORANGE_DARK),
    ("tb_range", f"{TB_TXT} outside {TB_RANGE[0]}-{TB_RANGE[1]} K", ORANGE),
]


def plot_filter_removal(raw, month=SCREEN_MONTH, marker_size=2.2):
    sub = raw[raw["datetime"].dt.month == month]
    if not len(sub):
        print(f"{MONTH_NAMES[month - 1]}: no pixels, nothing to plot")
        return None
    tb22, tb31 = sub["tb_22_measured"].to_numpy(float), sub["tb_31_measured"].to_numpy(float)
    reason = np.full(len(sub), "kept", dtype=object)          # first failed step wins
    reason[(~_in_range(sub).to_numpy()) & np.isfinite(tb22) & np.isfinite(tb31)] = "tb_range"
    reason[sub.isna().any(axis=1).to_numpy() & (reason == "kept")] = "nan"
    reason[(sub["lat"].abs() < MIN_ABS_LAT).to_numpy() & (reason == "kept")] = "latitude"
    sub = sub.assign(_reason=reason)
    year = int(sub["datetime"].dt.year.mode().iloc[0])

    left, gap, right, top, leg_h = 2.0, 4.0, 2.0, 7.0, 10.0
    pw = (FIG_W - left - gap - right) / 2
    H = top + pw + leg_h
    fig = plt.figure(figsize=(FIG_W * MM, H * MM))
    for c, (crs, pole, name, letter) in enumerate(((EASE2_N, "north", "Northern Hemisphere", "a"),
                                                   (EASE2_S, "south", "Southern Hemisphere", "b"))):
        ax = _ax_mm(fig, left + c * (pw + gap), top, pw, pw, H, projection=crs)
        _map_panel(ax, pole, letter, name)
        side = sub[sub["lat"] >= 0] if pole == "north" else sub[sub["lat"] < 0]
        for key, _, col in REMOVAL:
            s = side[side["_reason"] == key]
            if len(s):
                ax.scatter(s["lon"].to_numpy(), s["lat"].to_numpy(), s=marker_size, c=col, alpha=0.8,
                           linewidths=0, transform=ccrs.PlateCarree(), zorder=5, rasterized=True)

    order = ["kept", "tb_range", "nan", "latitude"]
    lookup = {k: (lab, col) for k, lab, col in REMOVAL}
    handles = [Patch(facecolor=lookup[k][1], label=lookup[k][0]) for k in order]
    fig.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.5, 0.01), ncol=4,
               fontsize=FS_LEGEND, handlelength=1.4, handleheight=1.0, columnspacing=1.8,
               handletextpad=0.5, borderaxespad=0.0)

    _save(fig, f"fig_filter_removal_{MONTH_NAMES[month - 1]}")
    counts = pd.Series(reason).value_counts()
    print(f"\n{MONTH_NAMES[month - 1]} {year} screening, {len(sub):,} pixels:")
    for k in order:
        n = int(counts.get(k, 0))
        print(f"  {k:<10}{n:>10,}  {100 * n / len(sub):6.2f} %")
    print(f"Caption: Pixels removed by the screening in {MONTH_NAMES[month - 1]} {year}, classified by the "
          "first step they fail (brightness-temperature range, incomplete record, latitude), so the classes "
          "do not overlap. Retained pixels are shown in grey underneath. Maps are on the EASE2 grid; "
          "pixels equatorward of the map windows are not shown but are counted.")
    plt.show()
    return counts


# =============================================================================
# Run
# =============================================================================
def _nems_colocated_ds():
    """The co-located NEMS dataset opened in 1a, or opened again if `ds` was reused."""
    cand = globals().get("ds")
    if cand is not None and "TBNEMS" in getattr(cand, "data_vars", {}):
        return cand
    return xr.open_mfdataset(sorted(glob.glob(str(NEMS_COLOCATED_DIR / "**.nc"))),
                             combine="nested", concat_dim="t")


tiepoint_pts = plot_tiepoint_locations(arctic_record, antarctic_record)
nems_ds = _nems_colocated_ds()
_raw_pixels = build_raw_frames(nems_ds)
attrition_counts = plot_filter_attrition(_raw_pixels)
removal_counts = plot_filter_removal(_raw_pixels, month=SCREEN_MONTH)

In [ ]:
# =============================================================================
# 5k. Every number quoted in the manuscript, with its source
# =============================================================================
lines = []


def out(section, label, value, fmt="{:,}"):
    try:
        txt = fmt.format(value)
    except (ValueError, TypeError):
        txt = str(value)
    lines.append(f"{section:<28}{label:<62}{txt}")


S = SCREENING
out("Data / Filtering", "observations in the co-located files", S["all"])
out("Data / Filtering", "  of which with a valid time stamp (Fig. 2 'all extracted')", S["all_with_time"])
out("Data / Filtering", "Tb within 100-290 K in both channels", S["tb_range"])
out("Data / Filtering", "no undefined ERA5 field", S["complete"])
out("Data / Filtering", "poleward of 42 deg, NH + SH", S["nh"] + S["sh"])
out("Data / Filtering", "  NH", S["nh"])
out("Data / Filtering", "  SH", S["sh"])
out("Data / Filtering", "NH share of polar-domain obs removed by the ERA5 step [%]", 100 * S["nh_discarded_frac"], "{:.1f}")
out("Data / Filtering", "SH share of polar-domain obs removed by the ERA5 step [%]", 100 * S["sh_discarded_frac"], "{:.1f}")
out("Data", "days with NEMS data", S["n_days"])
for (reg, ch), sk in BIAS_MODEL_SKILL.items():
    out("Methods / RTM", f"XGBoost {reg} {ch} GHz: residual std before / test RMSE after [K]",
        f"{sk['resid_std_before']:.2f} / {sk['test_rmse']:.2f}", "{}")
for reg, (s_ow, s_ice) in SIGMA_2CH.items():
    out("Methods / Uncertainty", f"{reg}: sigma_OW, sigma_ICE (full record)", f"{s_ow:.4f}, {s_ice:.4f}", "{}")

for h in ("N", "S"):
    m = MASKS[h]
    out("Methods / Pole hole", f"{h}H pole hole [million km2]", m["pole_cap"].sum() * PIXEL_AREA_KM2 / 1e6, "{:.2f}")
    out("Methods / GNN", f"{h}H periods (train / val / test)", " / ".join(map(str, map(len, GNN[h]["split"])))
        if "split" in GNN[h] else "checkpoint loaded", "{}")
    if "n_params" in GNN[h]:
        out("Methods / GNN", f"{h}H trainable parameters", GNN[h]["n_params"])
    out("Methods / GNN", f"{h}H test periods", ", ".join(GNN[h]["split"][2]) if "split" in GNN[h] else "-", "{}")
    sw = GNN[h]["sweep"]
    piv = sw.pivot(index="gap_km", columns="method", values="mae")
    bl = [c for c in BASELINES if c in piv]
    best = piv[bl].min(axis=1)
    for gap in piv.index:
        out("Results / GNN", f"{h}H {gap} km: MAE network / best baseline [pp]",
            f"{piv.loc[gap, 'network']:.2f} / {best.loc[gap]:.2f} ({piv.loc[gap, bl].astype(float).idxmin()})", "{}")

for h in ("N", "S"):
    P6 = FIG6[h]
    ocean_obs = np.isfinite(P6["unc"]) & ~P6["land"]
    out("Results / SIC (Fig. 6)", f"{h}H period", P6["label"], "{}")
    out("Results / SIC (Fig. 6)", f"{h}H total uncertainty, mean / 95th pct over retrieved cells [pp]",
        f"{np.nanmean(P6['unc'][ocean_obs]):.2f} / {np.nanpercentile(P6['unc'][ocean_obs], 95):.2f}", "{}")
    if P6["prov"] is not None:
        for k in (1, 2, 3, 4, 5, 6, 7):
            n = int((P6["prov"] == k).sum())
            if n:
                out("Results / SIC (Fig. 6)", f"{h}H cells: {PROVENANCE[k][:45]}", f"{n:,} ({100 * n / P6['prov'].size:.1f} % of grid)", "{}")
    rec = arctic_record if h == "N" else antarctic_record
    t0 = datetime.strptime(P6["pk"], "%Y%m%d")
    fyi, myi, ow = _period_dynamic_tiepoint_flags(rec, t0, t0 + timedelta(days=PERIOD_DAYS - 1))
    dt = pd.to_datetime(rec["datetime"])
    win = (dt >= t0) & (dt < t0 + timedelta(days=PERIOD_DAYS))
    n_ice = int((win & (rec["c_ice_hybrid"] >= ICE_TP_MIN_SIC)).sum())
    n_ow = int((win & rec["Ts"].between(*OW_TP_SST) & (rec["c_ice_hybrid"] <= 0)).sum())
    out("Results / SIC (Fig. 6)", f"{h}H tie-point obs FYI / MYI / OW (of ice / OW candidates)",
        f"{fyi.sum()} / {myi.sum()} / {ow.sum()} (of {n_ice} / {n_ow})", "{}")

N = NIC_STATS
out("Results / Ice charts", "NIC charts used", ", ".join(N["chart_dates"]), "{}")
out("Results / Ice charts", "cells compared (and % of grid)", f"{N['n']:,} ({100 * N['frac_grid']:.1f} %)", "{}")
for k in ("mean", "median", "std", "rms", "p5", "p95"):
    out("Results / Ice charts", f"NEMS - NIC {k} [pp]", N[k], "{:+.2f}")
out("Results / Ice charts", "cells within +/-10 pp [%]", 100 * N["within_10pp"], "{:.1f}")
out("Results / Ice charts", "cells <= -50 pp [%] (area, million km2)", f"{100 * N['below_50']:.1f} ({N['below_50_area_mkm2']:.2f})", "{}")
out("Results / Ice charts", "cells >= +50 pp [%]", 100 * N["above_50"], "{:.1f}")
out("Results / Ice charts", "of the <= -50 pp cells, within 100 km of NIC 30 % edge [%]", 100 * N["below_50_near_edge"], "{:.0f}")
out("Results / Ice charts", "of all compared cells, within 100 km of the edge [%]", 100 * N["all_near_edge"], "{:.0f}")
for name, tab in REGION_BINS.items():
    out("Results / Ice charts", f"{name}: NEMS-NIC lowest / highest NIC class [pp]",
        f"{tab['mean'].iloc[0]:+.1f} ({tab.index[0]}) / {tab['mean'].iloc[-1]:+.1f} ({tab.index[-1]})", "{}")

for (h, a, b), st in EXTENT_AGREEMENT.items():
    if "bias" in st:
        out("Results / Extent", f"{h} {a} vs {b}: n, bias, RMSE [million km2], r",
            f"{st['n']}, {st['bias']:+.3f}, {st['rmse']:.3f}, {st['r']:.3f}", "{}")
    else:
        out("Results / Extent", f"{h} windows with NEMS and no ESMR", ", ".join(st["windows"]), "{}")

for h, res in NEMS_ESMR.items():
    out("Results / NEMS-ESMR daily", f"{h}H same-day days compared", res["days"])
    a = res["all"]
    out("Results / NEMS-ESMR daily", f"{h}H all ocean cells: n, mean, RMS [pp], within 10 pp [%]",
        f"{a['n']:,}, {a['mean']:+.2f}, {a['rms']:.2f}, {100 * a['within_10pp']:.1f}", "{}")
    for k in ("ice", "ice_0-100km", "ice_100-infkm", "ice_esmr0-30", "ice_esmr30-70", "ice_esmr70-90",
              "ice_esmr90-100", "ice_Dec-Feb", "ice_Jun", "ice_Oct"):
        s_ = res[k]
        out("Results / NEMS-ESMR daily", f"{h}H {k} (>=30 % in either): n, mean, median, RMS, within 10 pp",
            f"{s_['n']:,}, {s_['mean']:+.2f}, {s_['median']:+.2f}, {s_['rms']:.2f}, {100 * s_['within_10pp']:.1f} %", "{}")
        out("Results / NEMS-ESMR daily", f"{h}H {k}: RMS stated (NEMS+ESMR in quadrature), inside 1 sigma",
            f"{s_['unc_rms']:.2f} pp, {100 * s_['within_1sigma']:.0f} %", "{}")
    out("Results / NEMS-ESMR daily", f"{h}H stated RMS over ice cells: NEMS / ESMR [pp]",
        f"{res['ice']['unc_nems_rms']:.1f} / {res['ice']['unc_esmr_rms']:.1f}", "{}")

for _, r in NIC_WINDOWS.iterrows():
    tag = f"{r['window']}{' (ESMR outage)' if r['outage'] else ''}"
    out("Results / NIC by window", f"{tag}: charts", r["charts"], "{}")
    out("Results / NIC by window", f"{tag}: all cells n, within 10 pp [%]",
        f"{r['all_n']:,}, {100 * r['all_within_10pp']:.1f}", "{}")
    out("Results / NIC by window", f"{tag}: >=30 % cells n, mean, median, RMS, within 10 pp",
        f"{r['ice_n']:,}, {r['ice_mean']:+.2f}, {r['ice_median']:+.2f}, {r['ice_rms']:.2f}, {100 * r['ice_within_10pp']:.1f} %", "{}")
    out("Results / NIC by window", f"{tag}: >=30 % NEMS-retrieved cells n, mean, RMS, within 10 pp",
        f"{r['ice_nems_n']:,}, {r['ice_nems_mean']:+.2f}, {r['ice_nems_rms']:.2f}, {100 * r['ice_nems_within_10pp']:.1f} %", "{}")
    out("Results / NIC by window", f"{tag}: share of >=30 % cells retrieved / reconstructed [%]",
        f"{100 * r['ice_frac_nems']:.0f} / {100 * r['ice_frac_gnn']:.0f}", "{}")
    out("Results / NIC by window", f"{tag}: retrieved ice cells >100 km / <=100 km from edge: RMS vs stated [pp]",
        f"{r['far_rms']:.1f} vs {r['far_unc_rms']:.1f} / {r['near_rms']:.1f} vs {r['near_unc_rms']:.1f}", "{}")


for h in ("N", "S"):
    for tag, sw in (("test", GNN[h]["sweep"]), ("test, simulated outage", GNN[h]["sweep_outage"])):
        piv = sw.pivot(index="gap_km", columns="method", values="mae")
        for gap in piv.index:
            out("Results / Gap-filler", f"{h}H {tag} {gap} km MAE [pp]",
                ", ".join(f"{m} {piv.loc[gap, m]:.2f}" for m in piv.columns), "{}")
        out("Results / Gap-filler", f"{h}H {tag} mean over gaps MAE [pp]",
            ", ".join(f"{m} {piv[m].mean():.2f}" for m in piv.columns), "{}")
        rm = sw.pivot(index="gap_km", columns="method", values="rmse")
        out("Results / Gap-filler", f"{h}H {tag} RMSE of the network by gap [pp]",
            ", ".join(f"{g} {rm.loc[g, 'network']:.1f}" for g in rm.index), "{}")
for h in ("N", "S"):
    P6 = FIG6[h]
    m30 = np.isfinite(P6["unc"]) & ~P6["land"] & (P6["raw"] >= SIE_THRESHOLD)
    out("Results / SIC (Fig. 6)", f"{h}H total uncertainty, mean over retrieved cells with SIC >= 30 % [pp]",
        np.nanmean(P6["unc"][m30]), "{:.2f}")
for h, t in PROV_UNC.items():
    for k, r in t.iterrows():
        out("Results / Uncertainty", f"{h}H SIC >= 30 %: {r['source']}: cells, mean / median / p95 [pp]",
            f"{r['cells']:,}, {r['mean']:.2f} / {r['median']:.2f} / {r['p95']:.2f}", "{}")
for _, r in TIME_INTERP_ERR.groupby("margin").agg(dist=("dist_days", "mean"), rmse=("rmse", lambda x: float(np.sqrt(np.mean(np.square(x))))), n=("window", "size")).iterrows():
    out("Methods / Pole hole", f"ESMR interpolated in time, margin {_}: mean distance [d], RMSE [pp], windows",
        f"{r['dist']:.1f}, {r['rmse']:.2f}, {r['n']}", "{}")
for _, r in EXTENT_BY_PROV.iterrows():
    out("Results / Extent by source", f"{r['hemi']}H {r['window']}{' (ESMR outage)' if r['outage'] else ''}: total; NEMS / ESMR / GNN / pole same / pole interp / NN [million km2]",
        f"{r['total']:.2f}; {r['nems']:.2f} / {r['esmr']:.2f} / {r['gnn']:.2f} / {r['pole_same']:.2f} / {r['pole_interp']:.2f} / {r['nn']:.2f}", "{}")
for _, r in NIC_SAME_CELLS.iterrows():
    out("Results / Extent by source", f"NH {r['window']}: NEMS-retrieved cells, NEMS / NIC extent [million km2]",
        f"{r['nems']:.2f} / {r['nic']:.2f} ({r['diff']:+.2f})", "{}")
out("Results / Climatology mask", "NH NIC ice (>= 30 %) outside the mask: max % of NIC extent (thousand km2)",
    f"{CLIM_CHECK[CLIM_CHECK.hemi == 'N'].nic_ice_outside_pct.max():.2f} ({CLIM_CHECK[CLIM_CHECK.hemi == 'N'].nic_ice_outside_km2.max() / 1e3:.0f})", "{}")
for h, res in RTM_EFFECT.items():
    for key in sorted({kk for _, kk in res}):
        b, a = res[("before", key)], res[("after", key)]
        out("Results / RTM correction", f"{h}H {key}: mean / std before -> after [%] (n)",
            f"{b['mean']:+.1f} / {b['std']:.1f} -> {a['mean']:+.1f} / {a['std']:.1f} ({a['n']:,})", "{}")
for h, c in TP_CONTAM.items():
    out("Results / Tie points", f"{h}H OW candidates with ESMR >= 15 % in footprint [%] (n)", f"{100 * c['ow_any_ice']:.0f} ({c['ow_n']:,})", "{}")
    out("Results / Tie points", f"{h}H ice candidates with ESMR < 90 % in footprint [%] (n)", f"{100 * c['ice_not_pack']:.0f} ({c['ice_n']:,})", "{}")
for _, r in TP_SENS.iterrows():
    out("Results / Tie points", f"{r['hemi']}H {r['block']}: clean - standard OW22/OW31/MYI22/MYI31/FYI22/FYI31 [K]",
        f"{r['d_OW_tp22']:+.2f}/{r['d_OW_tp31']:+.2f}/{r['d_MYI_tp22']:+.2f}/{r['d_MYI_tp31']:+.2f}/{r['d_FYI_tp22']:+.2f}/{r['d_FYI_tp31']:+.2f}", "{}")
    out("Results / Tie points", f"{r['hemi']}H {r['block']}: pack SIC standard -> clean (ESMR) [%], clean-OW SIC change [pp], n clean ice / OW",
        f"{r['sic_pack_std']:.1f} -> {r['sic_pack_cln']:.1f} ({r['esmr_pack']:.1f}), {r['d_sic_water']:+.2f}, {r['n_ice_cln']} / {r['n_ow_cln']}", "{}")

out("Results / Table 3", "see", VAL_DIR / "table1_sie_monthly.tex", "{}")

text = "\n".join(lines)
(VAL_DIR / "manuscript_numbers.txt").write_text(text)
print(text)